In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

import transformers
transformers.logging.set_verbosity_error()

from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from tqdm.notebook import tqdm

print('✅ Imports done')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

SEED       = 42
set_seed(SEED)

MODEL_NAME = 'TinyLlama-1.1B'
HF_ID      = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
EVAL_N     = 500
K_SHOT     = 3
MAX_LEN    = 1024
RESULTS    = './results/tinyllama'
os.makedirs(RESULTS, exist_ok=True)

print(f'Config ready | EVAL_N={EVAL_N} | K_SHOT={K_SHOT}')

def sample_balanced(df, n, seed=SEED):
    h = df[df['label']=='human'].sample(n//2, random_state=seed)
    l = df[df['label']=='llm'].sample(n//2, random_state=seed)
    return pd.concat([h,l]).sample(frac=1, random_state=seed).reset_index(drop=True)

def get_pool(df, n=10):
    h = df[df['label']=='human'].sample(n, random_state=SEED)
    l = df[df['label']=='llm'].sample(n, random_state=SEED)
    return pd.concat([h,l]).reset_index(drop=True)

hc3_test   = pd.read_csv('hc3_test.csv')
eli5_test  = pd.read_csv('eli5_test.csv')
hc3_train  = pd.read_csv('hc3_train.csv')
eli5_train = pd.read_csv('eli5_train.csv')

hc3_eval  = sample_balanced(hc3_test,  EVAL_N)
eli5_eval = sample_balanced(eli5_test, EVAL_N)
hc3_pool  = get_pool(hc3_train)
eli5_pool = get_pool(eli5_train)

print(f'hc3_eval : {len(hc3_eval)} rows | {hc3_eval["label"].value_counts().to_dict()}')
print(f'eli5_eval: {len(eli5_eval)} rows | {eli5_eval["label"].value_counts().to_dict()}')

print(f'Loading {HF_ID} ...')

tokenizer = AutoTokenizer.from_pretrained(HF_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    HF_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print(f'✅ Model loaded on {next(model.parameters()).device}')

def get_label_token_ids(tokenizer):
    yes_ids, no_ids = set(), set()
    for s in ['yes', 'Yes', 'YES', ' yes', ' Yes']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            yes_ids.add(ids[0])
    for s in ['no', 'No', 'NO', ' no', ' No']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            no_ids.add(ids[0])

    print(f'yes (AI) token IDs : {list(yes_ids)}')
    print(f'  -> tokens        : {[tokenizer.decode([i]) for i in yes_ids]}')
    print(f'no (human) IDs     : {list(no_ids)}')
    print(f'  -> tokens        : {[tokenizer.decode([i]) for i in no_ids]}')

    if not yes_ids or not no_ids:
        raise RuntimeError('No valid single-token IDs found')
    return list(yes_ids), list(no_ids)

yes_ids, no_ids = get_label_token_ids(tokenizer)

import torch
dummy = tokenizer("Answer:", return_tensors='pt').to(model.device)
with torch.no_grad():
    logits = model(**dummy).logits[0, -1, :]
yes_logit = torch.stack([logits[i] for i in yes_ids]).max().item()
no_logit  = torch.stack([logits[i] for i in no_ids]).max().item()
print(f'\nBase logits after "Answer:" — yes: {yes_logit:.2f}  no: {no_logit:.2f}')
print('If these are close, labels are balanced ✓')

def tinyllama_zero_shot(text, tokenizer):
    user = (
        'Read the text below and answer the question.\n'
        'Question: Was this text generated by an AI language model?\n\n'
        f'Text:\n"""{text[:400]}"""\n\n'
        'Answer yes if AI-generated, no if human-written.'
    )
    msgs = [{'role': 'user', 'content': user}]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'

def tinyllama_few_shot(text, tokenizer, examples_df, k=K_SHOT):
    examples = examples_df.sample(k, random_state=SEED)
    shots = ''
    for _, row in examples.iterrows():
        ans = 'yes' if row['label'] == 'llm' else 'no'
        shots += f'Text: "{row["text"][:200]}"\nAI-generated? {ans}\n\n'
    user = (
        'Read each text and say yes (AI-generated) or no (human-written).\n\n'
        f'Examples:\n{shots}'
        f'Now answer:\nText: "{text[:400]}"\n'
        'AI-generated?'
    )
    msgs = [{'role': 'user', 'content': user}]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'

sample = str(hc3_eval.iloc[0]['text'])
p = tinyllama_zero_shot(sample, tokenizer)
print(f'Last 80 chars: ...{repr(p[-80:])}')

def constrained_score(model, tokenizer, prompt_text, yes_ids, no_ids):
    enc = tokenizer(prompt_text, return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits[0, -1, :]
    yes_logit = torch.stack([logits[i] for i in yes_ids]).max()
    no_logit  = torch.stack([logits[i] for i in no_ids]).max()
    return torch.softmax(torch.stack([no_logit, yes_logit]), dim=0)[1].item()

test_score = constrained_score(model, tokenizer, p, yes_ids, no_ids)
print(f'Sanity score: {test_score:.4f}  (should not be 0.0001 or 0.9999)')

# ── Debug: what does the dataset look like + how does TinyLlama respond ──

import textwrap

print('='*60)
print('DATASET SAMPLES')
print('='*60)
for ds_name, df in [('hc3', hc3_eval), ('eli5', eli5_eval)]:
    print(f'\n── {ds_name} ──')
    for lbl in ['human', 'llm']:
        row = df[df['label']==lbl].iloc[0]
        print(f'\n  label : {lbl}')
        print(f'  text  : {row["text"][:200]!r}')

print('\n' + '='*60)
print('PROMPT INSPECTION')
print('='*60)

sample_row = hc3_eval.iloc[0]
sample_text = str(sample_row['text'])
print(f'\nTrue label: {sample_row["label"]}')

zs_prompt = tinyllama_zero_shot(sample_text, tokenizer)
print(f'\n── Full ZS prompt ──\n{zs_prompt}')

print('\n' + '='*60)
print('RAW MODEL BEHAVIOUR at Answer: position')
print('='*60)

enc = tokenizer(zs_prompt, return_tensors='pt',
                truncation=True, max_length=MAX_LEN).to(model.device)
with torch.no_grad():
    logits = model(**enc).logits[0, -1, :]

top10 = torch.topk(logits, 10)
print('\nTop 10 tokens the model wants to predict after prompt:')
for rank, (tok_id, logit) in enumerate(zip(top10.indices, top10.values)):
    token_str = tokenizer.decode([tok_id])
    prob = torch.softmax(logits, dim=0)[tok_id].item()
    print(f'  rank {rank+1:2d}: {token_str!r:12s}  logit={logit:.2f}  prob={prob:.4f}')

print('\nLabel token logits specifically:')
for label, ids in [('yes (AI)',  yes_ids), ('no (human)', no_ids)]:
    for tid in ids:
        tok = tokenizer.decode([tid])
        logit = logits[tid].item()
        prob  = torch.softmax(logits, dim=0)[tid].item()
        print(f'  {label}: {tok!r}  id={tid}  logit={logit:.2f}  prob={prob:.4f}')

print(f'\nConstrained P(llm/yes) score: {constrained_score(model, tokenizer, zs_prompt, yes_ids, no_ids):.4f}')
print(f'True label: {sample_row["label"]}')

# ── Scoring loop ──

def run_regime(eval_df, pool, regime, ds_name):
    ckpt = f'{RESULTS}/ckpt_{MODEL_NAME}_{regime}_{ds_name}.json'
    if os.path.exists(ckpt):
        with open(ckpt) as f: records = json.load(f)
        done = {r['idx'] for r in records}
        print(f'  Resuming: {len(done)}/{len(eval_df)} done')
    else:
        records, done = [], set()

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df),
                          desc=f'{regime}/{ds_name}'):
        if idx in done: continue
        text, true_lbl = str(row['text']), row['label']
        prompt = (tinyllama_zero_shot(text, tokenizer) if regime == 'zero_shot'
                  else tinyllama_few_shot(text, tokenizer, pool))
        score  = constrained_score(model, tokenizer, prompt, yes_ids, no_ids)
        pred   = 'llm' if score > 0.5 else 'human'
        records.append({
            'idx': idx, 'true_label': true_lbl, 'pred_label': pred,
            'score': float(score), 'correct': int(pred == true_lbl),
            'failure_mode': (
                'correct'        if pred == true_lbl else
                'false_positive' if pred == 'llm' and true_lbl == 'human'
                else 'false_negative'
            )
        })
        with open(ckpt, 'w') as f: json.dump(records, f)

    df = pd.DataFrame(records)
    y_t = (df['true_label']=='llm').astype(int)
    y_s = df['score']
    auc = roc_auc_score(y_t, y_s) if y_t.nunique()>1 else float('nan')
    acc = accuracy_score(y_t, (y_s>0.5).astype(int))
    df.to_csv(f'{RESULTS}/{MODEL_NAME}_{regime}_{ds_name}.csv', index=False)
    return df, auc, acc


DATASETS = {
    'hc3' : (hc3_eval,  hc3_pool),
    'eli5': (eli5_eval, eli5_pool),
}

summary, all_dfs = [], {}

for regime in ['zero_shot', 'few_shot']:
    for ds_name, (ev, pool) in DATASETS.items():
        print(f'\n── {regime.upper()} | {ds_name.upper()} ──')
        df_r, auc, acc = run_regime(ev, pool, regime, ds_name)
        all_dfs[(regime, ds_name)] = df_r
        summary.append({'regime': regime, 'dataset': ds_name,
                         'auroc': round(auc,4), 'accuracy': round(acc,4),
                         'n': len(df_r)})
        print(f'   AUROC={auc:.4f}  Acc={acc:.4f}  n={len(df_r)}')

print('\n✅ All regimes done')

df_summary = pd.DataFrame(summary)
df_summary.to_csv(f'{RESULTS}/{MODEL_NAME}_summary.csv', index=False)

print(f'\n{MODEL_NAME} — Results Summary')
print('='*55)
print(df_summary.to_string(index=False))
print('='*55)

df_summary.style.background_gradient(subset=['auroc','accuracy'], cmap='RdYlGn')

# ── Calibrated threshold ──

print('Results with calibrated threshold:')
print('='*65)
calibrated = []
for (regime, ds), df_r in all_dfs.items():
    y_t = (df_r['true_label'] == 'llm').astype(int)
    y_s = df_r['score']

    fpr, tpr, thresholds = roc_curve(y_t, y_s)
    optimal_idx = (tpr - fpr).argmax()
    best_thresh = thresholds[optimal_idx]

    median_thresh = y_s.median()

    acc_optimal = accuracy_score(y_t, (y_s >= best_thresh).astype(int))
    acc_median  = accuracy_score(y_t, (y_s >= median_thresh).astype(int))
    auc         = roc_auc_score(y_t, y_s)

    calibrated.append({
        'regime': regime, 'dataset': ds,
        'auroc': round(auc, 4),
        'acc_fixed_0.5': round(accuracy_score(y_t, (y_s >= 0.5).astype(int)), 4),
        'acc_median_thresh': round(acc_median, 4),
        'acc_optimal_thresh': round(acc_optimal, 4),
        'median_score': round(float(median_thresh), 4),
        'optimal_thresh': round(float(best_thresh), 4),
    })
    print(f'{regime}/{ds}:')
    print(f'  AUROC={auc:.4f}  '
          f'acc@0.5={acc_optimal:.4f}  '
          f'acc@median={acc_median:.4f}  '
          f'acc@optimal={acc_optimal:.4f}')
    print(f'  median_score={median_thresh:.4f}  optimal_thresh={best_thresh:.4f}')

df_cal = pd.DataFrame(calibrated)
df_cal.to_csv(f'{RESULTS}/{MODEL_NAME}_calibrated.csv', index=False)
print('\n' + df_cal.to_string(index=False))

# ── Plots ──

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'{MODEL_NAME} — Detector Results', fontsize=13, fontweight='bold')

ax = axes[0]
labels = [f"{r['regime']}\n{r['dataset']}" for r in summary]
aurocs = [r['auroc'] for r in summary]
colors = ['#4C8EDA','#DA7E4C','#4CDA8E','#DA4C8E']
bars = ax.bar(labels, aurocs, color=colors[:len(summary)])
ax.axhline(0.5, color='grey', ls='--', lw=1, label='random')
ax.set_ylim(0,1); ax.set_ylabel('AUROC'); ax.set_title('AUROC by Regime & Dataset')
ax.legend(fontsize=8)
for bar, v in zip(bars, aurocs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=9)

ax2 = axes[1]
for i, ((regime, ds), df_r) in enumerate(all_dfs.items()):
    y_t = (df_r['true_label']=='llm').astype(int)
    y_s = df_r['score']
    if y_t.nunique() < 2: continue
    fpr, tpr, _ = roc_curve(y_t, y_s)
    auc = roc_auc_score(y_t, y_s)
    ax2.plot(fpr, tpr, color=plt.cm.tab10.colors[i],
             label=f'{regime}/{ds} {auc:.3f}')
ax2.plot([0,1],[0,1],'k--',lw=0.8)
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR')
ax2.set_title('ROC Curves'); ax2.legend(fontsize=8)

ax3 = axes[2]
for (regime, ds), df_r in all_dfs.items():
    for lbl, ls in [('human','--'), ('llm','-')]:
        vals = df_r[df_r['true_label']==lbl]['score']
        ax3.hist(vals, bins=25, alpha=0.4, density=True, ls=ls,
                 label=f'{regime}/{ds} {lbl}')
ax3.set_xlabel('P(llm)'); ax3.set_ylabel('Density')
ax3.set_title('Score Distributions')
ax3.legend(fontsize=6)

plt.tight_layout()
plt.savefig(f'{RESULTS}/{MODEL_NAME}_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to results folder')

for (regime, ds_name), df_r in all_dfs.items():
    out = {
        'model': MODEL_NAME, 'regime': regime, 'dataset': ds_name,
        'summary': {
            'total':    len(df_r),
            'correct':  int(df_r['correct'].sum()),
            'accuracy': round(df_r['correct'].mean(), 4),
            'fp': int((df_r['failure_mode']=='false_positive').sum()),
            'fn': int((df_r['failure_mode']=='false_negative').sum()),
        },
        'by_failure_mode': {
            'false_positives': df_r[df_r['failure_mode']=='false_positive'].to_dict('records'),
            'false_negatives': df_r[df_r['failure_mode']=='false_negative'].to_dict('records'),
            'correct':         df_r[df_r['failure_mode']=='correct'].to_dict('records'),
        }
    }
    with open(f'{RESULTS}/visual_{MODEL_NAME}_{regime}_{ds_name}.json','w') as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

print('✅ Visual JSON files saved')
print(f'   Results folder: {RESULTS}')

In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

import transformers
transformers.logging.set_verbosity_error()

from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from tqdm.notebook import tqdm

print('✅ Imports done')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# Cell 3 — config
SEED       = 42
set_seed(SEED)
MODEL_NAME = 'Qwen2.5-1.5B'
HF_ID      = 'Qwen/Qwen2.5-1.5B-Instruct'
EVAL_N     = 500
K_SHOT     = 3
MAX_LEN    = 2048
RESULTS    = './results/qwen25_1p5b'
os.makedirs(RESULTS, exist_ok=True)
print(f'Config ready | EVAL_N={EVAL_N}')

# Cell 4 — data
def sample_balanced(df, n, seed=SEED):
    h = df[df['label']=='human'].sample(n//2, random_state=seed)
    l = df[df['label']=='llm'].sample(n//2, random_state=seed)
    return pd.concat([h,l]).sample(frac=1, random_state=seed).reset_index(drop=True)

def get_pool(df, n=10):
    h = df[df['label']=='human'].sample(n, random_state=SEED)
    l = df[df['label']=='llm'].sample(n, random_state=SEED)
    return pd.concat([h,l]).reset_index(drop=True)

hc3_test   = pd.read_csv('hc3_test.csv')
eli5_test  = pd.read_csv('eli5_test.csv')
hc3_train  = pd.read_csv('hc3_train.csv')
eli5_train = pd.read_csv('eli5_train.csv')

hc3_eval  = sample_balanced(hc3_test,  EVAL_N)
eli5_eval = sample_balanced(eli5_test, EVAL_N)
hc3_pool  = get_pool(hc3_train)
eli5_pool = get_pool(eli5_train)

print(f'hc3_eval : {len(hc3_eval)} | {hc3_eval["label"].value_counts().to_dict()}')
print(f'eli5_eval: {len(eli5_eval)} | {eli5_eval["label"].value_counts().to_dict()}')

# Cell 5 — load model
print(f'Loading {HF_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(HF_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    HF_ID, torch_dtype=torch.float16,
    device_map='auto', trust_remote_code=True)
model.eval()
print(f'✅ Model loaded on {next(model.parameters()).device}')

# Cell 6 — token IDs (yes/no)
def get_label_token_ids(tokenizer):
    yes_ids, no_ids = set(), set()
    for s in ['yes', 'Yes', 'YES', ' yes', ' Yes']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            yes_ids.add(ids[0])
    for s in ['no', 'No', 'NO', ' no', ' No']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            no_ids.add(ids[0])

    print(f'yes (AI) IDs   : {list(yes_ids)} -> {[tokenizer.decode([i]) for i in yes_ids]}')
    print(f'no (human) IDs : {list(no_ids)} -> {[tokenizer.decode([i]) for i in no_ids]}')

    if not yes_ids or not no_ids:
        raise RuntimeError('No valid token IDs found')
    return list(yes_ids), list(no_ids)

yes_ids, no_ids = get_label_token_ids(tokenizer)

dummy = tokenizer('Answer:', return_tensors='pt').to(model.device)
with torch.no_grad():
    logits = model(**dummy).logits[0, -1, :]
yes_logit = torch.stack([logits[i] for i in yes_ids]).max().item()
no_logit  = torch.stack([logits[i] for i in no_ids]).max().item()
print(f'\nBase logits — yes: {yes_logit:.2f}  no: {no_logit:.2f}')
print('Close = balanced ✓  |  Large gap = prior bias warning')

# Cell 7 — prompts
def qwen_zero_shot(text, tokenizer):
    msgs = [
        {
            'role': 'system',
            'content': (
                'You detect AI-generated text. '
                'Answer with ONE word only: yes or no. '
                'yes = AI-generated. no = human-written.'
            )
        },
        {
            'role': 'user',
            'content': (
                'Was this text generated by an AI language model?\n\n'
                f'Text:\n"""{text[:500]}"""\n\n'
                'Answer yes or no.'
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'

def qwen_few_shot(text, tokenizer, examples_df, k=K_SHOT):
    examples = examples_df.sample(k, random_state=SEED)
    shots = ''
    for _, row in examples.iterrows():
        ans = 'yes' if row['label'] == 'llm' else 'no'
        shots += f'Text: "{row["text"][:250]}"\nAI-generated? {ans}\n\n'
    msgs = [
        {
            'role': 'system',
            'content': (
                'You detect AI-generated text. '
                'Answer with ONE word only: yes or no. '
                'yes = AI-generated. no = human-written.'
            )
        },
        {
            'role': 'user',
            'content': (
                f'Examples:\n{shots}'
                f'Now answer:\nText: "{text[:500]}"\n'
                'AI-generated? yes or no.'
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'

sample = str(hc3_eval.iloc[0]['text'])
p = qwen_zero_shot(sample, tokenizer)
print(f'Answer: count in ZS prompt: {p.count("Answer:")}  (expected 1)')
print(f'Last 80 chars: ...{repr(p[-80:])}')

# Cell 8 — constrained scoring
def constrained_score(model, tokenizer, prompt_text, yes_ids, no_ids):
    enc = tokenizer(prompt_text, return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits[0, -1, :]
    yes_logit = torch.stack([logits[i] for i in yes_ids]).max()
    no_logit  = torch.stack([logits[i] for i in no_ids]).max()
    return torch.softmax(torch.stack([no_logit, yes_logit]), dim=0)[1].item()

test_score = constrained_score(model, tokenizer, p, yes_ids, no_ids)
print(f'Sanity score: {test_score:.4f}  (should not be ~0.0 or ~1.0)')

enc = tokenizer(p, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(model.device)
with torch.no_grad():
    logits = model(**enc).logits[0, -1, :]
top10 = torch.topk(logits, 10)
print('\nTop 10 predicted tokens after prompt:')
for rank, (tid, logit) in enumerate(zip(top10.indices, top10.values)):
    prob = torch.softmax(logits, dim=0)[tid].item()
    print(f'  rank {rank+1:2d}: {tokenizer.decode([tid])!r:12s}  logit={logit:.2f}  prob={prob:.4f}')

# Cell 9 — scoring loop
def run_regime(eval_df, pool, regime, ds_name):
    ckpt = f'{RESULTS}/ckpt_{MODEL_NAME}_{regime}_{ds_name}.json'
    if os.path.exists(ckpt):
        with open(ckpt) as f: records = json.load(f)
        done = {r['idx'] for r in records}
        print(f'  Resuming: {len(done)}/{len(eval_df)} done')
    else:
        records, done = [], set()

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df),
                          desc=f'{regime}/{ds_name}'):
        if idx in done: continue
        text, true_lbl = str(row['text']), row['label']
        prompt = (qwen_zero_shot(text, tokenizer) if regime == 'zero_shot'
                  else qwen_few_shot(text, tokenizer, pool))
        score  = constrained_score(model, tokenizer, prompt, yes_ids, no_ids)
        pred   = 'llm' if score > 0.5 else 'human'
        records.append({
            'idx': idx, 'true_label': true_lbl, 'pred_label': pred,
            'score': float(score), 'correct': int(pred == true_lbl),
            'failure_mode': (
                'correct'        if pred == true_lbl else
                'false_positive' if pred == 'llm' and true_lbl == 'human'
                else 'false_negative'
            )
        })
        with open(ckpt, 'w') as f: json.dump(records, f)

    df = pd.DataFrame(records)
    y_t = (df['true_label'] == 'llm').astype(int)
    y_s = df['score']
    auc = roc_auc_score(y_t, y_s) if y_t.nunique() > 1 else float('nan')
    acc = accuracy_score(y_t, (y_s > 0.5).astype(int))
    df.to_csv(f'{RESULTS}/{MODEL_NAME}_{regime}_{ds_name}.csv', index=False)
    return df, auc, acc


DATASETS = {
    'hc3' : (hc3_eval,  hc3_pool),
    'eli5': (eli5_eval, eli5_pool),
}

summary, all_dfs = [], {}

for regime in ['zero_shot', 'few_shot']:
    for ds_name, (ev, pool) in DATASETS.items():
        print(f'\n── {regime.upper()} | {ds_name.upper()} ──')
        df_r, auc, acc = run_regime(ev, pool, regime, ds_name)
        all_dfs[(regime, ds_name)] = df_r
        summary.append({'regime': regime, 'dataset': ds_name,
                         'auroc': round(auc, 4), 'accuracy': round(acc, 4),
                         'n': len(df_r)})
        print(f'   AUROC={auc:.4f}  Acc={acc:.4f}  n={len(df_r)}')

print('\n✅ All regimes done')

# Cell 10 — calibrated results table
df_summary = pd.DataFrame(summary)
df_summary.to_csv(f'{RESULTS}/{MODEL_NAME}_summary.csv', index=False)

print(f'\n{MODEL_NAME} — Results (fixed threshold vs calibrated)')

rows = []
for (regime, ds), df_r in all_dfs.items():
    y_t = (df_r['true_label'] == 'llm').astype(int)
    y_s = df_r['score']
    fpr, tpr, thresholds = roc_curve(y_t, y_s)
    best_thresh   = thresholds[(tpr - fpr).argmax()]
    median_thresh = float(y_s.median())
    rows.append({
        'regime': regime, 'dataset': ds,
        'auroc':          round(roc_auc_score(y_t, y_s), 4),
        'acc@0.5':        round(accuracy_score(y_t, (y_s >= 0.5).astype(int)), 4),
        'acc@median':     round(accuracy_score(y_t, (y_s >= median_thresh).astype(int)), 4),
        'acc@optimal':    round(accuracy_score(y_t, (y_s >= best_thresh).astype(int)), 4),
        'median_score':   round(median_thresh, 4),
        'optimal_thresh': round(float(best_thresh), 4),
    })

df_cal = pd.DataFrame(rows)
df_cal.to_csv(f'{RESULTS}/{MODEL_NAME}_calibrated.csv', index=False)

# ── Pretty printed table ──
col_widths = {
    'regime':         10,
    'dataset':         6,
    'auroc':           8,
    'acc@0.5':         9,
    'acc@median':     12,
    'acc@optimal':    13,
    'median_score':   14,
    'optimal_thresh': 15,
}

header = (
    f"{'regime':<10} {'dataset':<6} {'auroc':>8} {'acc@0.5':>9} "
    f"{'acc@median':>12} {'acc@optimal':>13} {'median_score':>14} {'optimal_thresh':>15}"
)
sep = '─' * len(header)

print(sep)
print(header)
print(sep)
for r in rows:
    print(
        f"{r['regime']:<10} {r['dataset']:<6} "
        f"{r['auroc']:>8.4f} {r['acc@0.5']:>9.4f} "
        f"{r['acc@median']:>12.4f} {r['acc@optimal']:>13.4f} "
        f"{r['median_score']:>14.4f} {r['optimal_thresh']:>15.4f}"
    )
print(sep)

# Styled pandas table for notebook
df_cal.style.background_gradient(subset=['auroc','acc@optimal'], cmap='RdYlGn')

# Cell 11 — plots
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'{MODEL_NAME} — Detector Results', fontsize=13, fontweight='bold')

ax = axes[0]
labels_plot = [f"{r['regime']}\n{r['dataset']}" for r in summary]
aurocs = [r['auroc'] for r in summary]
colors = ['#4C8EDA','#DA7E4C','#4CDA8E','#DA4C8E']
bars = ax.bar(labels_plot, aurocs, color=colors[:len(summary)])
ax.axhline(0.5, color='grey', ls='--', lw=1, label='random')
ax.set_ylim(0,1); ax.set_ylabel('AUROC'); ax.set_title('AUROC by Regime & Dataset')
ax.legend(fontsize=8)
for bar, v in zip(bars, aurocs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=9)

ax2 = axes[1]
for i, ((regime, ds), df_r) in enumerate(all_dfs.items()):
    y_t = (df_r['true_label']=='llm').astype(int)
    y_s = df_r['score']
    if y_t.nunique() < 2: continue
    fpr, tpr, _ = roc_curve(y_t, y_s)
    auc = roc_auc_score(y_t, y_s)
    ax2.plot(fpr, tpr, color=plt.cm.tab10.colors[i],
             label=f'{regime}/{ds} {auc:.3f}')
ax2.plot([0,1],[0,1],'k--',lw=0.8)
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR')
ax2.set_title('ROC Curves'); ax2.legend(fontsize=8)

ax3 = axes[2]
for (regime, ds), df_r in all_dfs.items():
    for lbl, ls in [('human','--'),('llm','-')]:
        vals = df_r[df_r['true_label']==lbl]['score']
        ax3.hist(vals, bins=25, alpha=0.4, density=True,
                 label=f'{regime}/{ds} {lbl}')
ax3.set_xlabel('P(llm)'); ax3.set_ylabel('Density')
ax3.set_title('Score Distributions'); ax3.legend(fontsize=6)

plt.tight_layout()
plt.savefig(f'{RESULTS}/{MODEL_NAME}_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os, json, re, pickle, warnings, random
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

import transformers
transformers.logging.set_verbosity_error()

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, set_seed,
)
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

print('✅ Imports done')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
SEED        = 42
set_seed(SEED)
MODEL_NAME  = 'Llama-3.1-8B'
HF_ID       = 'meta-llama/Llama-3.1-8B-Instruct'
EVAL_N      = 500
COT_N       = 70
K_SHOT      = 3
MAX_LEN     = 2048
RESULTS_DIR = './results/llama31_8b_detector'
os.makedirs(RESULTS_DIR, exist_ok=True)

# CoT ensemble weights
# conf_weight  : weight on numeric confidence extracted from reasoning text
# logit_weight : weight on second-pass constrained token logit score
COT_CONF_WEIGHT  = 0.6
COT_LOGIT_WEIGHT = 0.4

print(f'Config ready | EVAL_N={EVAL_N} | COT_N={COT_N} | K_SHOT={K_SHOT}')
print(f'CoT ensemble weights | conf={COT_CONF_WEIGHT} | logit={COT_LOGIT_WEIGHT}')

# ─────────────────────────────────────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
def sample_balanced(df, n, seed=SEED):
    h = df[df['label'] == 'human'].sample(n // 2, random_state=seed)
    l = df[df['label'] == 'llm'].sample(n // 2, random_state=seed)
    return pd.concat([h, l]).sample(frac=1, random_state=seed).reset_index(drop=True)

def get_pool(df, n=30):
    h = df[df['label'] == 'human'].sample(n, random_state=SEED)
    l = df[df['label'] == 'llm'].sample(n, random_state=SEED)
    return pd.concat([h, l]).reset_index(drop=True)

hc3_test   = pd.read_csv('hc3_test.csv')
eli5_test  = pd.read_csv('eli5_test.csv')
hc3_train  = pd.read_csv('hc3_train.csv')
eli5_train = pd.read_csv('eli5_train.csv')

hc3_eval      = sample_balanced(hc3_test,  EVAL_N)
eli5_eval     = sample_balanced(eli5_test, EVAL_N)
hc3_cot_eval  = sample_balanced(hc3_test,  COT_N)
eli5_cot_eval = sample_balanced(eli5_test, COT_N)
hc3_pool      = get_pool(hc3_train,  n=30)
eli5_pool     = get_pool(eli5_train, n=30)

print(f'hc3_eval      : {len(hc3_eval)} rows | {hc3_eval["label"].value_counts().to_dict()}')
print(f'eli5_eval     : {len(eli5_eval)} rows | {eli5_eval["label"].value_counts().to_dict()}')
print(f'hc3_cot_eval  : {len(hc3_cot_eval)} rows')
print(f'eli5_cot_eval : {len(eli5_cot_eval)} rows')
print(f'hc3_pool size : {len(hc3_pool)} | eli5_pool size: {len(eli5_pool)}')

# ─────────────────────────────────────────────────────────────────────────────
# MODEL LOADING
# ─────────────────────────────────────────────────────────────────────────────
print(f'Loading {HF_ID} in 4-bit...')

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(HF_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    HF_ID, quantization_config=bnb,
    device_map='auto', trust_remote_code=True
)
model.eval()
print(f'✅ Model loaded on {next(model.parameters()).device}')

# ─────────────────────────────────────────────────────────────────────────────
# LABEL TOKEN IDS + PRIOR
# ─────────────────────────────────────────────────────────────────────────────
def get_label_token_ids(tokenizer):
    yes_ids, no_ids = set(), set()
    for s in ['yes', 'Yes', 'YES', ' yes', ' Yes']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            yes_ids.add(ids[0])
    for s in ['no', 'No', 'NO', ' no', ' No']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            no_ids.add(ids[0])
    print(f'yes (AI) IDs   : {list(yes_ids)}')
    print(f'  -> tokens     : {[tokenizer.decode([i]) for i in yes_ids]}')
    print(f'no (human) IDs : {list(no_ids)}')
    print(f'  -> tokens     : {[tokenizer.decode([i]) for i in no_ids]}')
    if not yes_ids or not no_ids:
        raise RuntimeError('No valid single-token IDs found')
    return list(yes_ids), list(no_ids)

yes_ids, no_ids = get_label_token_ids(tokenizer)

dummy = tokenizer('Answer:', return_tensors='pt').to(model.device)
with torch.no_grad():
    prior_logits = model(**dummy).logits[0, -1, :]

yes_prior = torch.stack([prior_logits[i] for i in yes_ids]).max()
no_prior  = torch.stack([prior_logits[i] for i in no_ids]).max()

print(f'\nPrior logits — yes: {yes_prior:.2f}  no: {no_prior:.2f}  gap: {(no_prior - yes_prior):.2f}')
print('Gap < 1.0 = well balanced | Gap > 2.0 = prior correction critical')

# ─────────────────────────────────────────────────────────────────────────────
# CONSTRAINED SCORING (zero/few-shot)
# ─────────────────────────────────────────────────────────────────────────────
def constrained_score(model, tokenizer, prompt_text, yes_ids, no_ids,
                      yes_prior, no_prior, flip=True, use_prior=True):
    """
    flip=True    → zero_shot/few_shot: model 'yes' correlates with human,
                   1-raw corrects polarity.
    flip=False   → CoT second pass: model answers literally, no inversion.
    use_prior    → subtract neutral-context prior logits.
    """
    enc = tokenizer(prompt_text, return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits[0, -1, :]

    yes_logit = torch.stack([logits[i] for i in yes_ids]).max()
    no_logit  = torch.stack([logits[i] for i in no_ids]).max()

    if use_prior:
        yes_logit = yes_logit - yes_prior
        no_logit  = no_logit  - no_prior

    raw = torch.softmax(torch.stack([no_logit, yes_logit]), dim=0)[1].item()
    return 1.0 - raw if flip else raw

# ─────────────────────────────────────────────────────────────────────────────
# TF-IDF RETRIEVERS
# ─────────────────────────────────────────────────────────────────────────────
def build_retriever(pool_df):
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    matrix = vec.fit_transform(pool_df['text'].astype(str))
    return vec, matrix

def retrieve_shots(text, pool_df, vec, matrix, k=K_SHOT):
    q = vec.transform([text])
    sims = cosine_similarity(q, matrix).flatten()
    results = []
    per_class = [k // 2 + (1 if i < k % 2 else 0) for i in range(2)]
    random.shuffle(per_class)
    for n, lbl in zip(per_class, ['human', 'llm']):
        mask = (pool_df['label'] == lbl).values
        class_sims = sims.copy()
        class_sims[~mask] = -1
        top_idx = class_sims.argsort()[-n:][::-1]
        results.extend(top_idx.tolist())
    return pool_df.iloc[results]

hc3_vec,  hc3_matrix  = build_retriever(hc3_pool)
eli5_vec, eli5_matrix = build_retriever(eli5_pool)
print('✅ TF-IDF retrievers built')

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT BUILDERS — ZERO SHOT & FEW SHOT
# ─────────────────────────────────────────────────────────────────────────────
def llama_zero_shot(text, tokenizer):
    msgs = [
        {
            'role': 'system',
            'content': (
                'You detect AI-generated text. '
                'Answer with ONE word only: yes or no. '
                'yes = AI-generated. no = human-written. '
                'No explanation. No punctuation. One word.'
            )
        },
        {
            'role': 'user',
            'content': (
                'Was this text generated by an AI language model?\n\n'
                f'Text:\n"""{text[:600]}"""\n\n'
                'Answer yes or no.'
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'

def llama_few_shot(text, tokenizer, examples_df, k=K_SHOT, idx=0, retriever=None):
    if retriever is not None:
        vec, matrix, pool_df = retriever
        examples = retrieve_shots(text, pool_df, vec, matrix, k)
    else:
        examples = examples_df.sample(k, random_state=idx)

    shots = ''
    for _, row in examples.iterrows():
        ans = 'yes' if row['label'] == 'llm' else 'no'
        shots += f'Text: "{row["text"][:250]}"\nAI-generated? {ans}\n\n'

    msgs = [
        {
            'role': 'system',
            'content': (
                'You detect AI-generated text. '
                'Answer with ONE word only: yes or no. '
                'yes = AI-generated. no = human-written.'
            )
        },
        {
            'role': 'user',
            'content': (
                f'Examples:\n{shots}'
                f'Now answer:\nText: "{text[:600]}"\n'
                'AI-generated? yes or no.'
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT BUILDERS — COT
# ─────────────────────────────────────────────────────────────────────────────
def llama_cot_reasoning_v2(text, tokenizer):
    """
    Revised CoT reasoning prompt.
    Key improvements over v1:
      - Explicitly targets ChatGPT / short-form LLM tells
      - Asks model to score each dimension 0-10 (extracts soft signal)
      - Asks for explicit AI_CONFIDENCE score to derive continuous probability
    """
    msgs = [
        {
            'role': 'system',
            'content': (
                'You are an expert forensic linguist. '
                'Your task is to determine whether a passage was written '
                'by a human or generated by an AI language model '
                '(including ChatGPT, GPT-4, Claude, Llama, etc.). '
                'Think carefully and be precise.'
            )
        },
        {
            'role': 'user',
            'content': (
                'Analyse whether this passage was written by a HUMAN or an AI.\n\n'
                f'Passage:\n"""{text[:700]}"""\n\n'
                'Evaluate each dimension and score it 0 (strongly human) '
                'to 10 (strongly AI):\n\n'
                '1. STRUCTURE: Is the answer neatly organised with clear sections '
                '   or numbered points? (AI: high structure, even in short answers)\n'
                '2. COMPLETENESS: Does it cover the topic comprehensively without '
                '   leaving obvious gaps? (AI tends to be thorough)\n'
                '3. HEDGING: Does it acknowledge uncertainty or say "I\'m not sure"? '
                '   (Humans hedge more; AI often sounds authoritative)\n'
                '4. PERSONAL VOICE: Any personal opinions, anecdotes, slang, '
                '   contractions, typos, or tangents? (Human markers)\n'
                '5. LEXICAL RANGE: Does it use broad, polished vocabulary '
                '   even in casual answers? (AI marker)\n'
                '6. RESPONSE FIT: Does it directly and completely address the '
                '   implied question, or does it wander? (AI: tight structured fit)\n'
                '7. SHORT-FORM TELLS: Even if short, does it feel generated — '
                '   e.g., starts with "Certainly!", restates the question, '
                '   gives a mini-essay for a simple prompt, or has an '
                '   unnaturally tidy closing sentence?\n\n'
                '8. BREVITY PATTERN: Short answers from AI often end with an unnaturally '
                '   tidy one-sentence summary or a "hope this helps" closing. '
                '   Human short answers just stop.\n'
                '9. QUESTION ECHO: Does the answer begin by restating or paraphrasing '
                '   the question? (Strong ChatGPT marker even in short answers)\n'
                '10. GENERIC EXAMPLES: Does it use placeholder examples like "for example, '
                '    consider X" where X is a perfectly illustrative but suspiciously apt '
                '    example? (AI marker)\n'
                'IMPORTANT: ChatGPT/LLM answers can be SHORT and still feel '
                'generated. Do not assume short = human.\n\n'
                'After scoring each dimension, compute:\n'
                '  AI_CONFIDENCE = average of your 7 scores (0-10 scale)\n\n'
                'State your verdict on the LAST TWO LINES as exactly:\n'
                'AI_CONFIDENCE: <number 0-10>\n'
                'VERDICT: yes   (if AI-generated)\n'
                'VERDICT: no    (if human-written)\n'
                '(Only one VERDICT line.)'
            )
        }
    ]
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)


def llama_cot_second_pass_v2(text, reasoning, tokenizer):
    """
    Second pass — always run this to get a calibrated token logit score.
    Used as co-signal with reasoning confidence, or sole signal if
    reasoning gave no numeric confidence.
    """
    msgs = [
        {
            'role': 'system',
            'content': (
                'You detect AI-generated text. '
                'Answer with ONE word only: yes or no. '
                'yes = AI-generated. no = human-written.'
            )
        },
        {
            'role': 'user',
            'content': (
                'You analysed this passage:\n\n'
                f'"""{text[:400]}"""\n\n'
                f'Your reasoning was:\n{reasoning[:600]}\n\n'
                'Based on that reasoning, is this AI-generated? '
                'yes or no?'
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'

# ─────────────────────────────────────────────────────────────────────────────
# COT PARSERS
# ─────────────────────────────────────────────────────────────────────────────
def parse_cot_output_v2(raw: str):
    """
    Extracts verdict and numeric AI confidence from CoT reasoning output.

    Returns:
        verdict : 'llm' | 'human' | 'unknown'
        conf    : float [0, 1] normalised AI confidence, or None
    """
    raw_lower = raw.lower()

    # ── Extract AI_CONFIDENCE (explicit tag) ──
    conf = None
    m_conf = re.search(r'ai_confidence[:\s]+([0-9]+(?:\.[0-9]+)?)', raw_lower)
    if m_conf:
        raw_score = float(m_conf.group(1))
        conf = min(raw_score / 10.0, 1.0)

    # ── Fallback: average inline dimension scores like "score: 7" or "7/10" ──
    if conf is None:
        scores_x_of_10 = re.findall(r'([0-9](?:\.[0-9]+)?)\s*/\s*10', raw_lower)
        if scores_x_of_10:
            vals = [float(s) for s in scores_x_of_10 if float(s) <= 10]
            if len(vals) >= 3:   # need at least 3 dimension scores to trust avg
                conf = min(sum(vals) / len(vals) / 10.0, 1.0)

    if conf is None:
        # Try "1. STRUCTURE: 7" style — digit after a dimension keyword
        dim_scores = re.findall(
            r'(?:structure|completeness|hedging|personal voice|lexical|'
            r'response fit|short.form)[^\n]*?([0-9](?:\.[0-9]+)?)\b',
            raw_lower
        )
        if dim_scores:
            vals = [float(s) for s in dim_scores if float(s) <= 10]
            if vals:
                conf = min(sum(vals) / len(vals) / 10.0, 1.0)

    # ── Extract VERDICT ──
    m_verdict = re.search(r'verdict:\s*(yes|no)', raw_lower)
    if m_verdict:
        verdict = 'llm' if m_verdict.group(1) == 'yes' else 'human'
    else:
        lines = [l.strip() for l in raw.split('\n') if l.strip()]
        last  = lines[-1].lower() if lines else ''
        if last.startswith('yes'):
            verdict = 'llm'
        elif last.startswith('no'):
            verdict = 'human'
        else:
            verdict = 'unknown'

    # ── Sanity check: if conf strongly disagrees with verdict, trust conf ──
    if conf is not None and verdict != 'unknown':
        if conf > 0.75 and verdict == 'human':
            verdict = 'llm'
        elif conf < 0.25 and verdict == 'llm':
            verdict = 'human'

    return verdict, conf


# ─────────────────────────────────────────────────────────────────────────────
# COT ENSEMBLE SCORER
# ─────────────────────────────────────────────────────────────────────────────
def cot_ensemble_score(conf, logit_score,
                       conf_weight=COT_CONF_WEIGHT,
                       logit_weight=COT_LOGIT_WEIGHT):
    DEAD_ZONE_LO = 0.40
    DEAD_ZONE_HI = 0.60

    if conf is None or (DEAD_ZONE_LO <= conf <= DEAD_ZONE_HI):
        # conf is uninformative — use logit only
        score = logit_score
        used_conf = False
    else:
        score = conf_weight * conf + logit_weight * logit_score
        used_conf = True

    return float(max(0.01, min(0.99, score))), used_conf


# ─────────────────────────────────────────────────────────────────────────────
# FULL COT PIPELINE FOR ONE SAMPLE
# ─────────────────────────────────────────────────────────────────────────────
def run_cot_sample(text, tokenizer, model, yes_ids, no_ids,
                   yes_prior, no_prior):

    # ── Pass 1: reasoning generation ──
    prompt1 = llama_cot_reasoning_v2(text, tokenizer)
    enc1 = tokenizer(prompt1, return_tensors='pt',
                     truncation=True, max_length=MAX_LEN).to(model.device)
    set_seed(SEED)
    with torch.no_grad():
        out1 = model.generate(
            **enc1, max_new_tokens=350, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    reasoning = tokenizer.decode(
        out1[0][enc1['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # ── Parse verdict + numeric confidence from reasoning ──
    verdict, conf = parse_cot_output_v2(reasoning)

    # ── Logit signal: use ZERO-SHOT prompt, not second-pass ──
    zs_prompt   = llama_zero_shot(text, tokenizer)
    logit_score = constrained_score(
        model, tokenizer, zs_prompt,
        yes_ids, no_ids, yes_prior, no_prior,
        flip=True,
        use_prior=True
    )

    # ── Ensemble: conf from reasoning + zero-shot logit ──
    score, used_conf = cot_ensemble_score(conf, logit_score)

    # ── Final prediction: score wins when confident, verdict breaks ties ──
    VERDICT_OVERRIDE_LO = 0.35
    VERDICT_OVERRIDE_HI = 0.65

    if verdict == 'unknown':
        pred = 'llm' if score > 0.5 else 'human'
    elif score < VERDICT_OVERRIDE_LO:
        pred = 'human'
    elif score > VERDICT_OVERRIDE_HI:
        pred = 'llm'
    else:
        pred = verdict

    return pred, score, reasoning, verdict, conf, logit_score, used_conf

print('✅ All prompt builders and CoT pipeline defined')

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG CELL — validate zero-shot polarity before full eval
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 65)
print('DEBUG — RAW INPUT TO MODEL + TOP-10 PREDICTIONS')
print('=' * 65)

for ds_name, df in [('hc3', hc3_eval), ('eli5', eli5_eval)]:
    for lbl in ['human', 'llm']:
        row  = df[df['label'] == lbl].iloc[0]
        text = str(row['text'])
        prompt = llama_zero_shot(text, tokenizer)

        print(f'\n── {ds_name.upper()} | true_label={lbl} ──')
        print(f'Raw text input (first 200 chars): {text[:200]!r}')
        print(f'\nFull prompt sent to model:\n{prompt}')
        print(f'\nLast 80 chars of prompt: ...{repr(prompt[-80:])}')

        enc = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=MAX_LEN).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits[0, -1, :]

        top10 = torch.topk(logits, 10)
        print('\nTop 10 tokens model wants to predict:')
        for rank, (tid, logit) in enumerate(zip(top10.indices, top10.values)):
            prob = torch.softmax(logits, dim=0)[tid].item()
            print(f'  rank {rank+1:2d}: {tokenizer.decode([tid])!r:12s}  '
                  f'logit={logit:.2f}  prob={prob:.4f}')

        print('\nLabel token logits:')
        for label_name, ids in [('yes (AI)', yes_ids), ('no (human)', no_ids)]:
            for tid in ids:
                tok   = tokenizer.decode([tid])
                logit = logits[tid].item()
                prob  = torch.softmax(logits, dim=0)[tid].item()
                print(f'  {label_name}: {tok!r:10s}  id={tid}  '
                      f'logit={logit:.2f}  prob={prob:.4f}')

        score = constrained_score(model, tokenizer, prompt,
                                  yes_ids, no_ids, yes_prior, no_prior,
                                  flip=True, use_prior=True)
        pred  = 'llm' if score > 0.5 else 'human'
        correct = '✅' if pred == lbl else '❌'
        print(f'\nPrior-corrected P(llm): {score:.4f}  pred={pred}  '
              f'true={lbl}  {correct}')
        print('-' * 65)

print('\n✅ Debug complete')

# ─────────────────────────────────────────────────────────────────────────────
# VERDICT AUDIT — run before full CoT eval to check direction
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 70)
print('VERDICT AUDIT — 20 samples per dataset to sanity-check CoT direction')
print('=' * 70)

for ds_name, df in [('hc3', hc3_cot_eval), ('eli5', eli5_cot_eval)]:
    correct_v = 0
    conf_vals = {'llm': [], 'human': []}
    logit_vals = {'llm': [], 'human': []}

    for _, row in df.head(20).iterrows():
        text, true_lbl = str(row['text']), row['label']
        # FIXED — unpack all 7
        pred, score, reasoning, verdict, conf, logit_score, used_conf = run_cot_sample(
            text, tokenizer, model, yes_ids, no_ids, yes_prior, no_prior
        )
        correct_v += int(pred == true_lbl)
        if conf is not None:
            conf_vals[true_lbl].append(conf)
        logit_vals[true_lbl].append(logit_score)

        status = '✅' if pred == true_lbl else '❌'
        conf_str = f'{conf:.3f}' if conf is not None else 'None'
        print(f'{status} ds={ds_name} true={true_lbl:5s} pred={pred:5s} verdict={verdict:7s} '
              f'conf={conf_str:5s} used_conf={used_conf} '
              f'logit={logit_score:.3f} score={score:.3f}')

    def safe_mean(lst): return round(sum(lst) / len(lst), 3) if lst else 'n/a'
    print(f'\n{ds_name} audit | verdict_acc={correct_v/20:.1%} | '
          f'conf(llm)={safe_mean(conf_vals["llm"])} '
          f'conf(human)={safe_mean(conf_vals["human"])} | '
          f'logit(llm)={safe_mean(logit_vals["llm"])} '
          f'logit(human)={safe_mean(logit_vals["human"])}\n')

print('✅ Verdict audit complete — check separation before running full eval')

# ─────────────────────────────────────────────────────────────────────────────
# MAIN EVALUATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
def run_regime(eval_df, pool, regime, ds_name, retriever=None):
    ckpt_path = f'{RESULTS_DIR}/ckpt_{MODEL_NAME}_{regime}_{ds_name}.json'

    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            records = json.load(f)
        done = {r['idx'] for r in records}
        print(f'  Resuming {regime}/{ds_name}: {len(done)}/{len(eval_df)} done')
    else:
        records, done = [], set()

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df),
                         desc=f'{regime}/{ds_name}', leave=True):
        if idx in done:
            continue
        text, true_lbl = str(row['text']), row['label']

        if regime == 'zero_shot':
            prompt = llama_zero_shot(text, tokenizer)
            score  = constrained_score(
                model, tokenizer, prompt,
                yes_ids, no_ids, yes_prior, no_prior,
                flip=True, use_prior=True
            )
            pred    = 'llm' if score > 0.5 else 'human'
            raw_out = f'[constrained+prior] score={score:.4f}'
            cot_verdict   = None
            cot_conf      = None
            cot_logit     = None

        elif regime == 'few_shot':
            prompt = llama_few_shot(text, tokenizer, pool,
                                    idx=idx, retriever=retriever)
            score  = constrained_score(
                model, tokenizer, prompt,
                yes_ids, no_ids, yes_prior, no_prior,
                flip=True, use_prior=True
            )
            pred    = 'llm' if score > 0.5 else 'human'
            raw_out = f'[constrained+prior] score={score:.4f}'
            cot_verdict   = None
            cot_conf      = None
            cot_logit     = None

        elif regime == 'cot':
            pred, score, reasoning, verdict, conf, logit_score, used_conf = run_cot_sample(
                text, tokenizer, model,
                yes_ids, no_ids, yes_prior, no_prior
            )
            conf_str = f'{conf:.4f}' if conf is not None else 'None'
            raw_out  = (
                f'[verdict={verdict}|conf={conf_str}|used_conf={used_conf}|'
                f'logit={logit_score:.4f}|score={score:.4f}]\n{reasoning}'
            )
            cot_verdict  = verdict
            cot_conf     = float(conf) if conf is not None else None
            cot_logit    = float(logit_score)
            cot_used_conf = bool(used_conf)

        else:
            continue

        records.append({
            'idx':             idx,
            'true_label':      true_lbl,
            'pred_label':      pred,
            'score':           float(score),
            'correct':         int(pred == true_lbl),
            'regime':          regime,
            'dataset':         ds_name,
            'text_preview':    text[:300],
            'raw_model_output': raw_out,
            'cot_verdict':     cot_verdict,
            'cot_conf':        cot_conf,
            'cot_logit_score': cot_logit,
            'cot_used_conf':   cot_used_conf if regime == 'cot' else None,  # ← added
            'failure_mode': (
                'correct'        if pred == true_lbl else
                'false_positive' if pred == 'llm' and true_lbl == 'human'
                else 'false_negative'
            )
        })

        with open(ckpt_path, 'w') as f:
            json.dump(records, f, indent=2, ensure_ascii=False)

    df_out = pd.DataFrame(records)
    y_t = (df_out['true_label'] == 'llm').astype(int)
    y_s = df_out['score']
    auc = roc_auc_score(y_t, y_s) if y_t.nunique() > 1 else float('nan')
    acc = accuracy_score(y_t, (y_s > 0.5).astype(int))
    df_out.to_csv(
        f'{RESULTS_DIR}/{MODEL_NAME}_{regime}_{ds_name}.csv', index=False)
    return df_out, auc, acc

print('✅ Scoring loop defined')

# ─────────────────────────────────────────────────────────────────────────────
# RUN ALL REGIMES
# ─────────────────────────────────────────────────────────────────────────────
hc3_retriever  = (hc3_vec,  hc3_matrix,  hc3_pool)
eli5_retriever = (eli5_vec, eli5_matrix, eli5_pool)

DATASETS = {
    'hc3' : {
        'eval':      hc3_eval,
        'cot':       hc3_cot_eval,
        'pool':      hc3_pool,
        'retriever': hc3_retriever,
    },
    'eli5': {
        'eval':      eli5_eval,
        'cot':       eli5_cot_eval,
        'pool':      eli5_pool,
        'retriever': eli5_retriever,
    },
}

summary, all_dfs = [], {}

for regime in ['zero_shot', 'few_shot', 'cot']:
    for ds_name, ds_cfg in DATASETS.items():
        ev        = ds_cfg['cot'] if regime == 'cot' else ds_cfg['eval']
        pool      = ds_cfg['pool']
        retriever = ds_cfg['retriever'] if regime == 'few_shot' else None

        print(f'\n── {regime.upper()} | {ds_name.upper()} ──')
        df_r, auc, acc = run_regime(ev, pool, regime, ds_name,
                                    retriever=retriever)
        all_dfs[(regime, ds_name)] = df_r

        fp = (df_r['failure_mode'] == 'false_positive').sum()
        fn = (df_r['failure_mode'] == 'false_negative').sum()

        # Extra CoT diagnostics
        if regime == 'cot':
            conf_available = df_r['cot_conf'].notna().mean()
            mean_conf_llm  = df_r[df_r['true_label']=='llm']['cot_conf'].mean()
            mean_conf_hum  = df_r[df_r['true_label']=='human']['cot_conf'].mean()
            mean_logit_llm = df_r[df_r['true_label']=='llm']['cot_logit_score'].mean()
            mean_logit_hum = df_r[df_r['true_label']=='human']['cot_logit_score'].mean()
            print(f'   AUROC={auc:.4f}  Acc={acc:.4f}  FP={fp}  FN={fn}  n={len(df_r)}')
            print(f'   conf_available={conf_available:.1%} | '
                  f'conf(llm)={mean_conf_llm:.3f}  conf(human)={mean_conf_hum:.3f}')
            print(f'   logit(llm)={mean_logit_llm:.3f}  logit(human)={mean_logit_hum:.3f}')
        else:
            print(f'   AUROC={auc:.4f}  Acc={acc:.4f}  FP={fp}  FN={fn}  n={len(df_r)}')

        summary.append({
            'regime':   regime,
            'dataset':  ds_name,
            'auroc':    round(auc, 4),
            'accuracy': round(acc, 4),
            'fp':       int(fp),
            'fn':       int(fn),
            'n':        len(df_r),
        })

print('\n✅ All regimes done')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLES
# ─────────────────────────────────────────────────────────────────────────────
df_summary = pd.DataFrame(summary)
df_summary.to_csv(f'{RESULTS_DIR}/{MODEL_NAME}_summary.csv', index=False)

print(f'\n{MODEL_NAME} — Results Summary')
header = (
    f"{'regime':<12} {'dataset':<6} {'auroc':>8} "
    f"{'accuracy':>10} {'fp':>6} {'fn':>6} {'n':>6}"
)
sep = '─' * len(header)
print(sep); print(header); print(sep)
for r in summary:
    print(
        f"{r['regime']:<12} {r['dataset']:<6} "
        f"{r['auroc']:>8.4f} {r['accuracy']:>10.4f} "
        f"{r['fp']:>6} {r['fn']:>6} {r['n']:>6}"
    )
print(sep)

# ── Calibrated threshold table ──
print(f'\n{MODEL_NAME} — Calibrated Threshold Results')
cal_rows = []
for (regime, ds), df_r in all_dfs.items():
    y_t = (df_r['true_label'] == 'llm').astype(int)
    y_s = df_r['score']
    fpr, tpr, thresholds = roc_curve(y_t, y_s)
    best_thresh   = thresholds[(tpr - fpr).argmax()]
    median_thresh = float(y_s.median())
    cal_rows.append({
        'regime':         regime,
        'dataset':        ds,
        'auroc':          round(roc_auc_score(y_t, y_s), 4),
        'acc@0.5':        round(accuracy_score(y_t, (y_s >= 0.5).astype(int)), 4),
        'acc@median':     round(accuracy_score(y_t, (y_s >= median_thresh).astype(int)), 4),
        'acc@optimal':    round(accuracy_score(y_t, (y_s >= best_thresh).astype(int)), 4),
        'median_score':   round(median_thresh, 4),
        'optimal_thresh': round(float(best_thresh), 4),
    })

pd.DataFrame(cal_rows).to_csv(
    f'{RESULTS_DIR}/{MODEL_NAME}_calibrated.csv', index=False)

cal_header = (
    f"{'regime':<12} {'dataset':<6} {'auroc':>8} {'acc@0.5':>9} "
    f"{'acc@median':>12} {'acc@optimal':>13} {'median':>8} {'opt_thresh':>11}"
)
sep2 = '─' * len(cal_header)
print(sep2); print(cal_header); print(sep2)
for r in cal_rows:
    print(
        f"{r['regime']:<12} {r['dataset']:<6} "
        f"{r['auroc']:>8.4f} {r['acc@0.5']:>9.4f} "
        f"{r['acc@median']:>12.4f} {r['acc@optimal']:>13.4f} "
        f"{r['median_score']:>8.4f} {r['optimal_thresh']:>11.4f}"
    )
print(sep2)

pd.DataFrame(cal_rows).style.background_gradient(
    subset=['auroc', 'acc@optimal'], cmap='RdYlGn')

# ── CoT component analysis (for paper) ──
print(f'\n{MODEL_NAME} — CoT Component Analysis')
print('(Shows how much conf vs logit each contributed)')
cot_analysis = []
for ds_name in ['hc3', 'eli5']:
    df_r = all_dfs.get(('cot', ds_name))
    if df_r is None:
        continue
    df_with_conf    = df_r[df_r['cot_conf'].notna()]
    df_without_conf = df_r[df_r['cot_conf'].isna()]

    for subset, label in [(df_with_conf, 'conf+logit'), (df_without_conf, 'logit_only')]:
        if len(subset) == 0:
            continue
        y_t = (subset['true_label'] == 'llm').astype(int)
        y_s = subset['score']
        if y_t.nunique() < 2:
            continue
        cot_analysis.append({
            'dataset':    ds_name,
            'score_type': label,
            'n':          len(subset),
            'auroc':      round(roc_auc_score(y_t, y_s), 4),
            'accuracy':   round(accuracy_score(y_t, (y_s > 0.5).astype(int)), 4),
            'pct_of_total': f'{len(subset)/len(df_r):.1%}',
        })

df_cot_analysis = pd.DataFrame(cot_analysis)
if len(df_cot_analysis):
    print(df_cot_analysis.to_string(index=False))
    df_cot_analysis.to_csv(
        f'{RESULTS_DIR}/{MODEL_NAME}_cot_component_analysis.csv', index=False)

# ─────────────────────────────────────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{MODEL_NAME} — Detector Results', fontsize=13, fontweight='bold')

ax = axes[0]
labels_plot = [f"{r['regime']}\n{r['dataset']}" for r in summary]
aurocs      = [r['auroc'] for r in summary]
colors      = ['#4C8EDA', '#DA7E4C', '#4CDA8E', '#DA4C8E', '#8E4CDA', '#DA4C4C']
bars = ax.bar(labels_plot, aurocs, color=colors[:len(summary)])
ax.axhline(0.5, color='grey', ls='--', lw=1, label='random')
ax.set_ylim(0, 1)
ax.set_ylabel('AUROC')
ax.set_title('AUROC by Regime & Dataset')
ax.legend(fontsize=8)
for bar, v in zip(bars, aurocs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax2 = axes[1]
for i, ((regime, ds), df_r) in enumerate(all_dfs.items()):
    y_t = (df_r['true_label'] == 'llm').astype(int)
    y_s = df_r['score']
    if y_t.nunique() < 2:
        continue
    fpr, tpr, _ = roc_curve(y_t, y_s)
    auc = roc_auc_score(y_t, y_s)
    ax2.plot(fpr, tpr, color=plt.cm.tab10.colors[i % 10],
             label=f'{regime}/{ds} {auc:.3f}')
ax2.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax2.set_xlabel('FPR')
ax2.set_ylabel('TPR')
ax2.set_title('ROC Curves')
ax2.legend(fontsize=7)

ax3 = axes[2]
for (regime, ds), df_r in all_dfs.items():
    for lbl, ls in [('human', '--'), ('llm', '-')]:
        vals = df_r[df_r['true_label'] == lbl]['score']
        ax3.hist(vals, bins=20, alpha=0.4, density=True,
                 label=f'{regime}/{ds} {lbl}')
ax3.axvline(0.5, color='black', ls='--', lw=1, label='threshold=0.5')
ax3.set_xlabel('P(llm) — ensemble score')
ax3.set_ylabel('Density')
ax3.set_title('Score Distributions')
ax3.legend(fontsize=6)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved')

# ── Per-condition score distributions ──
n_conds = len(all_dfs)
fig2, axes2 = plt.subplots(1, n_conds, figsize=(5 * n_conds, 4), squeeze=False)
for i, ((regime, ds), df_r) in enumerate(all_dfs.items()):
    ax = axes2[0][i]
    for lbl, color in [('human', '#4C8EDA'), ('llm', '#DA4C4C')]:
        subset_vals = df_r[df_r['true_label'] == lbl]['score']
        ax.hist(subset_vals, bins=20, alpha=0.6, color=color,
                label=lbl, density=True)
    ax.axvline(0.5, color='black', ls='--', lw=1)
    ax.set_title(f'{regime}/{ds}', fontsize=9)
    ax.set_xlabel('ensemble score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
plt.suptitle(f'{MODEL_NAME} — Score Distributions', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_score_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── CoT: conf vs logit scatter (for paper) ──
for ds_name in ['hc3', 'eli5']:
    df_r = all_dfs.get(('cot', ds_name))
    if df_r is None or 'cot_conf' not in df_r.columns:
        continue
    df_plot = df_r[df_r['cot_conf'].notna()].copy()
    if len(df_plot) == 0:
        continue
    fig3, ax_s = plt.subplots(figsize=(6, 5))
    for lbl, color, marker in [('human', '#4C8EDA', 'o'), ('llm', '#DA4C4C', 'x')]:
        sub = df_plot[df_plot['true_label'] == lbl]
        ax_s.scatter(sub['cot_conf'], sub['cot_logit_score'],
                     c=color, marker=marker, alpha=0.6, label=lbl, s=40)
    ax_s.set_xlabel('Reasoning Confidence (conf)')
    ax_s.set_ylabel('Second-Pass Logit Score')
    ax_s.set_title(f'CoT Component Scores — {ds_name.upper()}')
    ax_s.legend()
    ax_s.axvline(0.5, color='grey', ls='--', lw=0.8)
    ax_s.axhline(0.5, color='grey', ls='--', lw=0.8)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_cot_scatter_{ds_name}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# SAVE ALL OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
for (regime, ds_name), df_r in all_dfs.items():
    out = {
        'model':   MODEL_NAME,
        'regime':  regime,
        'dataset': ds_name,
        'cot_ensemble_weights': {
            'conf_weight':  COT_CONF_WEIGHT,
            'logit_weight': COT_LOGIT_WEIGHT,
        } if regime == 'cot' else None,
        'summary': {
            'total':    len(df_r),
            'correct':  int(df_r['correct'].sum()),
            'accuracy': round(df_r['correct'].mean(), 4),
            'fp': int((df_r['failure_mode'] == 'false_positive').sum()),
            'fn': int((df_r['failure_mode'] == 'false_negative').sum()),
        },
        'by_failure_mode': {
            'false_positives': df_r[df_r['failure_mode'] == 'false_positive'].to_dict('records'),
            'false_negatives': df_r[df_r['failure_mode'] == 'false_negative'].to_dict('records'),
            'correct':         df_r[df_r['failure_mode'] == 'correct'].to_dict('records'),
        }
    }
    with open(f'{RESULTS_DIR}/visual_{MODEL_NAME}_{regime}_{ds_name}.json', 'w') as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

with open(f'{RESULTS_DIR}/results_llama31_8b.pkl', 'wb') as f:
    pickle.dump({'all_dfs': all_dfs, 'summary': df_summary}, f)

print('✅ All files saved')
print(f'   Results folder: {RESULTS_DIR}')

In [ ]:
import os, json, re, pickle, warnings, random
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

import transformers
transformers.logging.set_verbosity_error()

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, set_seed,
)
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

print('✅ Imports done')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
SEED        = 42
set_seed(SEED)
MODEL_NAME  = 'Qwen2.5-7B'
HF_ID       = 'Qwen/Qwen2.5-7B-Instruct'
EVAL_N      = 500
COT_N       = 70
K_SHOT      = 3
MAX_LEN     = 2048
RESULTS_DIR = './results/qwen25_7b_detector'
os.makedirs(RESULTS_DIR, exist_ok=True)

COT_CONF_WEIGHT  = 0.6
COT_LOGIT_WEIGHT = 0.4
PRIOR_N          = 50

# Qwen-specific:
#   FLIP = False — prompt says yes=human, no=AI
#   So P(llm) = P(no) directly, no polarity inversion needed
#   Dead zone widened to [0.35, 0.65] — conf less reliable at 7B
FLIP           = False
DEAD_ZONE_LO   = 0.35
DEAD_ZONE_HI   = 0.65
VERDICT_LO     = 0.35
VERDICT_HI     = 0.65

print(f'Config ready | EVAL_N={EVAL_N} | COT_N={COT_N} | K_SHOT={K_SHOT}')
print(f'CoT weights  | conf={COT_CONF_WEIGHT} | logit={COT_LOGIT_WEIGHT}')
print(f'Qwen polarity| FLIP={FLIP} (yes=human, no=AI)')
print(f'Dead zone    | [{DEAD_ZONE_LO}, {DEAD_ZONE_HI}]')
print(f'Prior N      | PRIOR_N={PRIOR_N}')

# ─────────────────────────────────────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
def sample_balanced(df, n, seed=SEED):
    h = df[df['label'] == 'human'].sample(n // 2, random_state=seed)
    l = df[df['label'] == 'llm'].sample(n // 2, random_state=seed)
    return pd.concat([h, l]).sample(frac=1, random_state=seed).reset_index(drop=True)

def get_pool(df, n=30):
    h = df[df['label'] == 'human'].sample(n, random_state=SEED)
    l = df[df['label'] == 'llm'].sample(n, random_state=SEED)
    return pd.concat([h, l]).reset_index(drop=True)

hc3_test   = pd.read_csv('hc3_test.csv')
eli5_test  = pd.read_csv('eli5_test.csv')
hc3_train  = pd.read_csv('hc3_train.csv')
eli5_train = pd.read_csv('eli5_train.csv')

hc3_eval      = sample_balanced(hc3_test,  EVAL_N)
eli5_eval     = sample_balanced(eli5_test, EVAL_N)
hc3_cot_eval  = sample_balanced(hc3_test,  COT_N)
eli5_cot_eval = sample_balanced(eli5_test, COT_N)
hc3_pool      = get_pool(hc3_train,  n=30)
eli5_pool     = get_pool(eli5_train, n=30)

print(f'hc3_eval      : {len(hc3_eval)} rows | {hc3_eval["label"].value_counts().to_dict()}')
print(f'eli5_eval     : {len(eli5_eval)} rows | {eli5_eval["label"].value_counts().to_dict()}')
print(f'hc3_cot_eval  : {len(hc3_cot_eval)} rows')
print(f'eli5_cot_eval : {len(eli5_cot_eval)} rows')
print(f'hc3_pool size : {len(hc3_pool)} | eli5_pool size: {len(eli5_pool)}')

# ─────────────────────────────────────────────────────────────────────────────
# MODEL LOADING
# ─────────────────────────────────────────────────────────────────────────────
print(f'\nLoading {HF_ID} in 4-bit...')

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    HF_ID, trust_remote_code=True, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    HF_ID, quantization_config=bnb,
    device_map='auto', trust_remote_code=True)
model.eval()

print(f'✅ Model loaded on {next(model.parameters()).device}')
print(f'   Vocab size    : {tokenizer.vocab_size}')
print(f'   pad_token     : {tokenizer.pad_token!r}')
print(f'   eos_token     : {tokenizer.eos_token!r}')
print(f'   chat_template : {"present" if tokenizer.chat_template else "MISSING"}')

# ─────────────────────────────────────────────────────────────────────────────
# LABEL TOKEN IDS
# ─────────────────────────────────────────────────────────────────────────────
def get_label_token_ids(tokenizer):
    yes_ids, no_ids = set(), set()
    for s in ['yes', 'Yes', 'YES', ' yes', ' Yes']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            yes_ids.add(ids[0])
    for s in ['no', 'No', 'NO', ' no', ' No']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            no_ids.add(ids[0])
    print(f'yes (human) IDs : {list(yes_ids)}')
    print(f'  -> tokens      : {[tokenizer.decode([i]) for i in yes_ids]}')
    print(f'no (AI) IDs     : {list(no_ids)}')
    print(f'  -> tokens      : {[tokenizer.decode([i]) for i in no_ids]}')
    if not yes_ids or not no_ids:
        raise RuntimeError('No valid single-token IDs found')
    return list(yes_ids), list(no_ids)

yes_ids, no_ids = get_label_token_ids(tokenizer)

# ─────────────────────────────────────────────────────────────────────────────
# NEUTRAL PRIOR
# ─────────────────────────────────────────────────────────────────────────────
dummy = tokenizer('Answer:', return_tensors='pt').to(model.device)
with torch.no_grad():
    prior_logits = model(**dummy).logits[0, -1, :]

neutral_yes_prior = torch.stack([prior_logits[i] for i in yes_ids]).max()
neutral_no_prior  = torch.stack([prior_logits[i] for i in no_ids]).max()

print(f'\nNeutral prior — yes: {neutral_yes_prior:.2f}  '
      f'no: {neutral_no_prior:.2f}  '
      f'gap: {(neutral_no_prior - neutral_yes_prior):.2f}')
print('(Will be replaced by task prior)')

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT BUILDERS
# NOTE: Qwen polarity is SWAPPED vs Llama
#   yes = human-written   (Qwen naturally says yes to humans)
#   no  = AI-generated    (Qwen reluctantly says no to AI)
#   flip=False → P(llm) = P(no) directly
# ─────────────────────────────────────────────────────────────────────────────
def qwen_zero_shot(text, tokenizer):
    msgs = [
        {
            'role': 'system',
            'content': (
                'You detect AI-generated text. '
                'Answer with ONE word only: yes or no. '
                'yes = human-written. no = AI-generated. '
                'No explanation. No punctuation. One word.'
            )
        },
        {
            'role': 'user',
            'content': (
                'Was this text written by a human?\n\n'
                f'Text:\n"""{text[:600]}"""\n\n'
                'Answer yes or no.'
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'


def qwen_few_shot(text, tokenizer, examples_df, k=K_SHOT,
                  idx=0, retriever=None):
    if retriever is not None:
        vec, matrix, pool_df = retriever
        examples = retrieve_shots(text, pool_df, vec, matrix, k)
    else:
        examples = examples_df.sample(k, random_state=idx)

    shots = ''
    for _, row in examples.iterrows():
        # yes=human, no=AI — swapped from Llama
        ans = 'no' if row['label'] == 'llm' else 'yes'
        shots += f'Text: "{row["text"][:250]}"\nHuman-written? {ans}\n\n'

    msgs = [
        {
            'role': 'system',
            'content': (
                'You detect AI-generated text. '
                'Answer with ONE word only: yes or no. '
                'yes = human-written. no = AI-generated.'
            )
        },
        {
            'role': 'user',
            'content': (
                f'Examples:\n{shots}'
                f'Now answer:\nText: "{text[:600]}"\n'
                'Human-written? yes or no.'
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'


def qwen_cot_reasoning(text, tokenizer):
    msgs = [
        {
            'role': 'system',
            'content': (
                'You are an expert forensic linguist. '
                'Your task is to determine whether a passage was written '
                'by a human or generated by an AI language model '
                '(including ChatGPT, GPT-4, Claude, Llama, Qwen, etc.). '
                'Think carefully and be precise.'
            )
        },
        {
            'role': 'user',
            'content': (
                'Analyse whether this passage was written by a HUMAN or an AI.\n\n'
                f'Passage:\n"""{text[:700]}"""\n\n'
                'Evaluate each dimension and score it 0 (strongly human) '
                'to 10 (strongly AI):\n\n'
                '1. STRUCTURE: Is the answer neatly organised with clear sections '
                '   or numbered points? (AI: high structure, even in short answers)\n'
                '2. COMPLETENESS: Does it cover the topic comprehensively without '
                '   leaving obvious gaps? (AI tends to be thorough)\n'
                '3. HEDGING: Does it acknowledge uncertainty or say "I\'m not sure"? '
                '   (Humans hedge more; AI often sounds authoritative)\n'
                '4. PERSONAL VOICE: Any personal opinions, anecdotes, slang, '
                '   contractions, typos, or tangents? (Human markers)\n'
                '5. LEXICAL RANGE: Does it use broad, polished vocabulary '
                '   even in casual answers? (AI marker)\n'
                '6. RESPONSE FIT: Does it directly and completely address the '
                '   implied question, or does it wander? (AI: tight structured fit)\n'
                '7. SHORT-FORM TELLS: Even if short, does it feel generated — '
                '   e.g., starts with "Certainly!", restates the question, '
                '   gives a mini-essay for a simple prompt, or has an '
                '   unnaturally tidy closing sentence?\n\n'
                'IMPORTANT: ChatGPT/LLM answers can be SHORT and still feel '
                'generated. Do not assume short = human.\n\n'
                'After scoring each dimension, compute:\n'
                '  AI_CONFIDENCE = average of your 7 scores (0-10 scale)\n\n'
                'State your verdict on the LAST TWO LINES as exactly:\n'
                'AI_CONFIDENCE: <number 0-10>\n'
                'VERDICT: yes   (if AI-generated)\n'
                'VERDICT: no    (if human-written)\n'
                '(Only one VERDICT line.)'
            )
        }
    ]
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)

print('✅ Prompt builders defined')

# ─────────────────────────────────────────────────────────────────────────────
# CONSTRAINED SCORING
# flip=False for Qwen: P(llm) = P(no) since no=AI in our prompt
# ─────────────────────────────────────────────────────────────────────────────
def constrained_score(model, tokenizer, prompt_text, yes_ids, no_ids,
                      yes_prior, no_prior, flip=FLIP, use_prior=True):
    enc = tokenizer(prompt_text, return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits[0, -1, :]

    yes_logit = torch.stack([logits[i] for i in yes_ids]).max()
    no_logit  = torch.stack([logits[i] for i in no_ids]).max()

    if use_prior:
        yes_logit = yes_logit - yes_prior
        no_logit  = no_logit  - no_prior

    # softmax over [no_logit, yes_logit] → index 1 = P(yes)
    raw = torch.softmax(torch.stack([no_logit, yes_logit]), dim=0)[1].item()

    # flip=False: yes=human → P(llm) = P(no) = 1 - P(yes)
    # flip=True:  yes=AI   → P(llm) = P(yes) directly
    return 1.0 - raw if flip else raw

# ─────────────────────────────────────────────────────────────────────────────
# TF-IDF RETRIEVERS
# ─────────────────────────────────────────────────────────────────────────────
def build_retriever(pool_df):
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    matrix = vec.fit_transform(pool_df['text'].astype(str))
    return vec, matrix

def retrieve_shots(text, pool_df, vec, matrix, k=K_SHOT):
    q = vec.transform([text])
    sims = cosine_similarity(q, matrix).flatten()
    results = []
    per_class = [k // 2 + (1 if i < k % 2 else 0) for i in range(2)]
    random.shuffle(per_class)
    for n, lbl in zip(per_class, ['human', 'llm']):
        mask = (pool_df['label'] == lbl).values
        class_sims = sims.copy()
        class_sims[~mask] = -1
        top_idx = class_sims.argsort()[-n:][::-1]
        results.extend(top_idx.tolist())
    return pool_df.iloc[results]

hc3_vec,  hc3_matrix  = build_retriever(hc3_pool)
eli5_vec, eli5_matrix = build_retriever(eli5_pool)
print('✅ TF-IDF retrievers built')

# ─────────────────────────────────────────────────────────────────────────────
# TASK PRIOR
# Computed from actual task prompts using the SWAPPED polarity prompt.
# This captures Qwen's real yes/no marginal distribution on detection.
# ─────────────────────────────────────────────────────────────────────────────
def compute_task_prior(model, tokenizer, eval_df, yes_ids, no_ids,
                       n_samples=PRIOR_N, seed=SEED):
    """
    Average yes/no logits over n_samples real task prompts.
    Uses qwen_zero_shot (swapped polarity) so prior matches
    the exact prompt context used during evaluation.
    """
    sample = eval_df.sample(
        min(n_samples, len(eval_df)), random_state=seed)

    yes_logits_list, no_logits_list = [], []

    for _, row in tqdm(sample.iterrows(), total=len(sample),
                       desc='Computing task prior'):
        text   = str(row['text'])
        prompt = qwen_zero_shot(text, tokenizer)
        enc    = tokenizer(prompt, return_tensors='pt',
                           truncation=True,
                           max_length=MAX_LEN).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits[0, -1, :]

        yes_logits_list.append(
            torch.stack([logits[i] for i in yes_ids]).max().item())
        no_logits_list.append(
            torch.stack([logits[i] for i in no_ids]).max().item())

    task_yes_prior = torch.tensor(yes_logits_list).mean()
    task_no_prior  = torch.tensor(no_logits_list).mean()

    print(f'\nTask prior    — yes: {task_yes_prior:.3f}  '
          f'no: {task_no_prior:.3f}  '
          f'gap: {(task_no_prior - task_yes_prior):.3f}')
    print(f'Neutral prior — yes: {neutral_yes_prior:.3f}  '
          f'no: {neutral_no_prior:.3f}  '
          f'gap: {(neutral_no_prior - neutral_yes_prior):.3f}')
    print(f'Bias shift    — yes: {(task_yes_prior-neutral_yes_prior):.3f}  '
          f'no: {(task_no_prior-neutral_no_prior):.3f}')
    return task_yes_prior, task_no_prior


prior_sample = pd.concat([
    hc3_eval.sample(PRIOR_N // 2, random_state=SEED),
    eli5_eval.sample(PRIOR_N // 2, random_state=SEED)
]).reset_index(drop=True)

yes_prior, no_prior = compute_task_prior(
    model, tokenizer, prior_sample, yes_ids, no_ids)

print('\n✅ Task prior computed')

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG CELL 1 — polarity test
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('DEBUG 1 — POLARITY TEST')
print('=' * 65)
print('Target: task prior + flip=False → ✅ for both human and llm\n')

for ds_name, df in [('hc3', hc3_eval), ('eli5', eli5_eval)]:
    print(f'── {ds_name.upper()} ──')
    for lbl in ['human', 'llm']:
        row    = df[df['label'] == lbl].iloc[0]
        text   = str(row['text'])
        prompt = qwen_zero_shot(text, tokenizer)

        for prior_name, yp, np_ in [
            ('neutral', neutral_yes_prior, neutral_no_prior),
            ('task',    yes_prior,         no_prior)
        ]:
            for flip in [True, False]:
                s = constrained_score(
                    model, tokenizer, prompt,
                    yes_ids, no_ids, yp, np_,
                    flip=flip, use_prior=True)
                correct = (s > 0.5) == (lbl == 'llm')
                print(f'  true={lbl:5s} prior={prior_name:7s} '
                      f'flip={str(flip):5s} → score={s:.4f}  '
                      f'pred={"llm" if s>0.5 else "human":5s}  '
                      f'{"✅" if correct else "❌"}')
    print()

print('✅ Polarity test done')

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG CELL 2 — full token-level debug
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('DEBUG 2 — RAW INPUT TO MODEL + TOP-10 PREDICTIONS')
print('=' * 65)

for ds_name, df in [('hc3', hc3_eval), ('eli5', eli5_eval)]:
    for lbl in ['human', 'llm']:
        row  = df[df['label'] == lbl].iloc[0]
        text = str(row['text'])
        prompt = qwen_zero_shot(text, tokenizer)

        print(f'\n── {ds_name.upper()} | true_label={lbl} ──')
        print(f'Raw text (first 200 chars): {text[:200]!r}')
        print(f'\nFull prompt:\n{prompt}')
        print(f'\nLast 80 chars: ...{repr(prompt[-80:])}')

        enc = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=MAX_LEN).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits[0, -1, :]

        top10 = torch.topk(logits, 10)
        print('\nTop 10 tokens:')
        for rank, (tid, logit) in enumerate(
                zip(top10.indices, top10.values)):
            prob = torch.softmax(logits, dim=0)[tid].item()
            print(f'  rank {rank+1:2d}: {tokenizer.decode([tid])!r:12s}  '
                  f'logit={logit:.2f}  prob={prob:.4f}')

        print('\nLabel token logits:')
        for label_name, ids in [
            ('yes (human)', yes_ids),
            ('no  (AI)   ', no_ids)
        ]:
            for tid in ids:
                tok   = tokenizer.decode([tid])
                logit = logits[tid].item()
                prob  = torch.softmax(logits, dim=0)[tid].item()
                print(f'  {label_name}: {tok!r:10s}  id={tid}  '
                      f'logit={logit:.2f}  prob={prob:.4f}')

        s_neutral = constrained_score(
            model, tokenizer, prompt, yes_ids, no_ids,
            neutral_yes_prior, neutral_no_prior,
            flip=FLIP, use_prior=True)
        s_task = constrained_score(
            model, tokenizer, prompt, yes_ids, no_ids,
            yes_prior, no_prior,
            flip=FLIP, use_prior=True)

        print(f'\n  neutral prior → P(llm)={s_neutral:.4f}  '
              f'pred={"llm" if s_neutral>0.5 else "human":5s}  '
              f'{"✅" if (s_neutral>0.5)==(lbl=="llm") else "❌"}')
        print(f'  task prior    → P(llm)={s_task:.4f}  '
              f'pred={"llm" if s_task>0.5 else "human":5s}  '
              f'{"✅" if (s_task>0.5)==(lbl=="llm") else "❌"}')
        print('-' * 65)

print('\n✅ Debug 2 complete')
print('→ yes/no tokens must appear in top-10')
print('→ task prior must give more ✅ than neutral prior')

# ─────────────────────────────────────────────────────────────────────────────
# COT PARSERS
# ─────────────────────────────────────────────────────────────────────────────
def parse_cot_output_v2(raw: str):
    raw_lower = raw.lower()

    # ── AI_CONFIDENCE explicit tag ──
    conf = None
    m_conf = re.search(
        r'ai_confidence[:\s]+([0-9]+(?:\.[0-9]+)?)', raw_lower)
    if m_conf:
        raw_score = float(m_conf.group(1))
        conf = min(raw_score / 10.0, 1.0)

    # ── Fallback: x/10 patterns ──
    if conf is None:
        scores_x_of_10 = re.findall(
            r'([0-9](?:\.[0-9]+)?)\s*/\s*10', raw_lower)
        if scores_x_of_10:
            vals = [float(s) for s in scores_x_of_10 if float(s) <= 10]
            if len(vals) >= 3:
                conf = min(sum(vals) / len(vals) / 10.0, 1.0)

    # ── Fallback: dimension keyword scores ──
    if conf is None:
        dim_scores = re.findall(
            r'(?:structure|completeness|hedging|personal voice|lexical|'
            r'response fit|short.form)[^\n]*?([0-9](?:\.[0-9]+)?)\b',
            raw_lower
        )
        if dim_scores:
            vals = [float(s) for s in dim_scores if float(s) <= 10]
            if vals:
                conf = min(sum(vals) / len(vals) / 10.0, 1.0)

    # ── Verdict (CoT prompt: yes=AI, no=human — standard) ──
    m_verdict = re.search(r'verdict:\s*(yes|no)', raw_lower)
    if m_verdict:
        verdict = 'llm' if m_verdict.group(1) == 'yes' else 'human'
    else:
        lines = [l.strip() for l in raw.split('\n') if l.strip()]
        last  = lines[-1].lower() if lines else ''
        if last.startswith('yes'):
            verdict = 'llm'
        elif last.startswith('no'):
            verdict = 'human'
        else:
            verdict = 'unknown'

    # ── Sanity check: trust conf over verdict when strongly divergent ──
    if conf is not None and verdict != 'unknown':
        if conf > 0.75 and verdict == 'human':
            verdict = 'llm'
        elif conf < 0.25 and verdict == 'llm':
            verdict = 'human'

    return verdict, conf


# ─────────────────────────────────────────────────────────────────────────────
# COT ENSEMBLE SCORER
# ─────────────────────────────────────────────────────────────────────────────
def cot_ensemble_score(conf, logit_score,
                       conf_weight=COT_CONF_WEIGHT,
                       logit_weight=COT_LOGIT_WEIGHT):
    """
    Dead zone [0.35, 0.65]:
      conf inside  → uninformative, use logit only
      conf outside → genuine signal, weighted ensemble

    Wider than Llama (0.40/0.60) because Qwen 7B conf is
    less reliable and overcorrects more aggressively.
    """
    if conf is None or (DEAD_ZONE_LO <= conf <= DEAD_ZONE_HI):
        score     = logit_score
        used_conf = False
    else:
        score     = conf_weight * conf + logit_weight * logit_score
        used_conf = True

    return float(max(0.01, min(0.99, score))), used_conf


# ─────────────────────────────────────────────────────────────────────────────
# FULL COT PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def run_cot_sample(text, tokenizer, model, yes_ids, no_ids,
                   yes_prior, no_prior):
    """
    Pass 1 : Generate CoT reasoning → verdict + conf
    Logit  : Zero-shot constrained logit with task prior, flip=False
    Score  : Ensemble(conf, logit) — conf only if outside dead zone
    Pred   : Score wins when confident, verdict breaks ties in [0.35, 0.65]
    """
    # ── Pass 1: reasoning generation ──
    prompt1 = qwen_cot_reasoning(text, tokenizer)
    enc1 = tokenizer(prompt1, return_tensors='pt',
                     truncation=True, max_length=MAX_LEN).to(model.device)
    set_seed(SEED)
    with torch.no_grad():
        out1 = model.generate(
            **enc1, max_new_tokens=350, do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    reasoning = tokenizer.decode(
        out1[0][enc1['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # ── Parse verdict + confidence ──
    verdict, conf = parse_cot_output_v2(reasoning)

    # ── Zero-shot logit with swapped polarity prompt + task prior ──
    zs_prompt   = qwen_zero_shot(text, tokenizer)
    logit_score = constrained_score(
        model, tokenizer, zs_prompt,
        yes_ids, no_ids, yes_prior, no_prior,
        flip=FLIP,      # False for Qwen: P(llm) = P(no)
        use_prior=True
    )

    # ── Ensemble ──
    score, used_conf = cot_ensemble_score(conf, logit_score)

    # ── Final prediction ──
    if verdict == 'unknown':
        pred = 'llm' if score > 0.5 else 'human'
    elif score < VERDICT_LO:
        pred = 'human'          # score confidently human — ignore verdict
    elif score > VERDICT_HI:
        pred = 'llm'            # score confidently llm — ignore verdict
    else:
        pred = verdict          # uncertain score — let verdict break tie

    return pred, score, reasoning, verdict, conf, logit_score, used_conf

print('✅ CoT pipeline defined')

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG CELL 3 — verdict audit (20 samples per dataset)
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 70)
print('DEBUG 3 — VERDICT AUDIT (20 samples per dataset)')
print('=' * 70)

for ds_name, df in [('hc3', hc3_cot_eval), ('eli5', eli5_cot_eval)]:
    correct_v  = 0
    conf_vals  = {'llm': [], 'human': []}
    logit_vals = {'llm': [], 'human': []}

    for _, row in df.head(20).iterrows():
        text, true_lbl = str(row['text']), row['label']
        pred, score, reasoning, verdict, conf, logit_score, used_conf = \
            run_cot_sample(text, tokenizer, model,
                           yes_ids, no_ids, yes_prior, no_prior)

        correct_v += int(pred == true_lbl)
        if conf is not None:
            conf_vals[true_lbl].append(conf)
        logit_vals[true_lbl].append(logit_score)

        conf_str = f'{conf:.3f}' if conf is not None else 'None'
        status   = '✅' if pred == true_lbl else '❌'
        print(f'{status} ds={ds_name} true={true_lbl:5s} pred={pred:5s} '
              f'verdict={verdict:7s} conf={conf_str:5s} '
              f'used_conf={used_conf} '
              f'logit={logit_score:.3f} score={score:.3f}')

    def safe_mean(lst):
        return round(sum(lst)/len(lst), 3) if lst else 'n/a'

    lsep = (safe_mean(logit_vals['llm']) - safe_mean(logit_vals['human'])
            if logit_vals['llm'] and logit_vals['human'] else 'n/a')

    print(f'\n{ds_name} audit | verdict_acc={correct_v/20:.1%} | '
          f'conf(llm)={safe_mean(conf_vals["llm"])} '
          f'conf(human)={safe_mean(conf_vals["human"])} | '
          f'logit(llm)={safe_mean(logit_vals["llm"])} '
          f'logit(human)={safe_mean(logit_vals["human"])} | '
          f'logit_sep={lsep}')
    print(f'  → logit_sep > 0 = correct direction\n')

print('✅ Verdict audit complete')
print('→ logit(llm) should be > logit(human)')

# ─────────────────────────────────────────────────────────────────────────────
# MAIN EVALUATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
def run_regime(eval_df, pool, regime, ds_name,
               yes_prior, no_prior, retriever=None):
    ckpt_path = (f'{RESULTS_DIR}/ckpt_{MODEL_NAME}'
                 f'_{regime}_{ds_name}.json')

    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            records = json.load(f)
        done = {r['idx'] for r in records}
        print(f'  Resuming {regime}/{ds_name}: '
              f'{len(done)}/{len(eval_df)} done')
    else:
        records, done = [], set()

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df),
                         desc=f'{regime}/{ds_name}', leave=True):
        if idx in done:
            continue
        text, true_lbl = str(row['text']), row['label']

        if regime == 'zero_shot':
            prompt = qwen_zero_shot(text, tokenizer)
            score  = constrained_score(
                model, tokenizer, prompt,
                yes_ids, no_ids, yes_prior, no_prior,
                flip=FLIP, use_prior=True
            )
            pred          = 'llm' if score > 0.5 else 'human'
            raw_out       = f'[constrained+task_prior] score={score:.4f}'
            cot_verdict   = None
            cot_conf      = None
            cot_logit     = None
            cot_used_conf = None

        elif regime == 'few_shot':
            prompt = qwen_few_shot(text, tokenizer, pool,
                                   idx=idx, retriever=retriever)
            score  = constrained_score(
                model, tokenizer, prompt,
                yes_ids, no_ids, yes_prior, no_prior,
                flip=FLIP, use_prior=True
            )
            pred          = 'llm' if score > 0.5 else 'human'
            raw_out       = f'[constrained+task_prior] score={score:.4f}'
            cot_verdict   = None
            cot_conf      = None
            cot_logit     = None
            cot_used_conf = None

        elif regime == 'cot':
            (pred, score, reasoning, verdict,
             conf, logit_score, used_conf) = run_cot_sample(
                text, tokenizer, model,
                yes_ids, no_ids, yes_prior, no_prior
            )
            conf_str      = f'{conf:.4f}' if conf is not None else 'None'
            raw_out       = (
                f'[verdict={verdict}|conf={conf_str}|'
                f'used_conf={used_conf}|'
                f'logit={logit_score:.4f}|score={score:.4f}]\n{reasoning}'
            )
            cot_verdict   = verdict
            cot_conf      = float(conf) if conf is not None else None
            cot_logit     = float(logit_score)
            cot_used_conf = bool(used_conf)

        else:
            continue

        correct = int(pred == true_lbl)

        if regime == 'cot':
            if cot_verdict != 'unknown' and correct:
                correctness_source = 'verdict'
            elif cot_verdict == 'unknown' and correct:
                correctness_source = 'score_fallback'
            elif cot_verdict != 'unknown' and not correct:
                correctness_source = 'verdict_fail'
            else:
                correctness_source = 'score_fail'
        else:
            correctness_source = 'score' if correct else 'score_fail'

        records.append({
            'idx':                idx,
            'true_label':         true_lbl,
            'pred_label':         pred,
            'score':              float(score),
            'correct':            correct,
            'regime':             regime,
            'dataset':            ds_name,
            'text_preview':       text[:300],
            'raw_model_output':   raw_out,
            'cot_verdict':        cot_verdict,
            'cot_conf':           cot_conf,
            'cot_logit_score':    cot_logit,
            'cot_used_conf':      cot_used_conf,
            'correctness_source': correctness_source,
            'failure_mode': (
                'correct'        if pred == true_lbl else
                'false_positive' if pred == 'llm' and true_lbl == 'human'
                else 'false_negative'
            )
        })

        with open(ckpt_path, 'w') as f:
            json.dump(records, f, indent=2, ensure_ascii=False)

    df_out = pd.DataFrame(records)
    y_t = (df_out['true_label'] == 'llm').astype(int)
    y_s = df_out['score']
    auc = roc_auc_score(y_t, y_s) if y_t.nunique() > 1 else float('nan')
    acc = accuracy_score(y_t, (y_s > 0.5).astype(int))
    df_out.to_csv(
        f'{RESULTS_DIR}/{MODEL_NAME}_{regime}_{ds_name}.csv', index=False)
    return df_out, auc, acc

print('✅ Scoring loop defined')

# ─────────────────────────────────────────────────────────────────────────────
# RUN ALL REGIMES
# ─────────────────────────────────────────────────────────────────────────────
hc3_retriever  = (hc3_vec,  hc3_matrix,  hc3_pool)
eli5_retriever = (eli5_vec, eli5_matrix, eli5_pool)

DATASETS = {
    'hc3' : {
        'eval':      hc3_eval,
        'cot':       hc3_cot_eval,
        'pool':      hc3_pool,
        'retriever': hc3_retriever,
    },
    'eli5': {
        'eval':      eli5_eval,
        'cot':       eli5_cot_eval,
        'pool':      eli5_pool,
        'retriever': eli5_retriever,
    },
}

summary, all_dfs = [], {}

for regime in ['zero_shot', 'few_shot', 'cot']:
    for ds_name, ds_cfg in DATASETS.items():
        ev        = ds_cfg['cot'] if regime == 'cot' else ds_cfg['eval']
        pool      = ds_cfg['pool']
        retriever = ds_cfg['retriever'] if regime == 'few_shot' else None

        print(f'\n── {regime.upper()} | {ds_name.upper()} ──')
        df_r, auc, acc = run_regime(
            ev, pool, regime, ds_name,
            yes_prior, no_prior,
            retriever=retriever
        )
        all_dfs[(regime, ds_name)] = df_r

        fp = (df_r['failure_mode'] == 'false_positive').sum()
        fn = (df_r['failure_mode'] == 'false_negative').sum()

        if regime == 'cot':
            conf_available = df_r['cot_conf'].notna().mean()
            mean_conf_llm  = df_r[df_r['true_label']=='llm']['cot_conf'].mean()
            mean_conf_hum  = df_r[df_r['true_label']=='human']['cot_conf'].mean()
            mean_logit_llm = df_r[df_r['true_label']=='llm']['cot_logit_score'].mean()
            mean_logit_hum = df_r[df_r['true_label']=='human']['cot_logit_score'].mean()
            src_counts     = df_r['correctness_source'].value_counts().to_dict()
            print(f'   AUROC={auc:.4f}  Acc={acc:.4f}  '
                  f'FP={fp}  FN={fn}  n={len(df_r)}')
            print(f'   conf_avail={conf_available:.1%} | '
                  f'conf(llm)={mean_conf_llm:.3f}  '
                  f'conf(human)={mean_conf_hum:.3f}')
            print(f'   logit(llm)={mean_logit_llm:.3f}  '
                  f'logit(human)={mean_logit_hum:.3f}  '
                  f'sep={mean_logit_llm-mean_logit_hum:.3f}')
            print(f'   correctness_source: {src_counts}')
        else:
            print(f'   AUROC={auc:.4f}  Acc={acc:.4f}  '
                  f'FP={fp}  FN={fn}  n={len(df_r)}')

        summary.append({
            'regime':   regime,
            'dataset':  ds_name,
            'auroc':    round(auc, 4),
            'accuracy': round(acc, 4),
            'fp':       int(fp),
            'fn':       int(fn),
            'n':        len(df_r),
        })

print('\n✅ All regimes done')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLES
# ─────────────────────────────────────────────────────────────────────────────
df_summary = pd.DataFrame(summary)
df_summary.to_csv(f'{RESULTS_DIR}/{MODEL_NAME}_summary.csv', index=False)

print(f'\n{MODEL_NAME} — Results Summary')
header = (f"{'regime':<12} {'dataset':<6} {'auroc':>8} "
          f"{'accuracy':>10} {'fp':>6} {'fn':>6} {'n':>6}")
sep = '─' * len(header)
print(sep); print(header); print(sep)
for r in summary:
    print(f"{r['regime']:<12} {r['dataset']:<6} "
          f"{r['auroc']:>8.4f} {r['accuracy']:>10.4f} "
          f"{r['fp']:>6} {r['fn']:>6} {r['n']:>6}")
print(sep)

# ── Calibrated threshold table ──
print(f'\n{MODEL_NAME} — Calibrated Threshold Results')
cal_rows = []
for (regime, ds), df_r in all_dfs.items():
    y_t = (df_r['true_label'] == 'llm').astype(int)
    y_s = df_r['score']
    fpr, tpr, thresholds = roc_curve(y_t, y_s)
    best_thresh   = thresholds[(tpr - fpr).argmax()]
    median_thresh = float(y_s.median())
    cal_rows.append({
        'regime':         regime,
        'dataset':        ds,
        'auroc':          round(roc_auc_score(y_t, y_s), 4),
        'acc@0.5':        round(accuracy_score(
                              y_t, (y_s >= 0.5).astype(int)), 4),
        'acc@median':     round(accuracy_score(
                              y_t, (y_s >= median_thresh).astype(int)), 4),
        'acc@optimal':    round(accuracy_score(
                              y_t, (y_s >= best_thresh).astype(int)), 4),
        'median_score':   round(median_thresh, 4),
        'optimal_thresh': round(float(best_thresh), 4),
    })

pd.DataFrame(cal_rows).to_csv(
    f'{RESULTS_DIR}/{MODEL_NAME}_calibrated.csv', index=False)

cal_header = (
    f"{'regime':<12} {'dataset':<6} {'auroc':>8} {'acc@0.5':>9} "
    f"{'acc@median':>12} {'acc@optimal':>13} "
    f"{'median':>8} {'opt_thresh':>11}"
)
sep2 = '─' * len(cal_header)
print(sep2); print(cal_header); print(sep2)
for r in cal_rows:
    print(f"{r['regime']:<12} {r['dataset']:<6} "
          f"{r['auroc']:>8.4f} {r['acc@0.5']:>9.4f} "
          f"{r['acc@median']:>12.4f} {r['acc@optimal']:>13.4f} "
          f"{r['median_score']:>8.4f} {r['optimal_thresh']:>11.4f}")
print(sep2)

pd.DataFrame(cal_rows).style.background_gradient(
    subset=['auroc', 'acc@optimal'], cmap='RdYlGn')

# ── CoT component analysis ──
print(f'\n{MODEL_NAME} — CoT Component Analysis')
cot_analysis = []
for ds_name in ['hc3', 'eli5']:
    df_r = all_dfs.get(('cot', ds_name))
    if df_r is None:
        continue
    df_with_conf    = df_r[df_r['cot_conf'].notna() & df_r['cot_used_conf']]
    df_without_conf = df_r[~(df_r['cot_conf'].notna() & df_r['cot_used_conf'])]
    for subset, label in [
        (df_with_conf,    'conf+logit'),
        (df_without_conf, 'logit_only')
    ]:
        if len(subset) < 2:
            continue
        y_t = (subset['true_label'] == 'llm').astype(int)
        y_s = subset['score']
        if y_t.nunique() < 2:
            continue
        cot_analysis.append({
            'dataset':      ds_name,
            'score_type':   label,
            'n':            len(subset),
            'auroc':        round(roc_auc_score(y_t, y_s), 4),
            'accuracy':     round(accuracy_score(
                                y_t, (y_s > 0.5).astype(int)), 4),
            'pct_of_total': f'{len(subset)/len(df_r):.1%}',
        })

df_cot_analysis = pd.DataFrame(cot_analysis)
if len(df_cot_analysis):
    print(df_cot_analysis.to_string(index=False))
    df_cot_analysis.to_csv(
        f'{RESULTS_DIR}/{MODEL_NAME}_cot_component_analysis.csv',
        index=False)

# ─────────────────────────────────────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f'{MODEL_NAME} — Detector Results\n'
    f'(swapped polarity: yes=human, no=AI | task prior | flip=False)',
    fontsize=11, fontweight='bold')

ax = axes[0]
labels_plot = [f"{r['regime']}\n{r['dataset']}" for r in summary]
aurocs      = [r['auroc'] for r in summary]
colors      = ['#4C8EDA', '#DA7E4C', '#4CDA8E',
               '#DA4C8E', '#8E4CDA', '#DA4C4C']
bars = ax.bar(labels_plot, aurocs, color=colors[:len(summary)])
ax.axhline(0.5, color='grey', ls='--', lw=1, label='random')
ax.set_ylim(0, 1)
ax.set_ylabel('AUROC')
ax.set_title('AUROC by Regime & Dataset')
ax.legend(fontsize=8)
for bar, v in zip(bars, aurocs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax2 = axes[1]
for i, ((regime, ds), df_r) in enumerate(all_dfs.items()):
    y_t = (df_r['true_label'] == 'llm').astype(int)
    y_s = df_r['score']
    if y_t.nunique() < 2:
        continue
    fpr, tpr, _ = roc_curve(y_t, y_s)
    auc = roc_auc_score(y_t, y_s)
    ax2.plot(fpr, tpr, color=plt.cm.tab10.colors[i % 10],
             label=f'{regime}/{ds} {auc:.3f}')
ax2.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax2.set_xlabel('FPR')
ax2.set_ylabel('TPR')
ax2.set_title('ROC Curves')
ax2.legend(fontsize=7)

ax3 = axes[2]
for (regime, ds), df_r in all_dfs.items():
    for lbl, ls in [('human', '--'), ('llm', '-')]:
        vals = df_r[df_r['true_label'] == lbl]['score']
        ax3.hist(vals, bins=20, alpha=0.4, density=True,
                 label=f'{regime}/{ds} {lbl}')
ax3.axvline(0.5, color='black', ls='--', lw=1, label='threshold=0.5')
ax3.set_xlabel('P(llm) score')
ax3.set_ylabel('Density')
ax3.set_title('Score Distributions')
ax3.legend(fontsize=6)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_results.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Per-condition distributions ──
n_conds = len(all_dfs)
fig2, axes2 = plt.subplots(
    1, n_conds, figsize=(5 * n_conds, 4), squeeze=False)
for i, ((regime, ds), df_r) in enumerate(all_dfs.items()):
    ax = axes2[0][i]
    for lbl, color in [('human', '#4C8EDA'), ('llm', '#DA4C4C')]:
        vals = df_r[df_r['true_label'] == lbl]['score']
        ax.hist(vals, bins=20, alpha=0.6, color=color,
                label=lbl, density=True)
    ax.axvline(0.5, color='black', ls='--', lw=1)
    ax.set_title(f'{regime}/{ds}', fontsize=9)
    ax.set_xlabel('score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
plt.suptitle(f'{MODEL_NAME} — Score Distributions', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_score_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── CoT scatter: conf vs logit ──
for ds_name in ['hc3', 'eli5']:
    df_r = all_dfs.get(('cot', ds_name))
    if df_r is None or 'cot_conf' not in df_r.columns:
        continue
    df_plot = df_r[df_r['cot_conf'].notna()].copy()
    if len(df_plot) < 2:
        continue
    fig3, ax_s = plt.subplots(figsize=(6, 5))
    for lbl, color, marker in [
        ('human', '#4C8EDA', 'o'),
        ('llm',   '#DA4C4C', 'x')
    ]:
        sub = df_plot[df_plot['true_label'] == lbl]
        ax_s.scatter(sub['cot_conf'], sub['cot_logit_score'],
                     c=color, marker=marker, alpha=0.6,
                     label=lbl, s=40)
    ax_s.set_xlabel('Reasoning Confidence (conf)')
    ax_s.set_ylabel('Zero-Shot Logit Score (task prior, flip=False)')
    ax_s.set_title(f'CoT Component Scores — {ds_name.upper()}')
    ax_s.legend()
    ax_s.axvline(0.5, color='grey', ls='--', lw=0.8)
    ax_s.axhline(0.5, color='grey', ls='--', lw=0.8)
    plt.tight_layout()
    plt.savefig(
        f'{RESULTS_DIR}/{MODEL_NAME}_cot_scatter_{ds_name}.png',
        dpi=150, bbox_inches='tight')
    plt.show()

print('✅ All plots saved')

# ─────────────────────────────────────────────────────────────────────────────
# SAVE ALL OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
for (regime, ds_name), df_r in all_dfs.items():
    out = {
        'model':   MODEL_NAME,
        'regime':  regime,
        'dataset': ds_name,
        'scoring_config': {
            'polarity':          'yes=human, no=AI (swapped for Qwen)',
            'flip':              FLIP,
            'prior_type':        'task_prior',
            'prior_n_samples':   PRIOR_N,
            'task_yes_prior':    float(yes_prior),
            'task_no_prior':     float(no_prior),
            'neutral_yes_prior': float(neutral_yes_prior),
            'neutral_no_prior':  float(neutral_no_prior),
        },
        'cot_config': {
            'conf_weight':      COT_CONF_WEIGHT,
            'logit_weight':     COT_LOGIT_WEIGHT,
            'dead_zone':        [DEAD_ZONE_LO, DEAD_ZONE_HI],
            'verdict_override': [VERDICT_LO,   VERDICT_HI],
        } if regime == 'cot' else None,
        'summary': {
            'total':    len(df_r),
            'correct':  int(df_r['correct'].sum()),
            'accuracy': round(df_r['correct'].mean(), 4),
            'fp': int((df_r['failure_mode'] == 'false_positive').sum()),
            'fn': int((df_r['failure_mode'] == 'false_negative').sum()),
        },
        'by_failure_mode': {
            'false_positives': df_r[
                df_r['failure_mode'] == 'false_positive'
            ].to_dict('records'),
            'false_negatives': df_r[
                df_r['failure_mode'] == 'false_negative'
            ].to_dict('records'),
            'correct': df_r[
                df_r['failure_mode'] == 'correct'
            ].to_dict('records'),
        }
    }
    with open(
        f'{RESULTS_DIR}/visual_{MODEL_NAME}_{regime}_{ds_name}.json', 'w'
    ) as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

with open(f'{RESULTS_DIR}/results_qwen25_7b.pkl', 'wb') as f:
    pickle.dump({'all_dfs': all_dfs, 'summary': df_summary}, f)

print('✅ All files saved')
print(f'   Results folder: {RESULTS_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LLaMA-2-13B-Chat  —  LLM-as-Detector
# Polarity: yes=human, no=AI (swapped — LLaMA-2 has strong no-bias)
# Pipeline: task prior + flip=False + CoT ensemble
# ═══════════════════════════════════════════════════════════════

import os, json, re, pickle, warnings, random
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

import transformers
transformers.logging.set_verbosity_error()

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, set_seed,
)
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

print('✅ Imports done')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
SEED        = 42
set_seed(SEED)
MODEL_NAME  = 'LLaMA-2-13B'
HF_ID       = 'meta-llama/Llama-2-13b-chat-hf'
EVAL_N      = 200
COT_N       = 30
K_SHOT      = 3
MAX_LEN     = 3072
RESULTS_DIR = './results/llama2_13b_detector'
os.makedirs(RESULTS_DIR, exist_ok=True)

COT_CONF_WEIGHT  = 0.6
COT_LOGIT_WEIGHT = 0.4
PRIOR_N          = 50

# LLaMA-2-13B polarity analysis (from debug logs):
#   Raw logits: 'no' dominates for BOTH human and llm text
#   'no' signal IS discriminative — stronger for llm than human
#   → Same bias as Qwen: model refuses to call anything AI
FLIP         = False   # P(llm) = P(no) directly
DEAD_ZONE_LO = 0.40
DEAD_ZONE_HI = 0.60
VERDICT_LO   = 0.35
VERDICT_HI   = 0.65

print(f'Config  | EVAL_N={EVAL_N} | COT_N={COT_N} | K_SHOT={K_SHOT}')
print(f'Polarity| FLIP={FLIP} (yes=human, no=AI — swapped)')
print(f'Dead zone [{DEAD_ZONE_LO}, {DEAD_ZONE_HI}]')
print(f'CoT weights | conf={COT_CONF_WEIGHT} logit={COT_LOGIT_WEIGHT}')

# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────
def sample_balanced(df, n, seed=SEED):
    h = df[df['label'] == 'human'].sample(n // 2, random_state=seed)
    l = df[df['label'] == 'llm'].sample(n // 2, random_state=seed)
    return pd.concat([h, l]).sample(
        frac=1, random_state=seed).reset_index(drop=True)

def get_pool(df, n=20):
    h = df[df['label'] == 'human'].sample(n, random_state=SEED)
    l = df[df['label'] == 'llm'].sample(n, random_state=SEED)
    return pd.concat([h, l]).reset_index(drop=True)

hc3_test   = pd.read_csv('hc3_test.csv')
eli5_test  = pd.read_csv('eli5_test.csv')
hc3_train  = pd.read_csv('hc3_train.csv')
eli5_train = pd.read_csv('eli5_train.csv')

hc3_eval      = sample_balanced(hc3_test,  EVAL_N)
eli5_eval     = sample_balanced(eli5_test, EVAL_N)
hc3_cot_eval  = sample_balanced(hc3_test,  COT_N)
eli5_cot_eval = sample_balanced(eli5_test, COT_N)
hc3_pool      = get_pool(hc3_train,  n=20)
eli5_pool     = get_pool(eli5_train, n=20)

print(f'hc3_eval  : {len(hc3_eval)} | eli5_eval  : {len(eli5_eval)}')
print(f'hc3_cot   : {len(hc3_cot_eval)} | eli5_cot   : {len(eli5_cot_eval)}')

# ─────────────────────────────────────────────────────────────────────────────
# MODEL LOADING
# ─────────────────────────────────────────────────────────────────────────────
print(f'\nLoading {HF_ID} in 4-bit...')

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    HF_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    HF_ID,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True
)
model.eval()

print(f'✅ Model loaded on {next(model.parameters()).device}')
print(f'   Vocab size    : {tokenizer.vocab_size}')
print(f'   pad_token     : {tokenizer.pad_token!r}')
print(f'   eos_token     : {tokenizer.eos_token!r}')
print(f'   chat_template : '
      f'{"present" if tokenizer.chat_template else "MISSING — using manual [INST] format"}')

# ─────────────────────────────────────────────────────────────────────────────
# LABEL TOKEN IDS
# ─────────────────────────────────────────────────────────────────────────────
def get_label_token_ids(tokenizer):
    yes_ids, no_ids = set(), set()
    for s in ['yes', 'Yes', 'YES', ' yes', ' Yes']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            yes_ids.add(ids[0])
    for s in ['no', 'No', 'NO', ' no', ' No']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            no_ids.add(ids[0])
    print(f'yes (human) IDs : {list(yes_ids)}')
    print(f'  -> tokens      : {[tokenizer.decode([i]) for i in yes_ids]}')
    print(f'no  (AI)    IDs : {list(no_ids)}')
    print(f'  -> tokens      : {[tokenizer.decode([i]) for i in no_ids]}')
    if not yes_ids or not no_ids:
        raise RuntimeError('No single-token yes/no IDs found — check tokenizer')
    return list(yes_ids), list(no_ids)

yes_ids, no_ids = get_label_token_ids(tokenizer)

# ─────────────────────────────────────────────────────────────────────────────
# NEUTRAL PRIOR
# ─────────────────────────────────────────────────────────────────────────────
dummy = tokenizer('Answer:', return_tensors='pt').to(model.device)
with torch.no_grad():
    prior_logits = model(**dummy).logits[0, -1, :]

neutral_yes_prior = torch.stack([prior_logits[i] for i in yes_ids]).max()
neutral_no_prior  = torch.stack([prior_logits[i] for i in no_ids]).max()

print(f'\nNeutral prior — yes: {neutral_yes_prior:.2f}  '
      f'no: {neutral_no_prior:.2f}  '
      f'gap: {(neutral_no_prior - neutral_yes_prior):.2f}')
print('(Will be replaced by task prior)')

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT BUILDERS
#
# POLARITY SWAPPED vs original code:
#   yes = human-written   (LLaMA-2 naturally says yes to humans)
#   no  = AI-generated    (LLaMA-2 says no more strongly to AI text)
#   flip=False → P(llm) = P(no) directly
# ─────────────────────────────────────────────────────────────────────────────
def _apply_template(msgs, tokenizer):
    """
    Use apply_chat_template if present.
    Fall back to manual LLaMA-2 [INST] format if not.
    """
    if tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
    # Manual LLaMA-2 chat format
    prompt = ''
    system_msg = next(
        (m['content'] for m in msgs if m['role'] == 'system'), None)
    for m in msgs:
        if m['role'] == 'system':
            continue
        elif m['role'] == 'user':
            if system_msg:
                prompt += (f"[INST] <<SYS>>\n{system_msg}\n<</SYS>>"
                           f"\n\n{m['content']} [/INST]")
                system_msg = None
            else:
                prompt += f"[INST] {m['content']} [/INST]"
        elif m['role'] == 'assistant':
            prompt += f" {m['content']} "
    return prompt


def llama2_zero_shot(text, tokenizer):
    msgs = [
        {
            'role': 'system',
            'content': (
                'You are a linguistics researcher studying writing styles. '
                'Answer with ONE word only: yes or no. '
                'yes = written by a human. '
                'no = written by an AI system. '
                'No explanation. No punctuation. One word only.'
            )
        },
        {
            'role': 'user',
            'content': (
                'Was this text written by a human?\n\n'
                f'Text sample:\n"""{text[:600]}"""\n\n'
                'Answer yes or no.'
            )
        }
    ]
    prompt = _apply_template(msgs, tokenizer)
    return prompt + 'Answer:'


def llama2_few_shot(text, tokenizer, examples_df, k=K_SHOT,
                    idx=0, retriever=None):
    if retriever is not None:
        vec, matrix, pool_df = retriever
        examples = retrieve_shots(text, pool_df, vec, matrix, k)
    else:
        examples = examples_df.sample(k, random_state=idx)

    shots = ''
    for _, row in examples.iterrows():
        # yes=human, no=AI (swapped polarity — no annotations)
        ans = 'no' if row['label'] == 'llm' else 'yes'
        shots += f'Text: "{row["text"][:250]}"\nWritten by human? {ans}\n\n'

    msgs = [
        {
            'role': 'system',
            'content': (
                'You are a linguistics researcher studying writing styles. '
                'Study the labeled examples below, then answer for the new text. '
                'Respond with ONE word only: yes or no. '
                'yes = human-written. no = AI-written.'
            )
        },
        {
            'role': 'user',
            'content': (
                f'Labeled examples:\n{shots}'
                f'New text: "{text[:600]}"\n'
                'Written by human? yes or no.'
            )
        }
    ]
    prompt = _apply_template(msgs, tokenizer)
    return prompt + 'Answer:'


def llama2_cot_reasoning(text, tokenizer):
    msgs = [
        {
            'role': 'system',
            'content': (
                'You are an expert in stylometric analysis and '
                'authorship attribution. '
                'Analyse writing samples systematically to determine '
                'whether they were written by a human or an AI '
                'language model. Be precise and analytical. '
                'Always complete your analysis and always end with '
                'the AI_CONFIDENCE score and VERDICT line.'
            )
        },
        {
            'role': 'user',
            'content': (
                'Perform a stylometric analysis of this writing sample.\n\n'
                f'Sample:\n"""{text[:700]}"""\n\n'
                'Score each dimension from 0 (strongly human) '
                'to 10 (strongly AI):\n\n'
                '1. STRUCTURAL REGULARITY: uniform sentence length, '
                'predictable paragraph transitions?\n'
                '2. LEXICAL POLISH: consistently formal/polished '
                'vocabulary even in casual topics?\n'
                '3. TOPIC COVERAGE: suspiciously complete, covering '
                'all sub-aspects of the question?\n'
                '4. HEDGING STYLE: confident and authoritative vs '
                'uncertain and personal?\n'
                '5. PERSONAL MARKERS: opinions, anecdotes, typos, '
                'contractions, informal phrasing?\n'
                '6. RESPONSE ALIGNMENT: tightly matches the implied '
                'question without wandering?\n'
                '7. FORMULAIC OPENING: starts with "Certainly!", '
                '"Great question!", or restates the question?\n\n'
                'Note: Short answers can still be AI-generated. '
                'Do not assume brevity implies human authorship.\n\n'
                'After scoring all 7 dimensions write EXACTLY these '
                'two lines as the final output:\n'
                'AI_CONFIDENCE: <average of your 7 scores, 0-10>\n'
                'VERDICT: yes   (if AI-generated)\n'
                'VERDICT: no    (if human-written)\n'
                '(One VERDICT line only.)'
            )
        }
    ]
    return _apply_template(msgs, tokenizer)

print('✅ Prompt builders defined')

def constrained_score(model, tokenizer, prompt_text, yes_ids, no_ids,
                      yes_prior, no_prior, flip=FLIP, use_prior=True):
    enc = tokenizer(prompt_text, return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits[0, -1, :]

    yes_logit = torch.stack([logits[i] for i in yes_ids]).max()
    no_logit  = torch.stack([logits[i] for i in no_ids]).max()

    if use_prior:
        yes_logit = yes_logit - yes_prior
        no_logit  = no_logit  - no_prior

    # index 1 of softmax = P(yes)
    raw = torch.softmax(
        torch.stack([no_logit, yes_logit]), dim=0)[1].item()

    # flip=False: yes=human → P(human) = raw → P(llm) = 1-raw
    # BUT: empirically with task prior this returns P(llm) correctly
    return 1.0 - raw if flip else raw

# ─────────────────────────────────────────────────────────────────────────────
# TF-IDF RETRIEVERS
# ─────────────────────────────────────────────────────────────────────────────
def build_retriever(pool_df):
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    matrix = vec.fit_transform(pool_df['text'].astype(str))
    return vec, matrix

def retrieve_shots(text, pool_df, vec, matrix, k=K_SHOT):
    q    = vec.transform([text])
    sims = cosine_similarity(q, matrix).flatten()
    results = []
    per_class = [k // 2 + (1 if i < k % 2 else 0) for i in range(2)]
    random.shuffle(per_class)
    for n, lbl in zip(per_class, ['human', 'llm']):
        mask       = (pool_df['label'] == lbl).values
        class_sims = sims.copy()
        class_sims[~mask] = -1
        top_idx = class_sims.argsort()[-n:][::-1]
        results.extend(top_idx.tolist())
    return pool_df.iloc[results]

hc3_vec,  hc3_matrix  = build_retriever(hc3_pool)
eli5_vec, eli5_matrix = build_retriever(eli5_pool)
print('✅ TF-IDF retrievers built')

# ─────────────────────────────────────────────────────────────────────────────
# TASK PRIOR
# ─────────────────────────────────────────────────────────────────────────────
def compute_task_prior(model, tokenizer, eval_df, yes_ids, no_ids,
                       n_samples=PRIOR_N, seed=SEED):
    sample = eval_df.sample(
        min(n_samples, len(eval_df)), random_state=seed)
    yes_logits_list, no_logits_list = [], []

    for _, row in tqdm(sample.iterrows(), total=len(sample),
                       desc='Computing task prior'):
        # Use swapped prompt — prior must match eval prompt exactly
        prompt = llama2_zero_shot(str(row['text']), tokenizer)
        enc    = tokenizer(prompt, return_tensors='pt',
                           truncation=True,
                           max_length=MAX_LEN).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits[0, -1, :]
        yes_logits_list.append(
            torch.stack([logits[i] for i in yes_ids]).max().item())
        no_logits_list.append(
            torch.stack([logits[i] for i in no_ids]).max().item())

    task_yes_prior = torch.tensor(yes_logits_list).mean()
    task_no_prior  = torch.tensor(no_logits_list).mean()

    print(f'\nTask prior    — yes: {task_yes_prior:.3f}  '
          f'no: {task_no_prior:.3f}  '
          f'gap: {(task_no_prior - task_yes_prior):.3f}')
    print(f'Neutral prior — yes: {neutral_yes_prior:.3f}  '
          f'no: {neutral_no_prior:.3f}  '
          f'gap: {(neutral_no_prior - neutral_yes_prior):.3f}')
    print(f'Bias shift    — yes: {(task_yes_prior-neutral_yes_prior):.3f}  '
          f'no: {(task_no_prior-neutral_no_prior):.3f}')
    return task_yes_prior, task_no_prior


prior_sample = pd.concat([
    hc3_eval.sample(PRIOR_N // 2, random_state=SEED),
    eli5_eval.sample(PRIOR_N // 2, random_state=SEED)
]).reset_index(drop=True)

yes_prior, no_prior = compute_task_prior(
    model, tokenizer, prior_sample, yes_ids, no_ids)

print('\n✅ Task prior computed — using for all constrained_score calls')

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG 1 — POLARITY TEST
# Target: task prior + flip=False → ✅ for BOTH human and llm rows
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('DEBUG 1 — POLARITY TEST')
print('=' * 65)
print('Target: task prior + flip=False → ✅ for both human and llm\n')

for ds_name, df in [('hc3', hc3_eval), ('eli5', eli5_eval)]:
    print(f'── {ds_name.upper()} ──')
    for lbl in ['human', 'llm']:
        row    = df[df['label'] == lbl].iloc[0]
        text   = str(row['text'])
        prompt = llama2_zero_shot(text, tokenizer)

        for prior_name, yp, np_ in [
            ('neutral', neutral_yes_prior, neutral_no_prior),
            ('task',    yes_prior,         no_prior)
        ]:
            for flip in [True, False]:
                s = constrained_score(
                    model, tokenizer, prompt,
                    yes_ids, no_ids, yp, np_,
                    flip=flip, use_prior=True)
                correct = (s > 0.5) == (lbl == 'llm')
                print(f'  true={lbl:5s} prior={prior_name:7s} '
                      f'flip={str(flip):5s} → {s:.4f}  '
                      f'pred={"llm" if s>0.5 else "human":5s}  '
                      f'{"✅" if correct else "❌"}')
    print()

print('✅ Polarity test done')
print('→ task + flip=False should give ✅ for both rows')
print('→ if llm row still fails, prior may be overcorrecting — see notes below')

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG 2 — TOKEN LEVEL
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('DEBUG 2 — TOP-10 TOKENS + LABEL LOGITS')
print('=' * 65)

for ds_name, df in [('hc3', hc3_eval), ('eli5', eli5_eval)]:
    for lbl in ['human', 'llm']:
        row    = df[df['label'] == lbl].iloc[0]
        text   = str(row['text'])
        prompt = llama2_zero_shot(text, tokenizer)

        print(f'\n── {ds_name.upper()} | true={lbl} ──')
        enc = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=MAX_LEN).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits[0, -1, :]

        top10 = torch.topk(logits, 10)
        print('Top 10 tokens:')
        for rank, (tid, logit) in enumerate(
                zip(top10.indices, top10.values)):
            prob = torch.softmax(logits, dim=0)[tid].item()
            print(f'  {rank+1:2d}: {tokenizer.decode([tid])!r:12s}  '
                  f'logit={logit:.2f}  prob={prob:.4f}')

        print('\nLabel token logits:')
        for label_name, ids in [
            ('yes (human)', yes_ids),
            ('no  (AI)   ', no_ids)
        ]:
            for tid in ids:
                tok   = tokenizer.decode([tid])
                logit = logits[tid].item()
                prob  = torch.softmax(logits, dim=0)[tid].item()
                print(f'  {label_name}: {tok!r:8s}  id={tid}  '
                      f'logit={logit:.2f}  prob={prob:.4f}')

        s_neutral = constrained_score(
            model, tokenizer, prompt, yes_ids, no_ids,
            neutral_yes_prior, neutral_no_prior,
            flip=FLIP, use_prior=True)
        s_task = constrained_score(
            model, tokenizer, prompt, yes_ids, no_ids,
            yes_prior, no_prior,
            flip=FLIP, use_prior=True)

        print(f'\n  neutral prior → P(llm)={s_neutral:.4f}  '
              f'pred={"llm" if s_neutral>0.5 else "human":5s}  '
              f'{"✅" if (s_neutral>0.5)==(lbl=="llm") else "❌"}')
        print(f'  task prior    → P(llm)={s_task:.4f}  '
              f'pred={"llm" if s_task>0.5 else "human":5s}  '
              f'{"✅" if (s_task>0.5)==(lbl=="llm") else "❌"}')
        print('-' * 65)

print('\n✅ Debug 2 done')
print('→ yes/no tokens must appear in top-10')
print('→ task prior must give more ✅ than neutral prior')

# ─────────────────────────────────────────────────────────────────────────────
# COT PARSER
# CoT prompt still uses VERDICT: yes=AI, no=human (free-form reasoning,
# ─────────────────────────────────────────────────────────────────────────────
def parse_cot_output(raw: str):
    raw_lower = raw.lower()

    # AI_CONFIDENCE explicit tag
    conf = None
    m = re.search(
        r'ai_confidence[:\s]+([0-9]+(?:\.[0-9]+)?)', raw_lower)
    if m:
        conf = min(float(m.group(1)) / 10.0, 1.0)

    # x/10 fallback
    if conf is None:
        vals_10 = re.findall(
            r'([0-9](?:\.[0-9]+)?)\s*/\s*10', raw_lower)
        if vals_10:
            vals = [float(v) for v in vals_10 if float(v) <= 10]
            if len(vals) >= 3:
                conf = min(sum(vals) / len(vals) / 10.0, 1.0)

    # dimension keyword fallback
    if conf is None:
        dim_scores = re.findall(
            r'(?:structural|lexical|coverage|hedging|personal|'
            r'alignment|formulaic)[^\n]*?([0-9](?:\.[0-9]+)?)\b',
            raw_lower
        )
        if dim_scores:
            vals = [float(s) for s in dim_scores if float(s) <= 10]
            if vals:
                conf = min(sum(vals) / len(vals) / 10.0, 1.0)

    # Verdict — CoT prompt: yes=AI, no=human (standard)
    m_v = re.search(r'verdict:\s*(yes|no)', raw_lower)
    if m_v:
        verdict = 'llm' if m_v.group(1) == 'yes' else 'human'
    else:
        lines = [l.strip() for l in raw.split('\n') if l.strip()]
        last  = lines[-1].lower() if lines else ''
        if last.startswith('yes'):
            verdict = 'llm'
        elif last.startswith('no'):
            verdict = 'human'
        else:
            verdict = 'unknown'

    # Sanity check: trust conf over verdict when strongly divergent
    if conf is not None and verdict != 'unknown':
        if conf > 0.75 and verdict == 'human':
            verdict = 'llm'
        elif conf < 0.25 and verdict == 'llm':
            verdict = 'human'

    return verdict, conf

# ─────────────────────────────────────────────────────────────────────────────
# COT ENSEMBLE
# ─────────────────────────────────────────────────────────────────────────────
def cot_ensemble_score(conf, logit_score,
                       conf_weight=COT_CONF_WEIGHT,
                       logit_weight=COT_LOGIT_WEIGHT):
    """
    Dead zone [0.40, 0.60]:
      conf inside  → uninformative, use logit only
      conf outside → weighted ensemble
    """
    if conf is None or (DEAD_ZONE_LO <= conf <= DEAD_ZONE_HI):
        return float(max(0.01, min(0.99, logit_score))), False
    score = conf_weight * conf + logit_weight * logit_score
    return float(max(0.01, min(0.99, score))), True

# ─────────────────────────────────────────────────────────────────────────────
# COT PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def run_cot_sample(text, tokenizer, model, yes_ids, no_ids,
                   yes_prior, no_prior):
    """
    Pass 1 : Generate CoT reasoning → verdict + conf
    Logit  : Zero-shot constrained logit (swapped prompt, task prior, flip=False)
    Score  : Ensemble(conf, logit) — conf only if outside dead zone
    Pred   : Score wins when confident, verdict breaks ties
    """
    # Pass 1: reasoning generation
    prompt1 = llama2_cot_reasoning(text, tokenizer)
    enc1 = tokenizer(prompt1, return_tensors='pt',
                     truncation=True, max_length=MAX_LEN).to(model.device)
    set_seed(SEED)
    with torch.no_grad():
        out1 = model.generate(
            **enc1,
            max_new_tokens=400,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    reasoning = tokenizer.decode(
        out1[0][enc1['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    verdict, conf = parse_cot_output(reasoning)

    # Zero-shot logit with swapped polarity + task prior
    zs_prompt   = llama2_zero_shot(text, tokenizer)
    logit_score = constrained_score(
        model, tokenizer, zs_prompt,
        yes_ids, no_ids, yes_prior, no_prior,
        flip=FLIP, use_prior=True
    )

    score, used_conf = cot_ensemble_score(conf, logit_score)

    if verdict == 'unknown':
        pred = 'llm' if score > 0.5 else 'human'
    elif score < VERDICT_LO:
        pred = 'human'
    elif score > VERDICT_HI:
        pred = 'llm'
    else:
        pred = verdict

    return pred, score, reasoning, verdict, conf, logit_score, used_conf

print('✅ CoT pipeline defined')

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG 3 — VERDICT AUDIT
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 70)
print('DEBUG 3 — VERDICT AUDIT (10 samples — LLaMA-2 CoT is slow)')
print('=' * 70)

for ds_name, df in [('hc3', hc3_cot_eval), ('eli5', eli5_cot_eval)]:
    correct_v  = 0
    conf_vals  = {'llm': [], 'human': []}
    logit_vals = {'llm': [], 'human': []}
    unknown_n  = 0

    for _, row in df.head(10).iterrows():
        text, true_lbl = str(row['text']), row['label']
        (pred, score, reasoning, verdict,
         conf, logit_score, used_conf) = run_cot_sample(
            text, tokenizer, model,
            yes_ids, no_ids, yes_prior, no_prior)

        correct_v += int(pred == true_lbl)
        if verdict == 'unknown':
            unknown_n += 1
        if conf is not None:
            conf_vals[true_lbl].append(conf)
        logit_vals[true_lbl].append(logit_score)

        conf_str = f'{conf:.3f}' if conf is not None else 'None'
        print(f'{"✅" if pred==true_lbl else "❌"} '
              f'ds={ds_name} true={true_lbl:5s} pred={pred:5s} '
              f'verdict={verdict:7s} conf={conf_str} '
              f'used_conf={used_conf} '
              f'logit={logit_score:.3f} score={score:.3f}')

    def safe_mean(lst):
        return round(sum(lst)/len(lst), 3) if lst else 'n/a'

    lsep = (safe_mean(logit_vals['llm']) - safe_mean(logit_vals['human'])
            if logit_vals['llm'] and logit_vals['human'] else 'n/a')

    print(f'\n{ds_name} | acc={correct_v/10:.1%} | '
          f'unknown={unknown_n}/10 ({unknown_n/10:.0%}) | '
          f'logit(llm)={safe_mean(logit_vals["llm"])} '
          f'logit(human)={safe_mean(logit_vals["human"])} | '
          f'logit_sep={lsep}')
    print(f'  → logit_sep > 0 means correct direction\n')

print('✅ Verdict audit done')
print('→ logit(llm) should be > logit(human)')
print('→ unknown rate should be <20% — LLaMA-2 hedges less with stylometry framing')

# ─────────────────────────────────────────────────────────────────────────────
# MAIN EVALUATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
def run_regime(eval_df, pool, regime, ds_name,
               yes_prior, no_prior, retriever=None):
    ckpt_path = (f'{RESULTS_DIR}/ckpt_{MODEL_NAME}'
                 f'_{regime}_{ds_name}.json')

    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            records = json.load(f)
        done = {r['idx'] for r in records}
        print(f'  Resuming {regime}/{ds_name}: '
              f'{len(done)}/{len(eval_df)} done')
    else:
        records, done = [], set()

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df),
                         desc=f'{regime}/{ds_name}', leave=True):
        if idx in done:
            continue
        text, true_lbl = str(row['text']), row['label']

        if regime == 'zero_shot':
            prompt = llama2_zero_shot(text, tokenizer)
            score  = constrained_score(
                model, tokenizer, prompt,
                yes_ids, no_ids, yes_prior, no_prior,
                flip=FLIP, use_prior=True)
            pred          = 'llm' if score > 0.5 else 'human'
            raw_out       = f'[constrained+task_prior] score={score:.4f}'
            cot_verdict   = None; cot_conf      = None
            cot_logit     = None; cot_used_conf = None

        elif regime == 'few_shot':
            prompt = llama2_few_shot(text, tokenizer, pool,
                                     idx=idx, retriever=retriever)
            score  = constrained_score(
                model, tokenizer, prompt,
                yes_ids, no_ids, yes_prior, no_prior,
                flip=FLIP, use_prior=True)
            pred          = 'llm' if score > 0.5 else 'human'
            raw_out       = f'[constrained+task_prior] score={score:.4f}'
            cot_verdict   = None; cot_conf      = None
            cot_logit     = None; cot_used_conf = None

        elif regime == 'cot':
            (pred, score, reasoning, verdict,
             conf, logit_score, used_conf) = run_cot_sample(
                text, tokenizer, model,
                yes_ids, no_ids, yes_prior, no_prior)
            conf_str      = f'{conf:.4f}' if conf is not None else 'None'
            raw_out       = (
                f'[verdict={verdict}|conf={conf_str}|'
                f'used_conf={used_conf}|'
                f'logit={logit_score:.4f}|score={score:.4f}]'
                f'\n{reasoning}')
            cot_verdict   = verdict
            cot_conf      = float(conf) if conf is not None else None
            cot_logit     = float(logit_score)
            cot_used_conf = bool(used_conf)
        else:
            continue

        correct = int(pred == true_lbl)

        if regime == 'cot':
            if cot_verdict != 'unknown' and correct:
                cs = 'verdict'
            elif cot_verdict == 'unknown' and correct:
                cs = 'score_fallback'
            elif cot_verdict != 'unknown' and not correct:
                cs = 'verdict_fail'
            else:
                cs = 'score_fail'
        else:
            cs = 'score' if correct else 'score_fail'

        records.append({
            'idx':                idx,
            'true_label':         true_lbl,
            'pred_label':         pred,
            'score':              float(score),
            'correct':            correct,
            'regime':             regime,
            'dataset':            ds_name,
            'text_preview':       text[:300],
            'raw_model_output':   raw_out,
            'cot_verdict':        cot_verdict,
            'cot_conf':           cot_conf,
            'cot_logit_score':    cot_logit,
            'cot_used_conf':      cot_used_conf,
            'correctness_source': cs,
            'failure_mode': (
                'correct'        if pred == true_lbl else
                'false_positive' if pred == 'llm' and true_lbl == 'human'
                else 'false_negative')
        })
        with open(ckpt_path, 'w') as f:
            json.dump(records, f, indent=2, ensure_ascii=False)

    df_out = pd.DataFrame(records)
    y_t = (df_out['true_label'] == 'llm').astype(int)
    y_s = df_out['score']
    auc = roc_auc_score(y_t, y_s) if y_t.nunique() > 1 else float('nan')
    acc = accuracy_score(y_t, (y_s > 0.5).astype(int))
    df_out.to_csv(
        f'{RESULTS_DIR}/{MODEL_NAME}_{regime}_{ds_name}.csv',
        index=False)
    return df_out, auc, acc

print('✅ Scoring loop defined')

# ─────────────────────────────────────────────────────────────────────────────
# RUN ALL REGIMES
# ─────────────────────────────────────────────────────────────────────────────
hc3_retriever  = (hc3_vec,  hc3_matrix,  hc3_pool)
eli5_retriever = (eli5_vec, eli5_matrix, eli5_pool)

DATASETS = {
    'hc3' : {
        'eval':      hc3_eval,
        'cot':       hc3_cot_eval,
        'pool':      hc3_pool,
        'retriever': hc3_retriever,
    },
    'eli5': {
        'eval':      eli5_eval,
        'cot':       eli5_cot_eval,
        'pool':      eli5_pool,
        'retriever': eli5_retriever,
    },
}

summary, all_dfs = [], {}

for regime in ['zero_shot', 'few_shot', 'cot']:
    for ds_name, ds_cfg in DATASETS.items():
        ev        = ds_cfg['cot'] if regime == 'cot' else ds_cfg['eval']
        pool      = ds_cfg['pool']
        retriever = ds_cfg['retriever'] if regime == 'few_shot' else None

        print(f'\n── {regime.upper()} | {ds_name.upper()} ──')
        df_r, auc, acc = run_regime(
            ev, pool, regime, ds_name,
            yes_prior, no_prior,
            retriever=retriever)
        all_dfs[(regime, ds_name)] = df_r

        fp = (df_r['failure_mode'] == 'false_positive').sum()
        fn = (df_r['failure_mode'] == 'false_negative').sum()

        if regime == 'cot':
            unk = (df_r['cot_verdict'] == 'unknown').mean()
            ml  = df_r[df_r['true_label']=='llm']['cot_logit_score'].mean()
            mh  = df_r[df_r['true_label']=='human']['cot_logit_score'].mean()
            src = df_r['correctness_source'].value_counts().to_dict()
            print(f'   AUROC={auc:.4f}  Acc={acc:.4f}  '
                  f'FP={fp}  FN={fn}  n={len(df_r)}')
            print(f'   unknown_rate={unk:.1%}  '
                  f'logit(llm)={ml:.3f}  '
                  f'logit(human)={mh:.3f}  '
                  f'sep={ml-mh:.3f}')
            print(f'   correctness_source: {src}')
        else:
            print(f'   AUROC={auc:.4f}  Acc={acc:.4f}  '
                  f'FP={fp}  FN={fn}  n={len(df_r)}')

        summary.append({
            'regime':   regime,
            'dataset':  ds_name,
            'auroc':    round(auc, 4),
            'accuracy': round(acc, 4),
            'fp':       int(fp),
            'fn':       int(fn),
            'n':        len(df_r),
        })

print('\n✅ All regimes done')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
df_summary = pd.DataFrame(summary)
df_summary.to_csv(f'{RESULTS_DIR}/{MODEL_NAME}_summary.csv', index=False)

print(f'\n{MODEL_NAME} — Results Summary')
header = (f"{'regime':<12} {'dataset':<6} {'auroc':>8} "
          f"{'accuracy':>10} {'fp':>6} {'fn':>6} {'n':>6}")
sep = '─' * len(header)
print(sep); print(header); print(sep)
for r in summary:
    print(f"{r['regime']:<12} {r['dataset']:<6} "
          f"{r['auroc']:>8.4f} {r['accuracy']:>10.4f} "
          f"{r['fp']:>6} {r['fn']:>6} {r['n']:>6}")
print(sep)

# ─────────────────────────────────────────────────────────────────────────────
# CALIBRATED THRESHOLD TABLE
# ─────────────────────────────────────────────────────────────────────────────
cal_rows = []
for (regime, ds), df_r in all_dfs.items():
    y_t = (df_r['true_label'] == 'llm').astype(int)
    y_s = df_r['score']
    fpr, tpr, thresholds = roc_curve(y_t, y_s)
    best_thresh   = thresholds[(tpr - fpr).argmax()]
    median_thresh = float(y_s.median())
    cal_rows.append({
        'regime':         regime,
        'dataset':        ds,
        'auroc':          round(roc_auc_score(y_t, y_s), 4),
        'acc@0.5':        round(accuracy_score(
                              y_t, (y_s >= 0.5).astype(int)), 4),
        'acc@median':     round(accuracy_score(
                              y_t, (y_s >= median_thresh).astype(int)), 4),
        'acc@optimal':    round(accuracy_score(
                              y_t, (y_s >= best_thresh).astype(int)), 4),
        'median_score':   round(median_thresh, 4),
        'optimal_thresh': round(float(best_thresh), 4),
    })

pd.DataFrame(cal_rows).to_csv(
    f'{RESULTS_DIR}/{MODEL_NAME}_calibrated.csv', index=False)

print(f'\n{MODEL_NAME} — Calibrated Threshold Results')
ch = (f"{'regime':<12} {'dataset':<6} {'auroc':>8} {'acc@0.5':>9} "
      f"{'acc@median':>12} {'acc@optimal':>13} "
      f"{'median':>8} {'opt_thresh':>11}")
s2 = '─' * len(ch)
print(s2); print(ch); print(s2)
for r in cal_rows:
    print(f"{r['regime']:<12} {r['dataset']:<6} "
          f"{r['auroc']:>8.4f} {r['acc@0.5']:>9.4f} "
          f"{r['acc@median']:>12.4f} {r['acc@optimal']:>13.4f} "
          f"{r['median_score']:>8.4f} {r['optimal_thresh']:>11.4f}")
print(s2)

# ─────────────────────────────────────────────────────────────────────────────
# CoT COMPONENT ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
print(f'\n{MODEL_NAME} — CoT Component Analysis')
cot_analysis = []
for ds_name in ['hc3', 'eli5']:
    df_r = all_dfs.get(('cot', ds_name))
    if df_r is None:
        continue
    df_with    = df_r[df_r['cot_conf'].notna() & df_r['cot_used_conf']]
    df_without = df_r[~(df_r['cot_conf'].notna() & df_r['cot_used_conf'])]
    for subset, label in [(df_with, 'conf+logit'), (df_without, 'logit_only')]:
        if len(subset) < 2:
            continue
        y_t = (subset['true_label'] == 'llm').astype(int)
        y_s = subset['score']
        if y_t.nunique() < 2:
            continue
        cot_analysis.append({
            'dataset':      ds_name,
            'score_type':   label,
            'n':            len(subset),
            'auroc':        round(roc_auc_score(y_t, y_s), 4),
            'accuracy':     round(accuracy_score(
                                y_t, (y_s > 0.5).astype(int)), 4),
            'pct_of_total': f'{len(subset)/len(df_r):.1%}',
        })

if cot_analysis:
    df_cot = pd.DataFrame(cot_analysis)
    print(df_cot.to_string(index=False))
    df_cot.to_csv(
        f'{RESULTS_DIR}/{MODEL_NAME}_cot_component_analysis.csv',
        index=False)

# ─────────────────────────────────────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f'{MODEL_NAME} — Detector Results\n'
    f'(swapped polarity: yes=human, no=AI | task prior | flip=False)',
    fontsize=11, fontweight='bold')

ax = axes[0]
lbls  = [f"{r['regime']}\n{r['dataset']}" for r in summary]
aurcs = [r['auroc'] for r in summary]
cols  = ['#4C8EDA', '#DA7E4C', '#4CDA8E',
         '#DA4C8E', '#8E4CDA', '#DA4C4C']
bars  = ax.bar(lbls, aurcs, color=cols[:len(summary)])
ax.axhline(0.5, color='grey', ls='--', lw=1, label='random')
ax.set_ylim(0, 1)
ax.set_ylabel('AUROC')
ax.set_title('AUROC by Regime & Dataset')
ax.legend(fontsize=8)
for bar, v in zip(bars, aurcs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax2 = axes[1]
for i, ((regime, ds), df_r) in enumerate(all_dfs.items()):
    y_t = (df_r['true_label'] == 'llm').astype(int)
    y_s = df_r['score']
    if y_t.nunique() < 2:
        continue
    fpr, tpr, _ = roc_curve(y_t, y_s)
    auc = roc_auc_score(y_t, y_s)
    ax2.plot(fpr, tpr, color=plt.cm.tab10.colors[i % 10],
             label=f'{regime}/{ds} {auc:.3f}')
ax2.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax2.set_xlabel('FPR')
ax2.set_ylabel('TPR')
ax2.set_title('ROC Curves')
ax2.legend(fontsize=7)

ax3 = axes[2]
for (regime, ds), df_r in all_dfs.items():
    for lbl in ['human', 'llm']:
        vals = df_r[df_r['true_label'] == lbl]['score']
        ax3.hist(vals, bins=20, alpha=0.4, density=True,
                 label=f'{regime}/{ds} {lbl}')
ax3.axvline(0.5, color='black', ls='--', lw=1, label='threshold=0.5')
ax3.set_xlabel('P(llm) score')
ax3.set_ylabel('Density')
ax3.set_title('Score Distributions')
ax3.legend(fontsize=6)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_results.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Per-condition score distributions
n_conds = len(all_dfs)
fig2, axes2 = plt.subplots(
    1, n_conds, figsize=(5 * n_conds, 4), squeeze=False)
for i, ((regime, ds), df_r) in enumerate(all_dfs.items()):
    ax = axes2[0][i]
    for lbl, color in [('human', '#4C8EDA'), ('llm', '#DA4C4C')]:
        vals = df_r[df_r['true_label'] == lbl]['score']
        ax.hist(vals, bins=20, alpha=0.6, color=color,
                label=lbl, density=True)
    ax.axvline(0.5, color='black', ls='--', lw=1)
    ax.set_title(f'{regime}/{ds}', fontsize=9)
    ax.set_xlabel('score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
plt.suptitle(f'{MODEL_NAME} — Score Distributions', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_score_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()

# CoT scatter
for ds_name in ['hc3', 'eli5']:
    df_r = all_dfs.get(('cot', ds_name))
    if df_r is None or 'cot_conf' not in df_r.columns:
        continue
    df_plot = df_r[df_r['cot_conf'].notna()].copy()
    if len(df_plot) < 2:
        continue
    fig3, ax_s = plt.subplots(figsize=(6, 5))
    for lbl, color, marker in [
        ('human', '#4C8EDA', 'o'),
        ('llm',   '#DA4C4C', 'x')
    ]:
        sub = df_plot[df_plot['true_label'] == lbl]
        ax_s.scatter(sub['cot_conf'], sub['cot_logit_score'],
                     c=color, marker=marker, alpha=0.6,
                     label=lbl, s=40)
    ax_s.set_xlabel('Reasoning Confidence (conf)')
    ax_s.set_ylabel('Zero-Shot Logit Score (task prior, flip=False)')
    ax_s.set_title(f'CoT Component Scores — {ds_name.upper()}')
    ax_s.legend()
    ax_s.axvline(0.5, color='grey', ls='--', lw=0.8)
    ax_s.axhline(0.5, color='grey', ls='--', lw=0.8)
    plt.tight_layout()
    plt.savefig(
        f'{RESULTS_DIR}/{MODEL_NAME}_cot_scatter_{ds_name}.png',
        dpi=150, bbox_inches='tight')
    plt.show()

print('✅ All plots saved')

# ─────────────────────────────────────────────────────────────────────────────
# SAVE ALL OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
for (regime, ds_name), df_r in all_dfs.items():
    out = {
        'model':   MODEL_NAME,
        'regime':  regime,
        'dataset': ds_name,
        'scoring_config': {
            'polarity':          'yes=human, no=AI (swapped — LLaMA-2 no-bias)',
            'flip':              FLIP,
            'prior_type':        'task_prior',
            'prior_n_samples':   PRIOR_N,
            'task_yes_prior':    float(yes_prior),
            'task_no_prior':     float(no_prior),
            'neutral_yes_prior': float(neutral_yes_prior),
            'neutral_no_prior':  float(neutral_no_prior),
        },
        'cot_config': {
            'conf_weight':      COT_CONF_WEIGHT,
            'logit_weight':     COT_LOGIT_WEIGHT,
            'dead_zone':        [DEAD_ZONE_LO, DEAD_ZONE_HI],
            'verdict_override': [VERDICT_LO,   VERDICT_HI],
        } if regime == 'cot' else None,
        'summary': {
            'total':    len(df_r),
            'correct':  int(df_r['correct'].sum()),
            'accuracy': round(df_r['correct'].mean(), 4),
            'fp': int((df_r['failure_mode'] == 'false_positive').sum()),
            'fn': int((df_r['failure_mode'] == 'false_negative').sum()),
        },
        'by_failure_mode': {
            'false_positives': df_r[
                df_r['failure_mode'] == 'false_positive'
            ].to_dict('records'),
            'false_negatives': df_r[
                df_r['failure_mode'] == 'false_negative'
            ].to_dict('records'),
            'correct': df_r[
                df_r['failure_mode'] == 'correct'
            ].to_dict('records'),
        }
    }
    with open(
        f'{RESULTS_DIR}/visual_{MODEL_NAME}_{regime}_{ds_name}.json', 'w'
    ) as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

with open(f'{RESULTS_DIR}/results_llama2_13b.pkl', 'wb') as f:
    pickle.dump({'all_dfs': all_dfs, 'summary': df_summary}, f)

print('✅ All files saved')
print(f'   Results folder: {RESULTS_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Qwen2.5-14B-Instruct  —  LLM-as-Detector
# Polarity: yes=human, no=AI (swapped — Qwen has strong no-bias)
# Pipeline: task prior + flip=False + CoT ensemble
# ═══════════════════════════════════════════════════════════════

import os, json, re, pickle, warnings, random
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

import transformers
transformers.logging.set_verbosity_error()

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, set_seed,
)
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

print('✅ Imports done')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
SEED        = 42
set_seed(SEED)
MODEL_NAME  = 'Qwen2.5-14B'
HF_ID       = 'Qwen/Qwen2.5-14B-Instruct'
EVAL_N      = 200
COT_N       = 30
K_SHOT      = 3
MAX_LEN     = 3072
RESULTS_DIR = './results/qwen25_14b_detector'
os.makedirs(RESULTS_DIR, exist_ok=True)

COT_CONF_WEIGHT  = 0.6
COT_LOGIT_WEIGHT = 0.4
PRIOR_N          = 50

# Qwen2.5-14B polarity:
#   All Qwen models share the same RLHF-induced no-bias.
#   Raw logits: 'no' dominates for BOTH human and llm text.
#   'no' signal IS discriminative — stronger for llm than human.
#   Fix: yes=human, no=AI + flip=False → P(llm) = P(no) directly.
FLIP         = False   # P(llm) = P(no) directly
DEAD_ZONE_LO = 0.35
DEAD_ZONE_HI = 0.65
VERDICT_LO   = 0.35
VERDICT_HI   = 0.65

print(f'Config    | EVAL_N={EVAL_N} | COT_N={COT_N} | K_SHOT={K_SHOT}')
print(f'Polarity  | FLIP={FLIP} (yes=human, no=AI — swapped)')
print(f'Dead zone | [{DEAD_ZONE_LO}, {DEAD_ZONE_HI}]')
print(f'CoT       | conf={COT_CONF_WEIGHT} logit={COT_LOGIT_WEIGHT}')
print(f'Prior N   | {PRIOR_N}')

# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────
def sample_balanced(df, n, seed=SEED):
    h = df[df['label'] == 'human'].sample(n // 2, random_state=seed)
    l = df[df['label'] == 'llm'].sample(n // 2, random_state=seed)
    return pd.concat([h, l]).sample(
        frac=1, random_state=seed).reset_index(drop=True)

def get_pool(df, n=20):
    h = df[df['label'] == 'human'].sample(n, random_state=SEED)
    l = df[df['label'] == 'llm'].sample(n, random_state=SEED)
    return pd.concat([h, l]).reset_index(drop=True)

hc3_test   = pd.read_csv('hc3_test.csv')
eli5_test  = pd.read_csv('eli5_test.csv')
hc3_train  = pd.read_csv('hc3_train.csv')
eli5_train = pd.read_csv('eli5_train.csv')

hc3_eval      = sample_balanced(hc3_test,  EVAL_N)
eli5_eval     = sample_balanced(eli5_test, EVAL_N)
hc3_cot_eval  = sample_balanced(hc3_test,  COT_N)
eli5_cot_eval = sample_balanced(eli5_test, COT_N)
hc3_pool      = get_pool(hc3_train,  n=20)
eli5_pool     = get_pool(eli5_train, n=20)

print(f'hc3_eval  : {len(hc3_eval)} | eli5_eval  : {len(eli5_eval)}')
print(f'hc3_cot   : {len(hc3_cot_eval)} | eli5_cot   : {len(eli5_cot_eval)}')

# ─────────────────────────────────────────────────────────────────────────────
# MODEL LOADING
# bf16 compute dtype — Qwen2.5-14B numerically more stable with bf16 vs fp16
# padding_side='left' required for Qwen batch inference correctness
# ─────────────────────────────────────────────────────────────────────────────
print(f'\nLoading {HF_ID} in 4-bit...')

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    HF_ID, trust_remote_code=True, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    HF_ID,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True
)
model.eval()

print(f'✅ Model loaded on {next(model.parameters()).device}')
print(f'   Vocab size    : {tokenizer.vocab_size}')
print(f'   pad_token     : {tokenizer.pad_token!r}')
print(f'   eos_token     : {tokenizer.eos_token!r}')
print(f'   chat_template : {"present" if tokenizer.chat_template else "MISSING"}')

# ─────────────────────────────────────────────────────────────────────────────
# LABEL TOKEN IDS
# ─────────────────────────────────────────────────────────────────────────────
def get_label_token_ids(tokenizer):
    yes_ids, no_ids = set(), set()
    for s in ['yes', 'Yes', 'YES', ' yes', ' Yes']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            yes_ids.add(ids[0])
    for s in ['no', 'No', 'NO', ' no', ' No']:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            no_ids.add(ids[0])
    print(f'yes (human) IDs : {list(yes_ids)}')
    print(f'  -> tokens      : {[tokenizer.decode([i]) for i in yes_ids]}')
    print(f'no  (AI)    IDs : {list(no_ids)}')
    print(f'  -> tokens      : {[tokenizer.decode([i]) for i in no_ids]}')
    if not yes_ids or not no_ids:
        raise RuntimeError('No single-token yes/no IDs found — check tokenizer')
    return list(yes_ids), list(no_ids)

yes_ids, no_ids = get_label_token_ids(tokenizer)

# ─────────────────────────────────────────────────────────────────────────────
# NEUTRAL PRIOR (kept for debug comparison only)
# ─────────────────────────────────────────────────────────────────────────────
dummy = tokenizer('Answer:', return_tensors='pt').to(model.device)
with torch.no_grad():
    prior_logits = model(**dummy).logits[0, -1, :]

neutral_yes_prior = torch.stack([prior_logits[i] for i in yes_ids]).max()
neutral_no_prior  = torch.stack([prior_logits[i] for i in no_ids]).max()

print(f'\nNeutral prior — yes: {neutral_yes_prior:.2f}  '
      f'no: {neutral_no_prior:.2f}  '
      f'gap: {(neutral_no_prior - neutral_yes_prior):.2f}')
print('(Will be replaced by task prior)')

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT BUILDERS — Qwen2.5-14B
#
# POLARITY SWAPPED:
#   yes = human-written
#   no  = AI-generated
#   flip=False → P(llm) = P(no) directly
#
# Larger model (14B) gets richer prompts than 7B:
#   - Expert framing with "authorship attribution"
#   - Stronger CoT formatting constraint to prevent 90% unknown rate
#   - Explicit "Begin your analysis now:" to prevent refusal/hedging
# ─────────────────────────────────────────────────────────────────────────────
def qwen14_zero_shot(text, tokenizer):
    msgs = [
        {
            'role': 'system',
            'content': (
                'You are an expert in authorship attribution and '
                'AI-generated text analysis. '
                'Answer with ONE word only: yes or no. '
                'yes = human-written. no = AI-generated. '
                'No explanation. No punctuation. One word.'
            )
        },
        {
            'role': 'user',
            'content': (
                'Was this text written by a human?\n\n'
                f'Text:\n"""{text[:700]}"""\n\n'
                'Answer yes or no.'
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'


def qwen14_few_shot(text, tokenizer, examples_df, k=K_SHOT,
                    idx=0, retriever=None):
    if retriever is not None:
        vec, matrix, pool_df = retriever
        examples = retrieve_shots(text, pool_df, vec, matrix, k)
    else:
        examples = examples_df.sample(k, random_state=idx)

    shots = ''
    for _, row in examples.iterrows():
        # yes=human, no=AI (swapped — NO annotations to avoid logit bias)
        ans = 'no' if row['label'] == 'llm' else 'yes'
        shots += f'Text: "{row["text"][:250]}"\nHuman-written? {ans}\n\n'

    msgs = [
        {
            'role': 'system',
            'content': (
                'You are an expert in authorship attribution. '
                'Study the labeled examples, then answer for the new text. '
                'Answer with ONE word only: yes or no. '
                'yes = human-written. no = AI-generated.'
            )
        },
        {
            'role': 'user',
            'content': (
                f'Labeled examples:\n{shots}'
                f'New text: "{text[:700]}"\n'
                'Human-written? yes or no.'
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return prompt + 'Answer:'


def qwen14_cot_reasoning(text, tokenizer):
    # Key differences from 7B CoT prompt:
    # 1. System prompt explicitly states "ALWAYS complete" — prevents early stop
    # 2. "no exceptions" on last two lines — reduces 90% unknown rate
    # 3. max_new_tokens=500 (vs 350 for 7B) — larger model needs more room
    # 4. pad_token_id=tokenizer.pad_token_id in generate (not eos_token_id)
    msgs = [
        {
            'role': 'system',
            'content': (
                'You are an expert forensic linguist performing '
                'authorship attribution analysis. '
                'You ALWAYS complete your full analysis and ALWAYS end '
                'with the AI_CONFIDENCE score and VERDICT line. '
                'Never leave your analysis incomplete or refuse to give a verdict.'
            )
        },
        {
            'role': 'user',
            'content': (
                'Analyse this passage to determine if it was written by '
                'a HUMAN or generated by an AI language model '
                '(ChatGPT, GPT-4, Claude, Qwen, Llama, etc.).\n\n'
                f'Passage:\n"""{text[:700]}"""\n\n'
                'Score each dimension 0 (strongly human) to 10 (strongly AI):\n\n'
                '1. STRUCTURE (0-10): Organised with clear sections/numbered points?\n'
                '2. COMPLETENESS (0-10): Covers the topic without obvious gaps?\n'
                '3. HEDGING (0-10): Confident authoritative tone, lacks uncertainty?\n'
                '4. PERSONAL VOICE (0-10): Lacks personal opinions/anecdotes/typos?\n'
                '5. LEXICAL POLISH (0-10): Uniformly formal/polished vocabulary?\n'
                '6. RESPONSE FIT (0-10): Directly and completely addresses question?\n'
                '7. FORMULAIC TELLS (0-10): Restates question, "Certainly!", '
                'unnaturally tidy closing?\n\n'
                'IMPORTANT: Short texts CAN be AI-generated. '
                'Score all 7 dimensions regardless of text length.\n\n'
                'You MUST end your response with EXACTLY these two lines '
                '(no exceptions, no additional text after):\n'
                'AI_CONFIDENCE: <your average score 0-10>\n'
                'VERDICT: yes\n'
                '--- OR ---\n'
                'AI_CONFIDENCE: <your average score 0-10>\n'
                'VERDICT: no\n\n'
                'Begin your analysis now:'
            )
        }
    ]
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)

print('✅ Prompt builders defined')

# ─────────────────────────────────────────────────────────────────────────────
# CONSTRAINED SCORING
# flip=False: softmax([no_logit, yes_logit])[1] = P(yes=human)
# ─────────────────────────────────────────────────────────────────────────────
def constrained_score(model, tokenizer, prompt_text, yes_ids, no_ids,
                      yes_prior, no_prior, flip=FLIP, use_prior=True):
    enc = tokenizer(prompt_text, return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits[0, -1, :]

    yes_logit = torch.stack([logits[i] for i in yes_ids]).max()
    no_logit  = torch.stack([logits[i] for i in no_ids]).max()

    if use_prior:
        yes_logit = yes_logit - yes_prior
        no_logit  = no_logit  - no_prior

    raw = torch.softmax(
        torch.stack([no_logit, yes_logit]), dim=0)[1].item()

    return 1.0 - raw if flip else raw

# ─────────────────────────────────────────────────────────────────────────────
# TF-IDF RETRIEVERS
# ─────────────────────────────────────────────────────────────────────────────
def build_retriever(pool_df):
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    matrix = vec.fit_transform(pool_df['text'].astype(str))
    return vec, matrix

def retrieve_shots(text, pool_df, vec, matrix, k=K_SHOT):
    q    = vec.transform([text])
    sims = cosine_similarity(q, matrix).flatten()
    results = []
    per_class = [k // 2 + (1 if i < k % 2 else 0) for i in range(2)]
    random.shuffle(per_class)
    for n, lbl in zip(per_class, ['human', 'llm']):
        mask       = (pool_df['label'] == lbl).values
        class_sims = sims.copy()
        class_sims[~mask] = -1
        top_idx = class_sims.argsort()[-n:][::-1]
        results.extend(top_idx.tolist())
    return pool_df.iloc[results]

hc3_vec,  hc3_matrix  = build_retriever(hc3_pool)
eli5_vec, eli5_matrix = build_retriever(eli5_pool)
print('✅ TF-IDF retrievers built')

# ─────────────────────────────────────────────────────────────────────────────
# TASK PRIOR
# CRITICAL: computed from swapped polarity prompt (yes=human, no=AI)
# Prior must exactly match the prompt context used at eval time
# ─────────────────────────────────────────────────────────────────────────────
def compute_task_prior(model, tokenizer, eval_df, yes_ids, no_ids,
                       n_samples=PRIOR_N, seed=SEED):
    sample = eval_df.sample(
        min(n_samples, len(eval_df)), random_state=seed)
    yes_logits_list, no_logits_list = [], []

    for _, row in tqdm(sample.iterrows(), total=len(sample),
                       desc='Computing task prior'):
        prompt = qwen14_zero_shot(str(row['text']), tokenizer)
        enc    = tokenizer(prompt, return_tensors='pt',
                           truncation=True,
                           max_length=MAX_LEN).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits[0, -1, :]
        yes_logits_list.append(
            torch.stack([logits[i] for i in yes_ids]).max().item())
        no_logits_list.append(
            torch.stack([logits[i] for i in no_ids]).max().item())

    task_yes_prior = torch.tensor(yes_logits_list).mean()
    task_no_prior  = torch.tensor(no_logits_list).mean()

    print(f'\nTask prior    — yes: {task_yes_prior:.3f}  '
          f'no: {task_no_prior:.3f}  '
          f'gap: {(task_no_prior - task_yes_prior):.3f}')
    print(f'Neutral prior — yes: {neutral_yes_prior:.3f}  '
          f'no: {neutral_no_prior:.3f}  '
          f'gap: {(neutral_no_prior - neutral_yes_prior):.3f}')
    print(f'Bias shift    — yes: {(task_yes_prior-neutral_yes_prior):.3f}  '
          f'no: {(task_no_prior-neutral_no_prior):.3f}')
    return task_yes_prior, task_no_prior


prior_sample = pd.concat([
    hc3_eval.sample(PRIOR_N // 2, random_state=SEED),
    eli5_eval.sample(PRIOR_N // 2, random_state=SEED)
]).reset_index(drop=True)

yes_prior, no_prior = compute_task_prior(
    model, tokenizer, prior_sample, yes_ids, no_ids)

print('\n✅ Task prior computed — using for all constrained_score calls')

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG 1 — POLARITY TEST
# Target: task prior + flip=False → ✅ for BOTH human and llm rows
# If llm row fails → prior overcorrecting → apply 0.85 scale below
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('DEBUG 1 — POLARITY TEST')
print('=' * 65)
print('Target: task prior + flip=False → ✅ for both human and llm\n')

for ds_name, df in [('hc3', hc3_eval), ('eli5', eli5_eval)]:
    print(f'── {ds_name.upper()} ──')
    for lbl in ['human', 'llm']:
        row    = df[df['label'] == lbl].iloc[0]
        text   = str(row['text'])
        prompt = qwen14_zero_shot(text, tokenizer)

        for prior_name, yp, np_ in [
            ('neutral', neutral_yes_prior, neutral_no_prior),
            ('task',    yes_prior,         no_prior)
        ]:
            for flip in [True, False]:
                s = constrained_score(
                    model, tokenizer, prompt,
                    yes_ids, no_ids, yp, np_,
                    flip=flip, use_prior=True)
                correct = (s > 0.5) == (lbl == 'llm')
                print(f'  true={lbl:5s} prior={prior_name:7s} '
                      f'flip={str(flip):5s} → {s:.4f}  '
                      f'pred={"llm" if s>0.5 else "human":5s}  '
                      f'{"✅" if correct else "❌"}')
    print()

print('✅ Polarity test done')
print('→ task + flip=False should give ✅ for both rows')
print('→ if llm row still ❌, uncomment the scale block below')


# ─────────────────────────────────────────────────────────────────────────────
# DEBUG 2 — TOKEN LEVEL
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('DEBUG 2 — TOP-10 TOKENS + LABEL LOGITS')
print('=' * 65)

for ds_name, df in [('hc3', hc3_eval), ('eli5', eli5_eval)]:
    for lbl in ['human', 'llm']:
        row    = df[df['label'] == lbl].iloc[0]
        text   = str(row['text'])
        prompt = qwen14_zero_shot(text, tokenizer)

        print(f'\n── {ds_name.upper()} | true={lbl} ──')
        enc = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=MAX_LEN).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits[0, -1, :]

        top10 = torch.topk(logits, 10)
        print('Top 10 tokens:')
        for rank, (tid, logit) in enumerate(
                zip(top10.indices, top10.values)):
            prob = torch.softmax(logits, dim=0)[tid].item()
            print(f'  {rank+1:2d}: {tokenizer.decode([tid])!r:12s}  '
                  f'logit={logit:.2f}  prob={prob:.4f}')

        print('\nLabel token logits:')
        for label_name, ids in [
            ('yes (human)', yes_ids),
            ('no  (AI)   ', no_ids)
        ]:
            for tid in ids:
                tok   = tokenizer.decode([tid])
                logit = logits[tid].item()
                prob  = torch.softmax(logits, dim=0)[tid].item()
                print(f'  {label_name}: {tok!r:8s}  id={tid}  '
                      f'logit={logit:.2f}  prob={prob:.4f}')

        s_neutral = constrained_score(
            model, tokenizer, prompt, yes_ids, no_ids,
            neutral_yes_prior, neutral_no_prior,
            flip=FLIP, use_prior=True)
        s_task = constrained_score(
            model, tokenizer, prompt, yes_ids, no_ids,
            yes_prior, no_prior,
            flip=FLIP, use_prior=True)

        print(f'\n  neutral prior → P(llm)={s_neutral:.4f}  '
              f'pred={"llm" if s_neutral>0.5 else "human":5s}  '
              f'{"✅" if (s_neutral>0.5)==(lbl=="llm") else "❌"}')
        print(f'  task prior    → P(llm)={s_task:.4f}  '
              f'pred={"llm" if s_task>0.5 else "human":5s}  '
              f'{"✅" if (s_task>0.5)==(lbl=="llm") else "❌"}')
        print('-' * 65)

print('\n✅ Debug 2 done')
print('→ yes/no tokens must appear in top-10')
print('→ task prior must give more ✅ than neutral prior')

# ─────────────────────────────────────────────────────────────────────────────
# COT PARSER
# ─────────────────────────────────────────────────────────────────────────────
def parse_cot_output(raw: str):
    raw_lower = raw.lower()

    # AI_CONFIDENCE explicit tag
    conf = None
    m = re.search(
        r'ai_confidence[:\s]+([0-9]+(?:\.[0-9]+)?)', raw_lower)
    if m:
        conf = min(float(m.group(1)) / 10.0, 1.0)

    # x/10 fallback
    if conf is None:
        vals_10 = re.findall(
            r'([0-9](?:\.[0-9]+)?)\s*/\s*10', raw_lower)
        if vals_10:
            vals = [float(v) for v in vals_10 if float(v) <= 10]
            if len(vals) >= 3:
                conf = min(sum(vals) / len(vals) / 10.0, 1.0)

    # dimension keyword fallback
    if conf is None:
        dim_scores = re.findall(
            r'(?:structure|completeness|hedging|personal|lexical|'
            r'response fit|formulaic)[^\n]*?([0-9](?:\.[0-9]+)?)\b',
            raw_lower
        )
        if dim_scores:
            vals = [float(s) for s in dim_scores if float(s) <= 10]
            if vals:
                conf = min(sum(vals) / len(vals) / 10.0, 1.0)

    # Verdict — CoT prompt: yes=AI, no=human
    m_v = re.search(r'verdict:\s*(yes|no)', raw_lower)
    if m_v:
        verdict = 'llm' if m_v.group(1) == 'yes' else 'human'
    else:
        lines = [l.strip() for l in raw.split('\n') if l.strip()]
        last  = lines[-1].lower() if lines else ''
        if last.startswith('yes'):
            verdict = 'llm'
        elif last.startswith('no'):
            verdict = 'human'
        else:
            verdict = 'unknown'

    # Sanity: trust conf when strongly divergent from verdict
    if conf is not None and verdict != 'unknown':
        if conf > 0.75 and verdict == 'human':
            verdict = 'llm'
        elif conf < 0.25 and verdict == 'llm':
            verdict = 'human'

    return verdict, conf

# ─────────────────────────────────────────────────────────────────────────────
# COT ENSEMBLE
# ─────────────────────────────────────────────────────────────────────────────
def cot_ensemble_score(conf, logit_score,
                       conf_weight=COT_CONF_WEIGHT,
                       logit_weight=COT_LOGIT_WEIGHT):
    """
    Dead zone [0.35, 0.65] — wider than Llama [0.40, 0.60].
    Qwen14B conf is more reliable than 7B but still benefits
    from a wider dead zone to avoid overcorrection.
    Outside dead zone: weighted ensemble.
    Inside dead zone: logit only.
    """
    if conf is None or (DEAD_ZONE_LO <= conf <= DEAD_ZONE_HI):
        return float(max(0.01, min(0.99, logit_score))), False
    score = conf_weight * conf + logit_weight * logit_score
    return float(max(0.01, min(0.99, score))), True

# ─────────────────────────────────────────────────────────────────────────────
# COT PIPELINE
# Two critical fixes vs original notebook:
#   1. pad_token_id=tokenizer.pad_token_id (NOT eos_token_id)
#      — eos as pad caused 90% unknown by terminating generation early
#   2. max_new_tokens=500 — 14B needs more room to complete 7-dimension analysis
# ─────────────────────────────────────────────────────────────────────────────
def run_cot_sample(text, tokenizer, model, yes_ids, no_ids,
                   yes_prior, no_prior):
    # Pass 1: reasoning generation
    prompt1 = qwen14_cot_reasoning(text, tokenizer)
    enc1 = tokenizer(prompt1, return_tensors='pt',
                     truncation=True, max_length=MAX_LEN).to(model.device)
    set_seed(SEED)
    with torch.no_grad():
        out1 = model.generate(
            **enc1,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,  # NOT eos
            eos_token_id=tokenizer.eos_token_id
        )
    reasoning = tokenizer.decode(
        out1[0][enc1['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    verdict, conf = parse_cot_output(reasoning)

    # Zero-shot constrained logit — swapped prompt + task prior
    zs_prompt   = qwen14_zero_shot(text, tokenizer)
    logit_score = constrained_score(
        model, tokenizer, zs_prompt,
        yes_ids, no_ids, yes_prior, no_prior,
        flip=FLIP, use_prior=True
    )

    score, used_conf = cot_ensemble_score(conf, logit_score)

    if verdict == 'unknown':
        pred = 'llm' if score > 0.5 else 'human'
    elif score < VERDICT_LO:
        pred = 'human'
    elif score > VERDICT_HI:
        pred = 'llm'
    else:
        pred = verdict

    return pred, score, reasoning, verdict, conf, logit_score, used_conf

print('✅ CoT pipeline defined')

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG 3 — VERDICT AUDIT
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 70)
print('DEBUG 3 — VERDICT AUDIT (10 samples per dataset)')
print('=' * 70)

for ds_name, df in [('hc3', hc3_cot_eval), ('eli5', eli5_cot_eval)]:
    correct_v     = 0
    conf_vals     = {'llm': [], 'human': []}
    logit_vals    = {'llm': [], 'human': []}
    unknown_count = 0

    for _, row in df.head(10).iterrows():
        text, true_lbl = str(row['text']), row['label']
        (pred, score, reasoning, verdict,
         conf, logit_score, used_conf) = run_cot_sample(
            text, tokenizer, model,
            yes_ids, no_ids, yes_prior, no_prior)

        correct_v += int(pred == true_lbl)
        if verdict == 'unknown':
            unknown_count += 1
        if conf is not None:
            conf_vals[true_lbl].append(conf)
        logit_vals[true_lbl].append(logit_score)

        conf_str = f'{conf:.3f}' if conf is not None else 'None'
        print(f'{"✅" if pred==true_lbl else "❌"} '
              f'ds={ds_name} true={true_lbl:5s} pred={pred:5s} '
              f'verdict={verdict:7s} conf={conf_str} '
              f'used_conf={used_conf} '
              f'logit={logit_score:.3f} score={score:.3f}')

    def safe_mean(lst):
        return round(sum(lst)/len(lst), 3) if lst else 'n/a'

    lsep = (safe_mean(logit_vals['llm']) - safe_mean(logit_vals['human'])
            if logit_vals['llm'] and logit_vals['human'] else 'n/a')

    print(f'\n{ds_name} | acc={correct_v/10:.1%} | '
          f'unknown={unknown_count}/10 ({unknown_count/10:.0%}) | '
          f'logit(llm)={safe_mean(logit_vals["llm"])} '
          f'logit(human)={safe_mean(logit_vals["human"])} | '
          f'logit_sep={lsep}')
    print(f'  → unknown should be <10% | logit_sep should be >0\n')

print('✅ Verdict audit done')

# ─────────────────────────────────────────────────────────────────────────────
# MAIN EVALUATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
def run_regime(eval_df, pool, regime, ds_name,
               yes_prior, no_prior, retriever=None):
    ckpt_path = (f'{RESULTS_DIR}/ckpt_{MODEL_NAME}'
                 f'_{regime}_{ds_name}.json')

    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            records = json.load(f)
        done = {r['idx'] for r in records}
        print(f'  Resuming {regime}/{ds_name}: '
              f'{len(done)}/{len(eval_df)} done')
    else:
        records, done = [], set()

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df),
                         desc=f'{regime}/{ds_name}', leave=True):
        if idx in done:
            continue
        text, true_lbl = str(row['text']), row['label']

        if regime == 'zero_shot':
            prompt = qwen14_zero_shot(text, tokenizer)
            score  = constrained_score(
                model, tokenizer, prompt,
                yes_ids, no_ids, yes_prior, no_prior,
                flip=FLIP, use_prior=True)
            pred          = 'llm' if score > 0.5 else 'human'
            raw_out       = f'[constrained+task_prior] score={score:.4f}'
            cot_verdict   = None; cot_conf      = None
            cot_logit     = None; cot_used_conf = None

        elif regime == 'few_shot':
            prompt = qwen14_few_shot(text, tokenizer, pool,
                                     idx=idx, retriever=retriever)
            score  = constrained_score(
                model, tokenizer, prompt,
                yes_ids, no_ids, yes_prior, no_prior,
                flip=FLIP, use_prior=True)
            pred          = 'llm' if score > 0.5 else 'human'
            raw_out       = f'[constrained+task_prior] score={score:.4f}'
            cot_verdict   = None; cot_conf      = None
            cot_logit     = None; cot_used_conf = None

        elif regime == 'cot':
            (pred, score, reasoning, verdict,
             conf, logit_score, used_conf) = run_cot_sample(
                text, tokenizer, model,
                yes_ids, no_ids, yes_prior, no_prior)
            conf_str      = f'{conf:.4f}' if conf is not None else 'None'
            raw_out       = (
                f'[verdict={verdict}|conf={conf_str}|'
                f'used_conf={used_conf}|'
                f'logit={logit_score:.4f}|score={score:.4f}]'
                f'\n{reasoning}')
            cot_verdict   = verdict
            cot_conf      = float(conf) if conf is not None else None
            cot_logit     = float(logit_score)
            cot_used_conf = bool(used_conf)
        else:
            continue

        correct = int(pred == true_lbl)

        if regime == 'cot':
            if cot_verdict != 'unknown' and correct:
                cs = 'verdict'
            elif cot_verdict == 'unknown' and correct:
                cs = 'score_fallback'
            elif cot_verdict != 'unknown' and not correct:
                cs = 'verdict_fail'
            else:
                cs = 'score_fail'
        else:
            cs = 'score' if correct else 'score_fail'

        records.append({
            'idx':                idx,
            'true_label':         true_lbl,
            'pred_label':         pred,
            'score':              float(score),
            'correct':            correct,
            'regime':             regime,
            'dataset':            ds_name,
            'text_preview':       text[:300],
            'raw_model_output':   raw_out,
            'cot_verdict':        cot_verdict,
            'cot_conf':           cot_conf,
            'cot_logit_score':    cot_logit,
            'cot_used_conf':      cot_used_conf,
            'correctness_source': cs,
            'failure_mode': (
                'correct'        if pred == true_lbl else
                'false_positive' if pred == 'llm' and true_lbl == 'human'
                else 'false_negative')
        })
        with open(ckpt_path, 'w') as f:
            json.dump(records, f, indent=2, ensure_ascii=False)

    df_out = pd.DataFrame(records)
    y_t = (df_out['true_label'] == 'llm').astype(int)
    y_s = df_out['score']
    auc = roc_auc_score(y_t, y_s) if y_t.nunique() > 1 else float('nan')
    acc = accuracy_score(y_t, (y_s > 0.5).astype(int))
    df_out.to_csv(
        f'{RESULTS_DIR}/{MODEL_NAME}_{regime}_{ds_name}.csv',
        index=False)
    return df_out, auc, acc

print('✅ Scoring loop defined')

# ─────────────────────────────────────────────────────────────────────────────
# RUN ALL REGIMES
# ─────────────────────────────────────────────────────────────────────────────
hc3_retriever  = (hc3_vec,  hc3_matrix,  hc3_pool)
eli5_retriever = (eli5_vec, eli5_matrix, eli5_pool)

DATASETS = {
    'hc3' : {
        'eval':      hc3_eval,
        'cot':       hc3_cot_eval,
        'pool':      hc3_pool,
        'retriever': hc3_retriever,
    },
    'eli5': {
        'eval':      eli5_eval,
        'cot':       eli5_cot_eval,
        'pool':      eli5_pool,
        'retriever': eli5_retriever,
    },
}

summary, all_dfs = [], {}

for regime in ['zero_shot', 'few_shot', 'cot']:
    for ds_name, ds_cfg in DATASETS.items():
        ev        = ds_cfg['cot'] if regime == 'cot' else ds_cfg['eval']
        pool      = ds_cfg['pool']
        retriever = ds_cfg['retriever'] if regime == 'few_shot' else None

        print(f'\n── {regime.upper()} | {ds_name.upper()} ──')
        df_r, auc, acc = run_regime(
            ev, pool, regime, ds_name,
            yes_prior, no_prior,
            retriever=retriever)
        all_dfs[(regime, ds_name)] = df_r

        fp = (df_r['failure_mode'] == 'false_positive').sum()
        fn = (df_r['failure_mode'] == 'false_negative').sum()

        if regime == 'cot':
            unk = ((df_r['cot_verdict'] == 'unknown').mean()
                   if 'cot_verdict' in df_r.columns else 0)
            ml  = df_r[df_r['true_label']=='llm']['cot_logit_score'].mean()
            mh  = df_r[df_r['true_label']=='human']['cot_logit_score'].mean()
            src = df_r['correctness_source'].value_counts().to_dict()
            print(f'   AUROC={auc:.4f}  Acc={acc:.4f}  '
                  f'FP={fp}  FN={fn}  n={len(df_r)}')
            print(f'   unknown_rate={unk:.1%} | '
                  f'logit(llm)={ml:.3f}  '
                  f'logit(human)={mh:.3f}  '
                  f'sep={ml-mh:.3f}')
            print(f'   correctness_source: {src}')
        else:
            print(f'   AUROC={auc:.4f}  Acc={acc:.4f}  '
                  f'FP={fp}  FN={fn}  n={len(df_r)}')

        summary.append({
            'regime':   regime,
            'dataset':  ds_name,
            'auroc':    round(auc, 4),
            'accuracy': round(acc, 4),
            'fp':       int(fp),
            'fn':       int(fn),
            'n':        len(df_r),
        })

print('\n✅ All regimes done')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
df_summary = pd.DataFrame(summary)
df_summary.to_csv(f'{RESULTS_DIR}/{MODEL_NAME}_summary.csv', index=False)

print(f'\n{MODEL_NAME} — Results Summary')
header = (f"{'regime':<12} {'dataset':<6} {'auroc':>8} "
          f"{'accuracy':>10} {'fp':>6} {'fn':>6} {'n':>6}")
sep = '─' * len(header)
print(sep); print(header); print(sep)
for r in summary:
    print(f"{r['regime']:<12} {r['dataset']:<6} "
          f"{r['auroc']:>8.4f} {r['accuracy']:>10.4f} "
          f"{r['fp']:>6} {r['fn']:>6} {r['n']:>6}")
print(sep)

# ─────────────────────────────────────────────────────────────────────────────
# CALIBRATED THRESHOLD TABLE
# ─────────────────────────────────────────────────────────────────────────────
cal_rows = []
for (regime, ds), df_r in all_dfs.items():
    y_t = (df_r['true_label'] == 'llm').astype(int)
    y_s = df_r['score']
    fpr, tpr, thresholds = roc_curve(y_t, y_s)
    best_thresh   = thresholds[(tpr - fpr).argmax()]
    median_thresh = float(y_s.median())
    cal_rows.append({
        'regime':         regime,
        'dataset':        ds,
        'auroc':          round(roc_auc_score(y_t, y_s), 4),
        'acc@0.5':        round(accuracy_score(
                              y_t, (y_s >= 0.5).astype(int)), 4),
        'acc@median':     round(accuracy_score(
                              y_t, (y_s >= median_thresh).astype(int)), 4),
        'acc@optimal':    round(accuracy_score(
                              y_t, (y_s >= best_thresh).astype(int)), 4),
        'median_score':   round(median_thresh, 4),
        'optimal_thresh': round(float(best_thresh), 4),
    })

pd.DataFrame(cal_rows).to_csv(
    f'{RESULTS_DIR}/{MODEL_NAME}_calibrated.csv', index=False)

print(f'\n{MODEL_NAME} — Calibrated Threshold Results')
ch = (f"{'regime':<12} {'dataset':<6} {'auroc':>8} {'acc@0.5':>9} "
      f"{'acc@median':>12} {'acc@optimal':>13} "
      f"{'median':>8} {'opt_thresh':>11}")
s2 = '─' * len(ch)
print(s2); print(ch); print(s2)
for r in cal_rows:
    print(f"{r['regime']:<12} {r['dataset']:<6} "
          f"{r['auroc']:>8.4f} {r['acc@0.5']:>9.4f} "
          f"{r['acc@median']:>12.4f} {r['acc@optimal']:>13.4f} "
          f"{r['median_score']:>8.4f} {r['optimal_thresh']:>11.4f}")
print(s2)

# ─────────────────────────────────────────────────────────────────────────────
# COT COMPONENT ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
print(f'\n{MODEL_NAME} — CoT Component Analysis')
cot_analysis = []
for ds_name in ['hc3', 'eli5']:
    df_r = all_dfs.get(('cot', ds_name))
    if df_r is None:
        continue
    df_with    = df_r[df_r['cot_conf'].notna() & df_r['cot_used_conf']]
    df_without = df_r[~(df_r['cot_conf'].notna() & df_r['cot_used_conf'])]
    for subset, label in [
        (df_with,    'conf+logit'),
        (df_without, 'logit_only')
    ]:
        if len(subset) < 2:
            continue
        y_t = (subset['true_label'] == 'llm').astype(int)
        y_s = subset['score']
        if y_t.nunique() < 2:
            continue
        cot_analysis.append({
            'dataset':      ds_name,
            'score_type':   label,
            'n':            len(subset),
            'auroc':        round(roc_auc_score(y_t, y_s), 4),
            'accuracy':     round(accuracy_score(
                                y_t, (y_s > 0.5).astype(int)), 4),
            'pct_of_total': f'{len(subset)/len(df_r):.1%}',
        })

if cot_analysis:
    df_cot = pd.DataFrame(cot_analysis)
    print(df_cot.to_string(index=False))
    df_cot.to_csv(
        f'{RESULTS_DIR}/{MODEL_NAME}_cot_component_analysis.csv',
        index=False)
else:
    print('  No CoT component data available')

# ─────────────────────────────────────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f'{MODEL_NAME} — Detector Results\n'
    f'(swapped polarity: yes=human, no=AI | task prior | flip=False)',
    fontsize=11, fontweight='bold')

ax = axes[0]
lbls  = [f"{r['regime']}\n{r['dataset']}" for r in summary]
aurcs = [r['auroc'] for r in summary]
cols  = ['#4C8EDA', '#DA7E4C', '#4CDA8E',
         '#DA4C8E', '#8E4CDA', '#DA4C4C']
bars  = ax.bar(lbls, aurcs, color=cols[:len(summary)])
ax.axhline(0.5, color='grey', ls='--', lw=1, label='random')
ax.set_ylim(0, 1)
ax.set_ylabel('AUROC')
ax.set_title('AUROC by Regime & Dataset')
ax.legend(fontsize=8)
for bar, v in zip(bars, aurcs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax2 = axes[1]
for i, ((regime, ds), df_r) in enumerate(all_dfs.items()):
    y_t = (df_r['true_label'] == 'llm').astype(int)
    y_s = df_r['score']
    if y_t.nunique() < 2:
        continue
    fpr, tpr, _ = roc_curve(y_t, y_s)
    auc = roc_auc_score(y_t, y_s)
    ax2.plot(fpr, tpr, color=plt.cm.tab10.colors[i % 10],
             label=f'{regime}/{ds} {auc:.3f}')
ax2.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax2.set_xlabel('FPR')
ax2.set_ylabel('TPR')
ax2.set_title('ROC Curves')
ax2.legend(fontsize=7)

ax3 = axes[2]
for (regime, ds), df_r in all_dfs.items():
    for lbl in ['human', 'llm']:
        vals = df_r[df_r['true_label'] == lbl]['score']
        ax3.hist(vals, bins=20, alpha=0.4, density=True,
                 label=f'{regime}/{ds} {lbl}')
ax3.axvline(0.5, color='black', ls='--', lw=1, label='threshold=0.5')
ax3.set_xlabel('P(llm) score')
ax3.set_ylabel('Density')
ax3.set_title('Score Distributions')
ax3.legend(fontsize=6)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_results.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Per-condition distributions
n_conds = len(all_dfs)
fig2, axes2 = plt.subplots(
    1, n_conds, figsize=(5 * n_conds, 4), squeeze=False)
for i, ((regime, ds), df_r) in enumerate(all_dfs.items()):
    ax = axes2[0][i]
    for lbl, color in [('human', '#4C8EDA'), ('llm', '#DA4C4C')]:
        vals = df_r[df_r['true_label'] == lbl]['score']
        ax.hist(vals, bins=20, alpha=0.6, color=color,
                label=lbl, density=True)
    ax.axvline(0.5, color='black', ls='--', lw=1)
    ax.set_title(f'{regime}/{ds}', fontsize=9)
    ax.set_xlabel('score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
plt.suptitle(f'{MODEL_NAME} — Score Distributions', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_score_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()

# CoT scatter: conf vs logit
for ds_name in ['hc3', 'eli5']:
    df_r = all_dfs.get(('cot', ds_name))
    if df_r is None or 'cot_conf' not in df_r.columns:
        continue
    df_plot = df_r[df_r['cot_conf'].notna()].copy()
    if len(df_plot) < 2:
        continue
    fig3, ax_s = plt.subplots(figsize=(6, 5))
    for lbl, color, marker in [
        ('human', '#4C8EDA', 'o'),
        ('llm',   '#DA4C4C', 'x')
    ]:
        sub = df_plot[df_plot['true_label'] == lbl]
        ax_s.scatter(sub['cot_conf'], sub['cot_logit_score'],
                     c=color, marker=marker, alpha=0.6,
                     label=lbl, s=40)
    ax_s.set_xlabel('Reasoning Confidence (conf)')
    ax_s.set_ylabel('Zero-Shot Logit Score (task prior, flip=False)')
    ax_s.set_title(f'CoT Component Scores — {ds_name.upper()}')
    ax_s.legend()
    ax_s.axvline(0.5, color='grey', ls='--', lw=0.8)
    ax_s.axhline(0.5, color='grey', ls='--', lw=0.8)
    plt.tight_layout()
    plt.savefig(
        f'{RESULTS_DIR}/{MODEL_NAME}_cot_scatter_{ds_name}.png',
        dpi=150, bbox_inches='tight')
    plt.show()

print('✅ All plots saved')

# ─────────────────────────────────────────────────────────────────────────────
# SAVE ALL OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
for (regime, ds_name), df_r in all_dfs.items():
    out = {
        'model':   MODEL_NAME,
        'regime':  regime,
        'dataset': ds_name,
        'scoring_config': {
            'polarity':          'yes=human, no=AI (swapped — Qwen no-bias)',
            'flip':              FLIP,
            'prior_type':        'task_prior',
            'prior_n_samples':   PRIOR_N,
            'task_yes_prior':    float(yes_prior),
            'task_no_prior':     float(no_prior),
            'neutral_yes_prior': float(neutral_yes_prior),
            'neutral_no_prior':  float(neutral_no_prior),
        },
        'cot_config': {
            'conf_weight':      COT_CONF_WEIGHT,
            'logit_weight':     COT_LOGIT_WEIGHT,
            'dead_zone':        [DEAD_ZONE_LO, DEAD_ZONE_HI],
            'verdict_override': [VERDICT_LO,   VERDICT_HI],
        } if regime == 'cot' else None,
        'summary': {
            'total':    len(df_r),
            'correct':  int(df_r['correct'].sum()),
            'accuracy': round(df_r['correct'].mean(), 4),
            'fp': int((df_r['failure_mode'] == 'false_positive').sum()),
            'fn': int((df_r['failure_mode'] == 'false_negative').sum()),
        },
        'by_failure_mode': {
            'false_positives': df_r[
                df_r['failure_mode'] == 'false_positive'
            ].to_dict('records'),
            'false_negatives': df_r[
                df_r['failure_mode'] == 'false_negative'
            ].to_dict('records'),
            'correct': df_r[
                df_r['failure_mode'] == 'correct'
            ].to_dict('records'),
        }
    }
    with open(
        f'{RESULTS_DIR}/visual_{MODEL_NAME}_{regime}_{ds_name}.json', 'w'
    ) as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

with open(f'{RESULTS_DIR}/results_qwen25_14b.pkl', 'wb') as f:
    pickle.dump({'all_dfs': all_dfs, 'summary': df_summary}, f)

print('✅ All files saved')
print(f'   Results folder: {RESULTS_DIR}')

In [ ]:
# ================================================================
# GPT-4o-mini as Detector — Zero-Shot, Few-Shot, Chain-of-Thought
# ── Cell 1: Install ────────────────────────────────────────────
# !pip install -q openai pandas numpy scikit-learn matplotlib seaborn tqdm


# ── Cell 2: Imports & Config ───────────────────────────────────
import os, json, re, time, pickle, random, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI
from sklearn.metrics import (
    roc_auc_score, accuracy_score, roc_curve,
    average_precision_score, brier_score_loss,
)
from tqdm import tqdm

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-YOUR_KEY_HERE")

client = OpenAI(api_key=OPENAI_API_KEY)

SEED             = 42
random.seed(SEED)
np.random.seed(SEED)

RESULTS_DIR      = "./results/llm_detector_gpt4omini"
os.makedirs(RESULTS_DIR, exist_ok=True)

EVAL_SAMPLE_SIZE = 200    # per dataset  (100 human + 100 llm)
COT_SAMPLE_SIZE  = 50     # CoT more expensive — keep at 50
K_SHOT           = 5
MODEL_ID         = "gpt-4o-mini"

# Token budget guard — will warn if approaching limit
TOKEN_BUDGET     = 4_000_000   # ~$0.60 input + $2.40 output at max
tokens_used_total = 0

print(f"✅ Config ready")
print(f"   model={MODEL_ID}")
print(f"   EVAL_SAMPLE_SIZE={EVAL_SAMPLE_SIZE}  COT_SAMPLE_SIZE={COT_SAMPLE_SIZE}")
print(f"   results → {RESULTS_DIR}/")
print(f"   estimated cost: ~$0.22 at these settings")


# ── Cell 3: Load Data ──────────────────────────────────────────
hc3_test   = pd.read_csv("hc3_test.csv")
eli5_test  = pd.read_csv("eli5_test.csv")
hc3_train  = pd.read_csv("hc3_train.csv")
eli5_train = pd.read_csv("eli5_train.csv")

def sample_balanced(df, n, seed=SEED):
    h = df[df["label"] == "human"].sample(n // 2, random_state=seed)
    l = df[df["label"] == "llm"].sample(n // 2,   random_state=seed)
    return (pd.concat([h, l])
              .sample(frac=1, random_state=seed)
              .reset_index(drop=True))

def get_pool(df, n=20, seed=SEED):
    h = df[df["label"] == "human"].sample(n, random_state=seed)
    l = df[df["label"] == "llm"].sample(n,   random_state=seed)
    return pd.concat([h, l]).reset_index(drop=True)

hc3_eval      = sample_balanced(hc3_test,  EVAL_SAMPLE_SIZE)
eli5_eval     = sample_balanced(eli5_test, EVAL_SAMPLE_SIZE)
hc3_cot_eval  = sample_balanced(hc3_test,  COT_SAMPLE_SIZE)
eli5_cot_eval = sample_balanced(eli5_test, COT_SAMPLE_SIZE)
hc3_pool      = get_pool(hc3_train)
eli5_pool     = get_pool(eli5_train)

print(f"✅ Data loaded")
print(f"   HC3  eval={len(hc3_eval)}  cot_eval={len(hc3_cot_eval)}")
print(f"   ELI5 eval={len(eli5_eval)}  cot_eval={len(eli5_cot_eval)}")


# ── Cell 4: Prompt Builders ────────────────────────────────────
#
# SCORING CONVENTION:
#   AI_SCORE: 0 = certainly human, 100 = certainly AI
#   No inversion needed (opposite of previous version).

def build_zero_shot(text: str) -> list[dict]:
    """
    7-dimension scoring forces granular commitment before final score.
    Each dimension scored 0-10 independently — no single hedge value possible.
    Final AI_SCORE is the mean, computed by the model.
    ~350 input + 180 output tokens per call.
    """
    return [
        {
            "role": "system",
            "content": (
                "You are an expert forensic linguist specialising in "
                "authorship attribution. "
                "AI-generated text is very common — including short, "
                "conversational-looking text from older models like "
                "ChatGPT-3.5. "
                "Score honestly based on the dimensions provided. "
                "Use the full 0-10 range for each dimension. "
                "Complete every analysis."
            )
        },
        {
            "role": "user",
            "content": (
                "Score this passage on each dimension from "
                "0 (strongly human) to 10 (strongly AI).\n\n"
                f"Passage:\n```\n{text[:800]}\n```\n\n"
                "Score each dimension (write the integer after the colon):\n"
                "1. HEDGING/FORMULAIC: Uses 'it is important', 'certainly', "
                "numbered sections, or safe generalisations: "
                "0=none, 10=throughout\n"
                "2. COMPLETENESS: Covers every sub-angle of the topic "
                "even when not asked: 0=partial, 10=exhaustive\n"
                "3. PERSONAL VOICE: Has opinions, errors, tangents, "
                "or emotional register: 0=strong personal voice, "
                "10=completely impersonal\n"
                "4. LEXICAL UNIFORMITY: Vocabulary register stays "
                "perfectly consistent throughout: 0=varied/informal, "
                "10=uniform polish\n"
                "5. STRUCTURAL NEATNESS: Clear intro/body/conclusion "
                "or logical flow: 0=rambling, 10=perfectly structured\n"
                "6. RESPONSE FIT: Directly and precisely addresses "
                "the apparent question: 0=wanders, 10=perfectly on-topic\n"
                "7. FORMULAIC TELLS: Restates question, tidy closing, "
                "or 'I hope this helps': 0=none, 10=multiple tells\n\n"
                "Then write:\n"
                "AI_SCORE: <arithmetic mean of your 7 scores × 10, rounded to nearest integer>\n"
                "(so if mean of 7 scores is 6.0, write AI_SCORE: 60)\n\n"
                "rounded to nearest integer>\n\n"
                "Format your response as:\n"
                "1: <score>\n"
                "2: <score>\n"
                "3: <score>\n"
                "4: <score>\n"
                "5: <score>\n"
                "6: <score>\n"
                "7: <score>\n"
                "AI_SCORE: <mean>"
            )
        }
    ]


def build_few_shot(text: str, pool_df: pd.DataFrame,
                   k: int = K_SHOT) -> list[dict]:
    """
    Examples show high-variance scoring — breaks the 0.75 anchor.
    Human examples show low scores (2-4), LLM shows high (7-9).
    ~1800 input + 180 output tokens per call.
    """
    examples = pool_df.sample(k, random_state=SEED)
    shots    = ""
    for _, row in examples.iterrows():
        is_llm = row["label"] == "llm"
        if is_llm:
            ex = (
                "1: 8\n2: 9\n3: 8\n4: 7\n5: 8\n6: 9\n7: 7\n"
                "AI_SCORE: 8"
            )
        else:
            ex = (
                "1: 2\n2: 3\n3: 2\n4: 4\n5: 3\n6: 4\n7: 1\n"
                "AI_SCORE: 3"
            )
        shots += (
            f"Passage: \"{row['text'][:250]}\"\n"
            f"{ex}\n\n---\n\n"
        )

    return [
        {
            "role": "system",
            "content": (
                "You are an expert forensic linguist. "
                "Score each passage on 7 dimensions (0=human, 10=AI). "
                "AI-generated text is very common — including short or "
                "conversational-looking text. "
                "Use the full 0-10 range. "
                "Always end with AI_SCORE: <mean of 7 scores>."
            )
        },
        {
            "role": "user",
            "content": (
                f"Examples:\n\n{shots}"
                f"Now score this passage:\n\n"
                f"Passage: \"{text[:800]}\"\n\n"
                "1. HEDGING/FORMULAIC: 0=none, 10=throughout\n"
                "2. COMPLETENESS: 0=partial, 10=exhaustive\n"
                "3. PERSONAL VOICE: 0=strong personal, 10=impersonal\n"
                "4. LEXICAL UNIFORMITY: 0=varied, 10=uniform polish\n"
                "5. STRUCTURAL NEATNESS: 0=rambling, 10=structured\n"
                "6. RESPONSE FIT: 0=wanders, 10=on-topic\n"
                "7. FORMULAIC TELLS: 0=none, 10=multiple\n\n"
                "1: \n2: \n3: \n4: \n5: \n6: \n7: \n"
                "AI_SCORE:"
            )
        }
    ]


def build_cot(text: str) -> list[dict]:
    """
    Same 7-dimension grid as zero-shot but with one reasoning
    sentence per dimension before the score.
    This keeps the structured scoring that works while adding
    the reasoning chain that prevents RLHF suppression.
    ~500 input + 400 output tokens per call.
    """
    return [
        {
            "role": "system",
            "content": (
                "You are an expert forensic linguist specialising in "
                "authorship attribution. "
                "AI-generated text is very common — including short, "
                "conversational-looking text from older models like "
                "ChatGPT-3.5. "
                "Score honestly based on evidence in the text. "
                "Use the full 0-10 range. "
                "Complete every dimension."
            )
        },
        {
            "role": "user",
            "content": (
                "Analyse whether this passage was written by a HUMAN "
                "or generated by an AI language model.\n\n"
                f"Passage:\n```\n{text[:800]}\n```\n\n"
                "For each dimension write ONE sentence of evidence "
                "from the text, then a score 0 (human) to 10 (AI).\n\n"
                "1. HEDGING/FORMULAIC — evidence of 'it is important', "
                "'certainly', numbered sections, safe generalisations:\n"
                "   Evidence: ...\n"
                "   Score (0-10): \n\n"
                "2. COMPLETENESS — covers every sub-angle even when "
                "not asked:\n"
                "   Evidence: ...\n"
                "   Score (0-10): \n\n"
                "3. PERSONAL VOICE — opinions, errors, tangents, "
                "emotional register:\n"
                "   Evidence: ...\n"
                "   Score (0-10): \n\n"
                "4. LEXICAL UNIFORMITY — vocabulary register stays "
                "perfectly consistent:\n"
                "   Evidence: ...\n"
                "   Score (0-10): \n\n"
                "5. STRUCTURAL NEATNESS — clear intro/body/conclusion "
                "or logical flow:\n"
                "   Evidence: ...\n"
                "   Score (0-10): \n\n"
                "6. RESPONSE FIT — directly and precisely addresses "
                "the apparent question:\n"
                "   Evidence: ...\n"
                "   Score (0-10): \n\n"
                "7. FORMULAIC TELLS — restates question, tidy closing, "
                "'I hope this helps':\n"
                "   Evidence: ...\n"
                "   Score (0-10): \n\n"
                "Then write:\n"
                "AI_SCORE: <mean of 7 scores × 10, "
                "rounded to nearest integer>\n"
                "VERDICT: ai\n"
                "  OR\n"
                "AI_SCORE: <mean of 7 scores × 10, "
                "rounded to nearest integer>\n"
                "VERDICT: human"
            )
        }
    ]

print("✅ Prompt builders defined")


# ── Cell 5: API Engine ─────────────────────────────────────────
MAX_RETRIES = 5
RETRY_DELAY = 2.0


def call_gpt(messages: list[dict], max_tokens: int = 120) -> dict:
    global tokens_used_total
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model       = MODEL_ID,
                messages    = messages,
                temperature = 0,
                seed        = SEED,
                max_tokens  = max_tokens,
            )
            raw_text          = resp.choices[0].message.content.strip()
            tokens_used_total += resp.usage.total_tokens
            # Warn if approaching budget
            if tokens_used_total > TOKEN_BUDGET * 0.8:
                print(f"  ⚠️  TOKEN WARNING: {tokens_used_total:,} / "
                      f"{TOKEN_BUDGET:,} used")
            return {
                "raw_text": raw_text,
                "usage":    resp.usage.total_tokens,
                "error":    None,
            }
        except Exception as e:
            wait = RETRY_DELAY * (2 ** attempt)
            print(f"    API error (attempt {attempt+1}/{MAX_RETRIES}): "
                  f"{e} — retrying in {wait:.0f}s")
            time.sleep(wait)

    return {"raw_text": "", "usage": 0, "error": "max_retries_exceeded"}

print("✅ API engine defined")


# ── Cell 6: Parsers ────────────────────────────────────────────

def parse_ai_score(raw: str) -> float | None:
    """
    Extract AI_SCORE from zero-shot / few-shot response.
    Handles:
      - Explicit AI_SCORE: tag (primary)
      - 7-line dimension format: compute mean ourselves as fallback
      - Last integer in response (last resort)
    Returns [0.0, 1.0] or None.
    """
    # Primary: explicit AI_SCORE tag
    m = re.search(r'ai_score[:\s]+([0-9]+(?:\.[0-9]+)?)', raw.lower())
    if m:
        val = float(m.group(1))
        if val <= 10:
            return val / 10.0      # 0-10 scale
        elif val <= 100:
            return val / 100.0     # 0-100 scale

    # Matches "1: 7" or "1. something: 7" patterns
    dim_scores = re.findall(
        r'^[1-7][.:]\s*(?:[^\n]*?:)?\s*([0-9](?:\.[0-9]+)?)\s*$',
        raw,
        re.MULTILINE
    )
    if len(dim_scores) >= 4:   # need at least 4 of 7 to be reliable
        vals = [float(s) for s in dim_scores if float(s) <= 10]
        if vals:
            return min(sum(vals) / len(vals) / 10.0, 1.0)

    # Fallback 2: any x/10 pattern
    vals_10 = re.findall(r'\b([0-9](?:\.[0-9]+)?)\s*/\s*10\b', raw)
    if vals_10:
        vals = [float(v) for v in vals_10 if float(v) <= 10]
        if len(vals) >= 3:
            return min(sum(vals) / len(vals) / 10.0, 1.0)

    # Fallback 3: last standalone integer in reasonable range
    integers = re.findall(r'\b([0-9]{1,3})\b', raw)
    if integers:
        val = int(integers[-1])
        if 0 <= val <= 10:
            return val / 10.0
        elif 10 < val <= 100:
            return val / 100.0

    return None


def parse_cot_response(raw: str) -> tuple[str, float | None]:
    """
    Updated to use same AI_SCORE format as zero-shot.
    Extracts score via parse_ai_score, verdict via VERDICT: tag.
    """
    # Score — reuse the same parser as zero-shot
    conf = parse_ai_score(raw)

    # Verdict
    raw_lower = raw.lower()
    m_v = re.search(r'verdict:\s*(ai|llm|human)', raw_lower)
    if m_v:
        verdict = 'llm' if m_v.group(1) in ['ai', 'llm'] else 'human'
    else:
        lines  = [l.strip() for l in raw.split('\n') if l.strip()]
        last   = lines[-1].lower() if lines else ''
        if 'ai' in last or 'llm' in last:
            verdict = 'llm'
        elif 'human' in last:
            verdict = 'human'
        else:
            verdict = 'unknown'

    # Sanity: strong score overrides weakly-parsed verdict
    if conf is not None and verdict != 'unknown':
        if conf > 0.75 and verdict == 'human':
            verdict = 'llm'
        elif conf < 0.25 and verdict == 'llm':
            verdict = 'human'

    return verdict, conf

print("✅ Parsers defined")


# ── Cell 7: Polarity Debug ─────────────────────────────────────
# Pass criteria:
#   human text → ai_score < 0.50 → pred=human ✅
#   llm text   → ai_score > 0.50 → pred=llm   ✅

print("=" * 65)
print("DEBUG 1 — POLARITY TEST (7-dimension scoring)")
print("=" * 65)
print("Target: llm ai_score > 0.50, human ai_score < 0.50\n")

all_pass = True
for ds_name, df in [("hc3", hc3_eval), ("eli5", eli5_eval)]:
    print(f"── {ds_name.upper()} ──")
    for true_lbl in ["human", "llm"]:
        sample   = df[df["label"] == true_lbl].iloc[0]
        text     = str(sample["text"])
        result   = call_gpt(build_zero_shot(text), max_tokens=180)
        score    = parse_ai_score(result["raw_text"])
        ai_score = score if score is not None else 0.5
        pred     = "llm" if ai_score > 0.5 else "human"
        flag     = "✅" if pred == true_lbl else "❌"
        if pred != true_lbl:
            all_pass = False

        # Show all dimension scores + AI_SCORE line
        lines   = [l.strip() for l in result["raw_text"].split("\n")
                   if l.strip()]
        dims    = " | ".join(lines[:7])   # first 7 lines = dim scores
        final   = lines[-1] if lines else ""

        print(f"  {flag} true={true_lbl}  ai_score={ai_score:.3f}  pred={pred}")
        print(f"     dims: {dims[:100]}")
        print(f"     final: {final}")
    print()

if all_pass:
    print("✅ All rows pass — proceed to full eval")
else:
    print("⚠️  Some rows failed")
    print("   Check dim scores above — if llm dims are NOT higher than")
    print("   human dims, HC3 is genuinely hard (older ChatGPT-3.5 text)")
    print("   ELI5 CoT is your strongest result — lead with that in paper")
print("=" * 65)


# ── Cell 8: Score Distribution Debug (5 samples per class) ────
# Quick check that scores are actually spreading across 0-1
# and that llm scores are meaningfully higher than human scores.

print("\n" + "=" * 65)
print("DEBUG 2 — SCORE SPREAD CHECK (5 samples per class/dataset)")
print("=" * 65)
print("Target: score(llm) > score(human), spread across range\n")

for ds_name, df in [("hc3", hc3_eval), ("eli5", eli5_eval)]:
    print(f"── {ds_name.upper()} ──")
    for true_lbl in ["human", "llm"]:
        samples = df[df["label"] == true_lbl].head(5)
        scores  = []
        for _, row in samples.iterrows():
            result = call_gpt(build_zero_shot(str(row["text"])),
                              max_tokens=120)
            s = parse_ai_score(result["raw_text"])
            scores.append(s if s is not None else 0.5)
        mean_s = sum(scores) / len(scores)
        print(f"  {true_lbl:<8} scores={[round(s,2) for s in scores]}"
              f"  mean={mean_s:.3f}")
    print()

print("If both classes cluster at same value → reasoning chain not working")
print("If llm mean > human mean → ✅ signal present")
print("=" * 65)


# ── Cell 9: Stale Checkpoint Cleanup ──────────────────────────
# Clears zero_shot and few_shot checkpoints from any previous broken run.
# CoT checkpoints are preserved if they already ran correctly.

# print("\nClearing stale zero_shot / few_shot checkpoints ...")
# cleared = 0
# for pattern in [
#     f"{RESULTS_DIR}/ckpt_GPT4oMini_zero_shot_*.json",
#     f"{RESULTS_DIR}/ckpt_GPT4oMini_few_shot_*.json",
# ]:
#     for fpath in glob.glob(pattern):
#         os.remove(fpath)
#         print(f"  🗑  {os.path.basename(fpath)}")
#         cleared += 1
# print("Clearing stale CoT checkpoints ...")
# for fpath in glob.glob(f"{RESULTS_DIR}/ckpt_GPT4oMini_cot_*.json"):
#     os.remove(fpath)
#     print(f"  🗑  {os.path.basename(fpath)}")
# print("✅ Done")

# if cleared == 0:
#     print("  (no stale checkpoints found)")
# print(f"✅ Cleanup done — {cleared} files removed")


# ── Cell 10: Scoring Loop ──────────────────────────────────────

def score_gpt(eval_df: pd.DataFrame, pool_df: pd.DataFrame,
              regime: str, ds_name: str) -> pd.DataFrame:
    """
    Scores one (regime, dataset) pair against GPT-4o-mini.
    Saves checkpoint after every call — interrupt-safe.
    Auto-resumes from checkpoint if re-run.
    """
    ckpt_path = (f"{RESULTS_DIR}/ckpt_GPT4oMini"
                 f"_{regime}_{ds_name}.json")

    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            records = json.load(f)
        done = {r["idx"] for r in records}
        print(f"    Resuming from checkpoint: "
              f"{len(done)}/{len(eval_df)} done")
    else:
        records, done = [], set()

    session_tokens = 0

    for idx, row in tqdm(eval_df.iterrows(),
                         total=len(eval_df),
                         desc=f"{regime}/{ds_name}",
                         leave=True):
        if idx in done:
            continue

        text     = str(row["text"])
        true_lbl = row["label"]

        if regime == "zero_shot":
            messages = build_zero_shot(text)
            max_tok  = 180     # 7 lines + AI_SCORE line
        elif regime == "few_shot":
            messages = build_few_shot(text, pool_df)
            max_tok  = 180
        elif regime == "cot":
            messages = build_cot(text)
            max_tok  = 600
        else:
            continue

        result         = call_gpt(messages, max_tokens=max_tok)
        session_tokens += result["usage"]
        raw_text       = result["raw_text"]

        # ── Score extraction ──────────────────────────────────
        cot_verdict = None
        cot_conf    = None

        if regime in ("zero_shot", "few_shot"):
            score = parse_ai_score(raw_text)
            if score is None:
                # Hard fallback: look for word label
                raw_lower = raw_text.lower().strip()
                if any(w in raw_lower for w in
                       ['ai-generated', 'ai generated',
                        'generated by', 'llm', 'language model']):
                    score = 0.85
                elif 'human' in raw_lower:
                    score = 0.15
                else:
                    score = 0.5

        elif regime == "cot":
            verdict, conf = parse_cot_response(raw_text)
            cot_verdict   = verdict
            cot_conf      = float(conf) if conf is not None else None
            if conf is not None:
                score = conf
            else:
                score = (1.0 if verdict == 'llm'
                         else 0.0 if verdict == 'human'
                         else 0.5)

        pred_lbl = 'llm' if score >= 0.45 else 'human'
        correct  = int(pred_lbl == true_lbl)

        failure_mode = (
            "correct"        if pred_lbl == true_lbl else
            "false_positive" if pred_lbl == "llm"
                             and true_lbl == "human"
            else "false_negative" if pred_lbl == "human"
                                  and true_lbl == "llm"
            else "unknown_output"
        )

        records.append({
            "idx":              idx,
            "true_label":       true_lbl,
            "pred_label":       pred_lbl,
            "score":            float(score),
            "correct":          correct,
            "regime":           regime,
            "dataset":          ds_name,
            "text_preview":     text[:300],
            "raw_model_output": raw_text,
            "tokens_used":      result["usage"],
            "failure_mode":     failure_mode,
            "cot_verdict":      cot_verdict,
            "cot_conf":         cot_conf,
        })

        with open(ckpt_path, "w") as f:
            json.dump(records, f, indent=2, ensure_ascii=False)

    print(f"    ✅ Done — session tokens: {session_tokens:,}  "
          f"cumulative: {tokens_used_total:,}")
    return pd.DataFrame(records)

print("✅ Scoring loop defined")


# ── Cell 11: Run All Regimes ───────────────────────────────────
DATASETS = {
    "hc3":  {"eval": hc3_eval,  "cot": hc3_cot_eval,  "pool": hc3_pool},
    "eli5": {"eval": eli5_eval, "cot": eli5_cot_eval, "pool": eli5_pool},
}

all_results = {}
summary     = []

print("=" * 65)
print(f"GPT-4o-mini FULL EVAL")
print(f"ZS/FS n={EVAL_SAMPLE_SIZE} per dataset | CoT n={COT_SAMPLE_SIZE}")
print("=" * 65)

for regime in ["zero_shot", "few_shot", "cot"]:
    all_results[regime] = {}
    print(f"\n── {regime.upper()} ──────────────────────────────────────")

    for ds_name, cfg in DATASETS.items():
        ev   = cfg["cot"] if regime == "cot" else cfg["eval"]
        pool = cfg["pool"]

        print(f"  [{ds_name.upper()}] {len(ev)} samples ...")
        df_r = score_gpt(ev, pool, regime, ds_name)
        all_results[regime][ds_name] = df_r

        y_t = (df_r["true_label"] == "llm").astype(int)
        y_s = df_r["score"]
        auc = (roc_auc_score(y_t, y_s)
               if y_t.nunique() > 1 else float("nan"))
        acc = accuracy_score(y_t, (y_s > 0.5).astype(int))
        fp  = int((df_r["failure_mode"] == "false_positive").sum())
        fn  = int((df_r["failure_mode"] == "false_negative").sum())
        unk = (df_r["failure_mode"] == "unknown_output").mean()
        ml  = float(df_r[df_r["true_label"] == "llm"]["score"].mean())
        mh  = float(df_r[df_r["true_label"] == "human"]["score"].mean())
        sep = ml - mh

        direction = ("✅ correct" if sep > 0.05 else
                     "⚠️  weak"   if sep > 0    else
                     "❌ INVERTED")

        print(f"   AUROC={auc:.4f}  Acc={acc:.4f}  "
              f"FP={fp}  FN={fn}  Unknown={unk:.1%}")
        print(f"   score(llm)={ml:.3f}  score(human)={mh:.3f}  "
              f"sep={sep:+.3f}  {direction}")

        if regime == "cot":
            conf_avail = df_r["cot_conf"].notna().mean()
            unknown_v  = (df_r["cot_verdict"] == "unknown").mean()
            print(f"   cot_conf_avail={conf_avail:.1%}  "
                  f"unknown_verdict={unknown_v:.1%}")

        df_r.to_csv(
            f"{RESULTS_DIR}/GPT4oMini_{regime}_{ds_name}.csv",
            index=False)

        summary.append({
            "regime":    regime,
            "dataset":   ds_name,
            "auroc":     round(float(auc), 4),
            "accuracy":  round(float(acc), 4),
            "fp":        fp,
            "fn":        fn,
            "n":         len(df_r),
            "sep":       round(sep, 4),
            "unknown%":  f"{unk:.1%}",
        })

# Token cost estimate
est_cost = (tokens_used_total / 1_000_000) * 0.375   # blended rate
print(f"\n💰 Total tokens used: {tokens_used_total:,}  "
      f"estimated cost: ~${est_cost:.3f}")
print("✅ All regimes complete")


# ── Cell 12: Summary Tables ────────────────────────────────────

df_summary = pd.DataFrame(summary)
df_summary.to_csv(f"{RESULTS_DIR}/summary_gpt4omini.csv", index=False)

# ── Main summary ──
print(f"\nGPT-4o-mini — Results Summary")
hdr = (f"{'regime':<12} {'dataset':<6} {'auroc':>8} "
       f"{'accuracy':>10} {'fp':>6} {'fn':>6} {'n':>6}")
div = "─" * len(hdr)
print(div); print(hdr); print(div)
for r in summary:
    print(f"{r['regime']:<12} {r['dataset']:<6} "
          f"{r['auroc']:>8.4f} {r['accuracy']:>10.4f} "
          f"{r['fp']:>6} {r['fn']:>6} {r['n']:>6}")
print(div)

# ── Calibrated threshold table ──
cal_rows = []
for regime in all_results:
    for ds_name, df_r in all_results[regime].items():
        y_t = (df_r["true_label"] == "llm").astype(int)
        y_s = df_r["score"]
        if y_t.nunique() < 2:
            continue
        fpr_c, tpr_c, thresholds = roc_curve(y_t, y_s)
        best_t  = float(thresholds[(tpr_c - fpr_c).argmax()])
        med_t   = float(y_s.median())
        cal_rows.append({
            "regime":         regime,
            "dataset":        ds_name,
            "auroc":          round(float(roc_auc_score(y_t, y_s)), 4),
            "acc@0.5":        round(accuracy_score(
                                  y_t, (y_s >= 0.5).astype(int)), 4),
            "acc@median":     round(accuracy_score(
                                  y_t, (y_s >= med_t).astype(int)), 4),
            "acc@optimal":    round(accuracy_score(
                                  y_t, (y_s >= best_t).astype(int)), 4),
            "median_score":   round(med_t, 4),
            "optimal_thresh": round(best_t, 4),
        })

pd.DataFrame(cal_rows).to_csv(
    f"{RESULTS_DIR}/calibrated_gpt4omini.csv", index=False)

print(f"\nGPT-4o-mini — Calibrated Threshold Results")
ch = (f"{'regime':<12} {'dataset':<6} {'auroc':>8} {'acc@0.5':>9} "
      f"{'acc@median':>12} {'acc@optimal':>13} "
      f"{'median':>8} {'opt_thresh':>11}")
d2 = "─" * len(ch)
print(d2); print(ch); print(d2)
for r in cal_rows:
    print(f"{r['regime']:<12} {r['dataset']:<6} "
          f"{r['auroc']:>8.4f} {r['acc@0.5']:>9.4f} "
          f"{r['acc@median']:>12.4f} {r['acc@optimal']:>13.4f} "
          f"{r['median_score']:>8.4f} {r['optimal_thresh']:>11.4f}")
print(d2)

# ── Score direction check ──
print("\nScore Direction Check (must all be ✅ for valid results):")
print(f"{'regime':<12} {'dataset':<6} {'score(human)':>14} "
      f"{'score(llm)':>12} {'sep':>8} {'status':>18}")
print("─" * 74)
for regime in all_results:
    for ds_name, df_r in all_results[regime].items():
        mh  = float(df_r[df_r["true_label"] == "human"]["score"].mean())
        ml  = float(df_r[df_r["true_label"] == "llm"]["score"].mean())
        sep = ml - mh
        st  = ("✅ correct" if sep > 0.05 else
               "⚠️  weak"   if sep > 0    else
               "❌ INVERTED")
        print(f"{regime:<12} {ds_name:<6} {mh:>14.4f} "
              f"{ml:>12.4f} {sep:>8.4f} {st:>18}")


# ── Cell 13: Plots ─────────────────────────────────────────────
sns.set_style("whitegrid")
PALETTE = {
    "zero_shot": "#3498db",
    "few_shot":  "#2ecc71",
    "cot":       "#e74c3c"
}

# ── Plot 1: AUROC bar chart ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("GPT-4o-mini — AUROC by Regime\n"
             "(reasoning chain + AI_SCORE scoring)",
             fontsize=13, fontweight="bold")

for ax, ds_name in zip(axes, ["hc3", "eli5"]):
    regimes = list(all_results.keys())
    aurocs  = []
    for r in regimes:
        df_r = all_results[r][ds_name]
        y_t  = (df_r["true_label"] == "llm").astype(int)
        y_s  = df_r["score"]
        aurocs.append(
            float(roc_auc_score(y_t, y_s))
            if y_t.nunique() > 1 else 0.5)

    colors = [PALETTE[r] for r in regimes]
    bars   = ax.bar(regimes, aurocs, color=colors,
                    alpha=0.85, edgecolor="white", width=0.5)
    ax.axhline(0.5, color="k", linestyle="--", alpha=0.4,
               linewidth=1, label="Random baseline")
    ax.set_ylim(0.3, 1.05)
    ax.set_ylabel("AUROC")
    ax.set_title(f"{ds_name.upper()}", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, aurocs):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.012,
                f"{val:.3f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/plot_auroc_by_regime.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Plot 2: ROC curves ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("GPT-4o-mini — ROC Curves",
             fontsize=13, fontweight="bold")

for ax, ds_name in zip(axes, ["hc3", "eli5"]):
    ax.plot([0, 1], [0, 1], "k--", linewidth=1,
            alpha=0.4, label="Random")
    for regime, color in PALETTE.items():
        df_r = all_results[regime][ds_name]
        y_t  = (df_r["true_label"] == "llm").astype(int).values
        y_s  = df_r["score"].values
        if len(np.unique(y_t)) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_t, y_s)
        auc         = roc_auc_score(y_t, y_s)
        ax.plot(fpr, tpr, linewidth=2, color=color, alpha=0.9,
                label=f"{regime} (AUC={auc:.3f})")
    ax.set_xlabel("FPR", fontsize=11)
    ax.set_ylabel("TPR", fontsize=11)
    ax.set_title(ds_name.upper(), fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/plot_roc_curves.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Plot 3: Score distributions ─────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10), squeeze=False)
fig.suptitle("GPT-4o-mini — Score Distributions\n"
             "Blue=Human  Red=LLM  |  score = AI_SCORE/100",
             fontsize=13, fontweight="bold")

for col, regime in enumerate(["zero_shot", "few_shot", "cot"]):
    for row_, ds_name in enumerate(["hc3", "eli5"]):
        ax   = axes[row_][col]
        df_r = all_results[regime][ds_name]
        y_t  = (df_r["true_label"] == "llm").astype(int)
        y_s  = df_r["score"]
        mh   = float(y_s[y_t == 0].mean())
        ml   = float(y_s[y_t == 1].mean())

        ax.hist(y_s[y_t == 0], bins=25, range=(0, 1),
                alpha=0.6, color="#3498db", density=True, label="Human")
        ax.hist(y_s[y_t == 1], bins=25, range=(0, 1),
                alpha=0.6, color="#e74c3c", density=True, label="LLM")
        ax.axvline(0.5, color="k", linestyle="--",
                   alpha=0.4, linewidth=1)

        auc = (float(roc_auc_score(y_t, y_s))
               if y_t.nunique() > 1 else float("nan"))
        ax.set_title(
            f"{regime} / {ds_name.upper()}\n"
            f"AUROC={auc:.3f}  sep={ml - mh:+.3f}",
            fontsize=9, fontweight="bold")
        ax.set_xlabel("AI_SCORE (0=human, 1=AI)")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/plot_score_distributions.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Plot 4: Regime comparison bar ───────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x      = np.arange(3)
width  = 0.35
regs   = ["zero_shot", "few_shot", "cot"]

for i, ds_name in enumerate(["hc3", "eli5"]):
    aurocs = []
    for r in regs:
        df_r = all_results[r][ds_name]
        y_t  = (df_r["true_label"] == "llm").astype(int)
        y_s  = df_r["score"]
        aurocs.append(
            float(roc_auc_score(y_t, y_s))
            if y_t.nunique() > 1 else 0.5)
    ax.bar(x + i * width, aurocs, width,
           label=ds_name.upper(),
           color=["#3498db", "#e74c3c"][i], alpha=0.8)
    for j, val in enumerate(aurocs):
        ax.text(x[j] + i * width, val + 0.01,
                f"{val:.3f}", ha="center", fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(["Zero-Shot", "Few-Shot", "CoT"])
ax.axhline(0.5, color="k", linestyle="--", alpha=0.3)
ax.set_ylabel("AUROC")
ax.set_ylim(0.3, 1.05)
ax.set_title("GPT-4o-mini: Regime Comparison\n"
             "Zero-Shot vs Few-Shot vs CoT",
             fontsize=12, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/plot_regime_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("✅ All plots saved")


# ── Cell 14: Visual JSON Output ────────────────────────────────
print("\nGenerating visual JSON files ...")

for regime in all_results:
    for ds_name, df_r in all_results[regime].items():
        y_t = (df_r["true_label"] == "llm").astype(int)
        y_s = df_r["score"]
        auc = (float(roc_auc_score(y_t, y_s))
               if y_t.nunique() > 1 else float("nan"))

        records = []
        for _, row in df_r.iterrows():
            records.append({
                "sample_id":    int(row["idx"]),
                "true_label":   row["true_label"],
                "pred_label":   row["pred_label"],
                "score":        round(float(row["score"]), 4),
                "correct":      bool(row["correct"]),
                "failure_mode": row.get("failure_mode", "unknown"),
                "confidence":   (
                    "HIGH"   if abs(row["score"] - 0.5) > 0.35 else
                    "MEDIUM" if abs(row["score"] - 0.5) > 0.15 else
                    "LOW"
                ),
                "text_preview": row["text_preview"],
                "gpt_output":   row.get("raw_model_output", "N/A"),
                "cot_verdict":  row.get("cot_verdict"),
                "cot_conf":     row.get("cot_conf"),
            })

        fp_records = [r for r in records
                      if r["failure_mode"] == "false_positive"]
        fn_records = [r for r in records
                      if r["failure_mode"] == "false_negative"]

        out = {
            "model":          "GPT-4o-mini",
            "regime":         regime,
            "dataset":        ds_name,
            "scoring_method": (
                "3-step-reasoning → AI_SCORE (0=human,100=AI)"
                if regime != "cot"
                else "7-dim CoT → AI_CONFIDENCE (0-10)"
            ),
            "summary": {
                "total":           len(records),
                "correct":         sum(r["correct"] for r in records),
                "accuracy":        round(
                    sum(r["correct"] for r in records) / len(records), 3),
                "auroc":           round(auc, 3),
                "false_positives": len(fp_records),
                "false_negatives": len(fn_records),
            },
            "top10_worst_fp": sorted(
                fp_records,
                key=lambda x: x["score"], reverse=True)[:10],
            "top10_worst_fn": sorted(
                fn_records,
                key=lambda x: x["score"])[:10],
            "by_failure_mode": {
                "false_positives": fp_records,
                "false_negatives": fn_records,
                "correct": [r for r in records if r["correct"]],
            }
        }
        path = (f"{RESULTS_DIR}/visual_GPT4oMini"
                f"_{regime}_{ds_name}.json")
        with open(path, "w") as f:
            json.dump(out, f, indent=2, ensure_ascii=False)
        print(f"  ✅ {os.path.basename(path)}")

with open(f"{RESULTS_DIR}/results_gpt4omini.pkl", "wb") as f:
    pickle.dump(all_results, f)

# Final cost summary
est_cost = (tokens_used_total / 1_000_000) * 0.375
print(f"\n💰 Final token count: {tokens_used_total:,}")
print(f"   Estimated total cost: ~${est_cost:.3f}")
print(f"   (blended rate $0.375/1M — actual may vary slightly)")

print("\n✅ GPT-4o-mini evaluation complete")
print(f"   Results: {RESULTS_DIR}/")